# NUCLEAR ULTIMATE V1 — Data-driven electrostatic surface-patch reasoning

**Task:** `nuclear_binding` residue prediction  
**Development:** exact DevSet1010 official bindNode24 5-fold protein split  
**External confirmation:** TestSet300 + TestSetNew46  
**Primary selection metric:** residue-level Dev CV AUPRC

This notebook is built from the completed three-dataset anatomy audit.

## Audit-derived design

The observed nuclear phenotype differs strongly from metal and small-molecule binding:

- DevSet1010 nuclear positives occur in only ~10.7% of proteins, but positive proteins have broad interfaces (median ~19.5 residues).
- Nuclear-positive residues are strongly enriched for **R/K**, with a reproducible **basic local-context** signal on Dev1010, Test300 and Test46.
- Existing cached features show **ELEC3D** as the strongest nuclear discriminator across all three sets, with **ACCESS** also consistently informative.
- Nuclear positives are highly clustered along sequence (about 84% in multi-residue clusters at gap ≤2 on Dev/Test300).
- Generic residue graphs previously performed poorly for nuclear, implying that generic message passing is not the correct inductive bias.
- Test46 has much smaller nuclear interfaces and a stronger distribution shift; therefore patch reasoning must be a **soft residual**, never a hard requirement.

## V1 hypotheses

### H1 — Backbone attraction / electrostatics
Nuclear-binding interfaces are enriched for accessible basic/electrostatic environments.

### H2 — H-bond recognition
Explicit cationic and donor/acceptor chemistry adds information beyond ESM.

### H3 — Surface-patch organization
Higher-order cationic/polar patch context adds information beyond residue-local features.

### H4 — Biological context
Evolutionary and sequence-motif information can distinguish functional nucleic-acid interfaces from accidental positive patches.

### H5 — Patch hypergraph
A residue↔surface-patch bipartite model should be more appropriate than a generic residue graph.

---

## Scientific contract

```text
DevSet1010 only
   │
   ├── feature-family MLP screen
   ├── combination ladder
   ├── multimodal architecture screen
   └── patch-hypergraph screen
   │
   ▼
GLOBAL LOCK
   │
   ▼
TestSet300 + TestSetNew46
```

No feature/model choice is allowed to depend on TestSet300/TestSetNew46.


In [ ]:
# ============================================================
# 0. DRIVE / INSTALL / IMPORT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, math, shutil, subprocess, platform
from collections import Counter, defaultdict

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn",
     "biopython", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
from scipy.spatial import cKDTree
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score
)
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1
from copy import deepcopy

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]
ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)

if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE.'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

try:
    from gear_ultimate import GEAR_ULTIMATE_VERSION as GEAR_VERSION
except Exception:
    from gear_ultimate import __version__ as GEAR_VERSION

from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_graph as graph
from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

RUN_ROOT = BASE/'results'/'NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = BASE/'cache'/'nuclear_ultimate_v1'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('GEAR version:', GEAR_VERSION)
print('Device:', DEVICE)
print('Run root:', RUN_ROOT)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
GEAR backend: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE_OTHER_BINDINGS
GEAR version: 2026.08-other-bindings-v1
Device: cuda
Run root: /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH


## 1. Master switches

Default = full nuclear experiment.

All expensive generated blocks and all custom model folds are persisted and resume-safe.


In [ ]:
# ============================================================
# 1. MASTER SWITCHES
# ============================================================
ESM_BATCH_SIZE = 8

RUN_FEATURE_GENERATION = True
RUN_FAMILY_SCREEN = True
RUN_COMBINATION_LADDER = True
RUN_CUSTOM_ARCHITECTURES = True
RUN_PATCH_HYPERGRAPH = True
RUN_FINAL_EXTERNAL = True

ATTACH_EVO_IF_CACHED = True
ATTACH_ESMIF_IF_CACHED = True

FORCE_REBUILD_NA_FEATURES = False
FORCE_REBUILD_PATCHES = False

MIN_MEAN_AUPRC_GAIN = 0.002
GLOBAL_COMPLEXITY_MARGIN = 0.002

CUSTOM_BATCH = 4096
CUSTOM_EPOCHS = 45
CUSTOM_PATIENCE = 7
CUSTOM_LR = 3e-4
CUSTOM_WEIGHT_DECAY = 1e-4
SEED = 2026

PATCH_RADIUS_A = 10.0
PATCH_MAX_PER_PROTEIN = 128
PATCH_PROTEIN_BATCH = 4
PATCH_EPOCHS = 32
PATCH_PATIENCE = 6
PATCH_LR = 3e-4
PATCH_WEIGHT_DECAY = 1e-4
PATCH_LATENT = 192

print('Full run:', {
    'feature_generation': RUN_FEATURE_GENERATION,
    'family_screen': RUN_FAMILY_SCREEN,
    'combination_ladder': RUN_COMBINATION_LADDER,
    'custom_architectures': RUN_CUSTOM_ARCHITECTURES,
    'patch_hypergraph': RUN_PATCH_HYPERGRAPH,
    'external': RUN_FINAL_EXTERNAL,
})


Full run: {'feature_generation': True, 'family_screen': True, 'combination_ladder': True, 'custom_architectures': True, 'patch_hypergraph': True, 'external': True}


In [ ]:
# ============================================================
# 2. PREPARE / REUSE NUCLEAR GEAR CONTEXT
# ============================================================
ctx = suite.prepare_context(
    base=BASE,
    run_version='nuclear_ultimate_v1_context',
    label_column='nuclear_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

# Existing GEAR stores must already be exact DevSet1010 / Test300 / Test46.
assert len(ctx.stores['train']) == 1010
assert len(ctx.stores['test300']) == 300
assert len(ctx.stores['test46']) == 46

print('Context sizes: PASS')


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,2689,0.015801
1,test300,300,62689,1470,0.023449
2,test46,46,6227,77,0.012366


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


Context sizes: PASS


In [ ]:
# ============================================================
# 3. DATAFRAME / OFFICIAL-FOLD AUDIT
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in [
        'metal_binding',
        'nuclear_binding',
        'small_binding',
        'any_ligand_binding_sites',
    ]:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    for r in df.itertuples():
        if len(r.sequence) != len(r.nuclear_binding):
            raise RuntimeError(f'{r.prot_id}: nuclear label/sequence mismatch')

    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_nuclear_v1')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')
    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)
missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}')

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}
STORE_KEY = {
    'DevSet1010':'train',
    'TestSet300':'test300',
    'TestSetNew46':'test46',
}

# Cross-check store IDs.
for split, df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))
    if store_ids != csv_ids:
        raise RuntimeError(
            f'{split}: store/CSV mismatch. '
            f'missing_in_store={sorted(csv_ids-store_ids)[:10]} '
            f'extra_in_store={sorted(store_ids-csv_ids)[:10]}'
        )

print('Exact dataset/fold alignment: PASS')
print('Fold sizes:', {k:len(v) for k,v in OFFICIAL_FOLDS.items()})


Exact dataset/fold alignment: PASS
Fold sizes: {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


## 4. Reuse orthogonal biological context where available

`EVO48` is retained because the previous nuclear experiments showed a modest but reproducible Dev gain.

ESM-IF is **not required** for V1. If a complete per-residue ESM-IF cache already exists from earlier runs, it is attached as an optional structural control; otherwise V1 proceeds without generating a new 1010-protein ESM-IF cache.


In [ ]:
# ============================================================
# 4. ATTACH EVO / OPTIONAL EXISTING ESMIF
# ============================================================
EVO_AVAILABLE = False

if ATTACH_EVO_IF_CACHED:
    try:
        coverage = evo.attach_evo(
            ctx.stores,
            BASE,
            splits=('train','test300','test46'),
            require_msa=True,
        )
        display(coverage)
        EVO_AVAILABLE = True
        print('EVO48 attached.')
    except Exception as e:
        print('EVO cache unavailable/incomplete -> skip EVO:', repr(e))

# Robust store helpers.
def get_record(store, pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store, pid, block):
    rec = get_record(store, pid)

    if isinstance(rec, dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'), dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec, block):
        return np.asarray(getattr(rec, block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store, pid, block, arr):
    rec = get_record(store, pid)
    arr = np.asarray(arr, dtype=np.float32)

    if isinstance(rec, dict):
        if isinstance(rec.get('features'), dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict):
            d[block] = arr
            return

    setattr(rec, block, arr)

# Optional reuse-only ESMIF.
ESMIF_AVAILABLE = False
ESMIF_CANDIDATE_ROOTS = [
    BASE/'cache'/'gear_metal_v3'/'esmif_v1',
    BASE/'cache'/'nuclear_ultimate_v1'/'esmif_v1',
]

if ATTACH_ESMIF_IF_CACHED:
    for root in ESMIF_CANDIDATE_ROOTS:
        split_dirs = {
            'DevSet1010': root/'DevSet1010'/'esm_if',
            'TestSet300': root/'TestSet300'/'esm_if',
            'TestSetNew46': root/'TestSetNew46'/'esm_if',
        }

        if not all(d.exists() for d in split_dirs.values()):
            continue

        good = True
        for split, df in SPLIT_DF.items():
            d = split_dirs[split]
            for r in df.itertuples():
                fp = d/f'{r.prot_id}.npy'
                if not fp.exists():
                    good = False
                    break
                a = np.load(fp, mmap_mode='r')
                if a.shape != (len(r.sequence),512):
                    good = False
                    break
            if not good:
                break

        if good:
            for split, df in SPLIT_DF.items():
                store = ctx.stores[STORE_KEY[split]]
                d = split_dirs[split]
                for r in df.itertuples():
                    set_block(
                        store,
                        r.prot_id,
                        'ESMIF',
                        np.load(d/f'{r.prot_id}.npy').astype(np.float32)
                    )

            ESMIF_AVAILABLE = True
            print('Reused complete ESMIF cache:', root)
            break

print('EVO_AVAILABLE  =', EVO_AVAILABLE)
print('ESMIF_AVAILABLE=', ESMIF_AVAILABLE)


EVO cache:   0%|          | 0/1353 [00:00<?, ?it/s]

,split,protein,msa_depth,has_real_msa
0,train,A0A0F7RDM3,977,True
1,train,A0A0F7RHX8,1024,True
2,train,A0A0H2W6Y8,1023,True
3,train,A0A2B6C3P9,1024,True
4,train,A0A384LKY8,542,True
...,...,...,...,...
1351,test46,Q9KDJ7,1023,True
1352,test46,Q9LFM3,1018,True
1353,test46,Q9SJ89,1022,True
1354,test46,U2EQ00,1021,True


EVO48 attached.
EVO_AVAILABLE  = True
ESMIF_AVAILABLE= False


# Part A — Nuclear-specific feature generation

The new blocks are intentionally different from the metal ontology.

## `NA_SEQELECTRO32`
Interpretable sequence-local cationic/electrostatic environment.

## `NA_PATCH3D40`
3D neighborhood composition and electrostatic proxy around each residue.

## `NA_HBOND16`
Cationic + donor/acceptor chemistry and local H-bond-capable environment.

## `NA_TOPO12`
Topology/connectedness of spatial basic/polar neighborhoods.

## `NA_PROTCTX8`
Soft protein-level physicochemical context. This is a control/conditioning feature, never a hard gate.

## `NA_AROMCTRL8`
Aromatic/stacking-oriented control. Audit data does **not** support making aromatic chemistry a core nuclear assumption, so this block must earn its place on Dev.


In [ ]:
# # ============================================================
# # 5. NORMALIZE EXACT-SEQUENCE PDB STRUCTURES
# # ============================================================
# AA3 = {
#     'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
#     'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
# }

# PDB_NORM_ROOT = CACHE_ROOT/'normalized_pdb'
# PDB_NORM_ROOT.mkdir(parents=True, exist_ok=True)

# # Build a priority-ordered PDB index.
# pdb_roots = [
#     BASE/'esmFold_pdb_files',
#     BASE/'experimental_pdb',
#     BASELINE_WORK/'cache'/'metal_external_normalized_structures',
#     BN24/'dataset'/'AlphaFold_predictions',
# ]

# all_pdb = []
# for root in pdb_roots:
#     if root.exists():
#         all_pdb.extend(root.rglob('*.pdb'))

# exact_index = {}
# for p in all_pdb:
#     exact_index.setdefault(p.stem, p)

# def standard_residues(chain):
#     return [
#         r for r in chain
#         if r.id[0] == ' ' and r.resname.strip().upper() in AA3
#     ]

# def residue_sequence(residues):
#     return ''.join(
#         seq1(r.resname.strip().upper())
#         for r in residues
#     )

# def find_source_pdb(pid):
#     pid = str(pid)

#     if pid in exact_index:
#         return exact_index[pid]

#     # Common explicit locations.
#     for p in [
#         BASE/'esmFold_pdb_files'/f'{pid}.pdb',
#         BN24/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb',
#     ]:
#         if p.exists():
#             return p

#     # Last-resort unambiguous stem containment.
#     hits = [p for p in all_pdb if pid in p.stem]
#     if len(hits) == 1:
#         return hits[0]

#     return None

# def cached_norm_ok(path, target_seq):
#     if not path.exists():
#         return False
#     try:
#         st = PDBParser(QUIET=True).get_structure('x', str(path))
#         m = next(st.get_models())
#         if 'A' not in m:
#             return False
#         return residue_sequence(standard_residues(m['A'])) == target_seq
#     except Exception:
#         return False

# def normalize_exact_chain_A(src, target_seq, dst):
#     st = PDBParser(QUIET=True).get_structure('src', str(src))
#     m = next(st.get_models())

#     exact = []
#     diag = []

#     for ch in m:
#         rr = standard_residues(ch)
#         seq = residue_sequence(rr)
#         diag.append((str(ch.id),len(seq)))
#         if seq == target_seq:
#             exact.append((ch,rr))

#     if not exact:
#         raise RuntimeError(
#             f'{src}: no chain exactly matches sequence length {len(target_seq)}; '
#             f'chains={diag}'
#         )

#     exact.sort(key=lambda x:(0 if str(x[0].id)=='A' else 1,str(x[0].id)))
#     _, rr = exact[0]

#     out_st = Structure('normalized')
#     out_m = Model(0)
#     out_ch = Chain('A')
#     out_st.add(out_m)
#     out_m.add(out_ch)

#     for i,res in enumerate(rr,start=1):
#         r2 = deepcopy(res)
#         r2.detach_parent()
#         r2.id = (' ',i,' ')
#         out_ch.add(r2)

#     dst.parent.mkdir(parents=True, exist_ok=True)

#     io = PDBIO()
#     io.set_structure(out_st)
#     io.save(str(dst))

#     if not cached_norm_ok(dst,target_seq):
#         raise RuntimeError(f'{dst}: normalized read-back mismatch')

# for split, df in SPLIT_DF.items():
#     d = PDB_NORM_ROOT/split
#     d.mkdir(parents=True, exist_ok=True)

#     missing = []

#     for n,r in enumerate(df.itertuples(),start=1):
#         dst = d/f'{r.prot_id}.pdb'

#         if cached_norm_ok(dst,r.sequence):
#             continue

#         src = find_source_pdb(r.prot_id)
#         if src is None:
#             missing.append(str(r.prot_id))
#             continue

#         normalize_exact_chain_A(src,r.sequence,dst)

#         if n % 250 == 0:
#             print(split,n,'/',len(df))

#     if missing:
#         raise FileNotFoundError(
#             f'{split}: source PDB missing for {len(missing)} proteins: {missing[:20]}'
#         )

# print('Exact-sequence normalized structures: PASS')


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 5. SHARED EXACT PDB CACHE — NUCLEAR + SMALL
# ============================================================
from pathlib import Path
import shutil
import tarfile
import json
import hashlib
import os

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

# ------------------------------------------------------------
# ONE local cache shared by nuclear + small
# ------------------------------------------------------------
LOCAL_SHARED_ROOT = Path('/content/shared_exact_pdb_v1')

DRIVE_ARCHIVE = (
    BASE / 'cache' / 'shared_exact_pdb_v1.tar.gz'
)

DRIVE_ARCHIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Downstream cells continue using PDB_NORM_ROOT.
PDB_NORM_ROOT = LOCAL_SHARED_ROOT

# Existing work that we should salvage rather than redo.
OLD_NORMALIZED_ROOTS = [
    BASE / 'cache' / 'nuclear_ultimate_v1' / 'normalized_pdb',
    BASE / 'cache' / 'small_ultimate_v1' / 'normalized_pdb',

    BASELINE_WORK
    / 'cache'
    / 'metal_external_normalized_structures',
]

# Raw source roots only used when no normalized copy exists.
RAW_ROOTS = [
    BASE / 'esmFold_pdb_files',
    BASE / 'experimental_pdb',
    BN24 / 'dataset' / 'AlphaFold_predictions',
]

_parser = PDBParser(QUIET=True)


# ============================================================
# helpers
# ============================================================
def standard_residues(chain):
    return [
        r for r in chain
        if (
            r.id[0] == ' '
            and r.resname.strip().upper() in AA3
        )
    ]


def residue_sequence(rr):
    return ''.join(
        seq1(r.resname.strip().upper())
        for r in rr
    )


def pdb_exact_ok(path, target_seq):
    """
    Check whether PDB is already exact chain-A normalized.

    Important:
    use this mainly on /content or on a candidate file,
    NOT recursively over Drive.
    """
    path = Path(path)

    if not path.exists():
        return False

    try:
        st = _parser.get_structure(
            'check',
            str(path)
        )

        model = next(st.get_models())

        if 'A' not in model:
            return False

        rr = standard_residues(
            model['A']
        )

        return (
            residue_sequence(rr)
            == target_seq
        )

    except Exception:
        return False


def normalize_exact_chain_A(src, target_seq, dst):
    """
    Normalize ONLY if we truly do not already have
    an exact-normalized copy.
    """
    st = _parser.get_structure(
        'src',
        str(src)
    )

    model = next(
        st.get_models()
    )

    exact = []
    diag = []

    for ch in model:
        rr = standard_residues(ch)
        seq = residue_sequence(rr)

        diag.append(
            (
                str(ch.id),
                len(seq)
            )
        )

        if seq == target_seq:
            exact.append(
                (
                    ch,
                    rr
                )
            )

    if not exact:
        raise RuntimeError(
            f'{src}: no exact chain match; '
            f'target={len(target_seq)}, '
            f'chains={diag}'
        )

    exact.sort(
        key=lambda x: (
            0
            if str(x[0].id) == 'A'
            else 1,

            str(x[0].id)
        )
    )

    _, rr = exact[0]

    out_st = Structure('normalized')
    out_model = Model(0)
    out_chain = Chain('A')

    out_st.add(out_model)
    out_model.add(out_chain)

    for i, res in enumerate(
        rr,
        start=1
    ):
        r2 = deepcopy(res)
        r2.detach_parent()

        r2.id = (
            ' ',
            i,
            ' '
        )

        out_chain.add(
            r2
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    io = PDBIO()
    io.set_structure(
        out_st
    )

    io.save(
        str(dst)
    )


# ============================================================
# 1. RESTORE SHARED CACHE IF IT EXISTS
# ============================================================
if (
    DRIVE_ARCHIVE.exists()
    and not LOCAL_SHARED_ROOT.exists()
):
    print(
        'Found shared PDB archive.'
    )

    print(
        'Copy Drive -> /content ...'
    )

    local_tar = (
        Path('/content')
        / DRIVE_ARCHIVE.name
    )

    shutil.copy2(
        DRIVE_ARCHIVE,
        local_tar
    )

    print(
        'Extracting locally ...'
    )

    with tarfile.open(
        local_tar,
        'r:gz'
    ) as tf:
        tf.extractall(
            '/content'
        )

    local_tar.unlink(
        missing_ok=True
    )


LOCAL_SHARED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FAST LOOKUP OF OLD NORMALIZED STRUCTURES
# ============================================================
# Direct paths first — no rglob.
def normalized_candidates(
    split,
    pid
):
    pid = str(pid)

    aliases = {
        'DevSet1010': [
            'DevSet1010',
            'train',
        ],

        'TestSet300': [
            'TestSet300',
            'test300',
        ],

        'TestSetNew46': [
            'TestSetNew46',
            'test46',
        ],
    }

    out = []

    for root in OLD_NORMALIZED_ROOTS:
        out.append(
            root / f'{pid}.pdb'
        )

        for alias in aliases.get(
            split,
            []
        ):
            out.append(
                root
                / alias
                / f'{pid}.pdb'
            )

    return out


def raw_candidates(
    pid
):
    pid = str(pid)

    return [
        BASE
        / 'esmFold_pdb_files'
        / f'{pid}.pdb',

        BASE
        / 'experimental_pdb'
        / f'{pid}.pdb',

        BN24
        / 'dataset'
        / 'AlphaFold_predictions'
        / f'{pid}.pdb',
    ]


# Only build fallback index if direct paths actually fail.
RAW_FALLBACK_INDEX = None


def get_raw_fallback_index():
    global RAW_FALLBACK_INDEX

    if RAW_FALLBACK_INDEX is not None:
        return RAW_FALLBACK_INDEX

    print(
        '\nDirect raw paths missed at least one protein.'
    )

    print(
        'Building ONE fallback source index...'
    )

    idx = {}

    for root in RAW_ROOTS:
        if not root.exists():
            continue

        print(
            ' scanning:',
            root
        )

        for dirpath, _, filenames in os.walk(
            root
        ):
            for fn in filenames:
                if not fn.endswith(
                    '.pdb'
                ):
                    continue

                p = (
                    Path(dirpath)
                    / fn
                )

                idx.setdefault(
                    p.stem,
                    p
                )

    RAW_FALLBACK_INDEX = idx

    print(
        'fallback PDBs indexed:',
        len(idx)
    )

    return idx


# ============================================================
# 3. BUILD SHARED LOCAL CACHE
# ============================================================
stats = {
    'already_local': 0,
    'reused_old_normalized': 0,
    'normalized_raw': 0,
}

missing = []

TOTAL = sum(
    len(df)
    for df in SPLIT_DF.values()
)

done = 0


for split, df in SPLIT_DF.items():

    outdir = (
        LOCAL_SHARED_ROOT
        / split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        '\n',
        split,
        len(df),
        'proteins'
    )

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        dst = (
            outdir
            / f'{pid}.pdb'
        )

        # ----------------------------------
        # A. already local shared
        # ----------------------------------
        if pdb_exact_ok(
            dst,
            r.sequence
        ):
            stats[
                'already_local'
            ] += 1

            done += 1
            continue


        # ----------------------------------
        # B. reuse nuclear/small/metal
        #    normalized PDB AS-IS
        # ----------------------------------
        reused = False

        for cand in normalized_candidates(
            split,
            pid
        ):

            if not cand.exists():
                continue

            if pdb_exact_ok(
                cand,
                r.sequence
            ):
                shutil.copy2(
                    cand,
                    dst
                )

                stats[
                    'reused_old_normalized'
                ] += 1

                reused = True
                break


        if reused:
            done += 1

            if done % 100 == 0:
                print(
                    f'  {done}/{TOTAL}',
                    stats
                )

            continue


        # ----------------------------------
        # C. raw source direct lookup
        # ----------------------------------
        src = None

        for cand in raw_candidates(
            pid
        ):
            if cand.exists():
                src = cand
                break


        # ----------------------------------
        # D. last-resort ONE-TIME index
        # ----------------------------------
        if src is None:
            src = get_raw_fallback_index().get(
                pid
            )


        if src is None:
            missing.append(
                (
                    split,
                    pid
                )
            )

            continue


        # ----------------------------------
        # E. normalize only genuinely missing
        # ----------------------------------
        normalize_exact_chain_A(
            src,
            r.sequence,
            dst
        )

        if not pdb_exact_ok(
            dst,
            r.sequence
        ):
            raise RuntimeError(
                f'{split}/{pid}: '
                'normalized PDB failed validation'
            )

        stats[
            'normalized_raw'
        ] += 1

        done += 1


        if done % 100 == 0:
            print(
                f'  {done}/{TOTAL}',
                stats
            )


if missing:
    raise FileNotFoundError(
        f'Missing source PDB for '
        f'{len(missing)} proteins: '
        f'{missing[:20]}'
    )


print(
    '\nCACHE BUILD COMPLETE'
)

print(
    stats
)


# ============================================================
# 4. SAVE ONE ARCHIVE TO DRIVE
# ============================================================
print(
    '\nPacking shared local cache...'
)

local_tar = (
    Path('/content')
    / 'shared_exact_pdb_v1.tar.gz'
)

with tarfile.open(
    local_tar,
    'w:gz'
) as tf:

    tf.add(
        LOCAL_SHARED_ROOT,
        arcname='shared_exact_pdb_v1'
    )


print(
    'Copying ONE archive to Drive...'
)

shutil.copy2(
    local_tar,
    DRIVE_ARCHIVE
)

local_tar.unlink(
    missing_ok=True
)


print(
    '\nExact-sequence shared structures: PASS'
)

print(
    'PDB_NORM_ROOT =',
    PDB_NORM_ROOT
)

print(
    'Persistent archive =',
    DRIVE_ARCHIVE
)

Found shared PDB archive.
Copy Drive -> /content ...
Extracting locally ...


/tmp/ipykernel_3538/1301324207.py:237: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(



 DevSet1010 1010 proteins

 TestSet300 300 proteins

 TestSetNew46 46 proteins

CACHE BUILD COMPLETE
{'already_local': 1356, 'reused_old_normalized': 0, 'normalized_raw': 0}

Packing shared local cache...
Copying ONE archive to Drive...

Exact-sequence shared structures: PASS
PDB_NORM_ROOT = /content/shared_exact_pdb_v1
Persistent archive = /content/drive/MyDrive/Protein-binding-Phi/cache/shared_exact_pdb_v1.tar.gz


In [ ]:
# ============================================================
# 6. NUCLEAR-SPECIFIC FEATURE DEFINITIONS
# ============================================================
FEATURE_CACHE = CACHE_ROOT/'nuclear_explicit_v1'
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)

CHARGE = {
    'K':1.0,
    'R':1.0,
    'H':0.10,
    'D':-1.0,
    'E':-1.0,
}

GROUPS = {
    'KR': set('KR'),
    'KRH': set('KRH'),
    'DE': set('DE'),
    'POLAR': set('NQSTY'),
    'AROM': set('FWYH'),
    'HYDRO': set('AVILMFWY'),
    'HBD': set('KRHNQSTYW'),
    'HBA': set('DEHNQSTYC'),
}

BACKBONE_ATOMS = {'N','CA','C','O','OXT'}

def representative_coord(res):
    side = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in BACKBONE_ATOMS
    ]
    if side:
        return np.mean(np.stack(side),axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord,dtype=np.float32)

    atoms = [np.asarray(a.coord,dtype=np.float32) for a in res]
    if not atoms:
        raise RuntimeError('Residue has no atoms')
    return np.mean(np.stack(atoms),axis=0)

def sequence_block(seq):
    L = len(seq)
    out = np.zeros((L,32),dtype=np.float32)

    q = np.array([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    for i,a in enumerate(seq):
        out[i,0] = q[i]
        out[i,1] = float(a=='K')
        out[i,2] = float(a=='R')
        out[i,3] = float(a=='H')
        out[i,4] = float(a in GROUPS['DE'])
        out[i,5] = float(a in GROUPS['POLAR'])
        out[i,6] = float(a in GROUPS['AROM'])
        out[i,7] = float(a in GROUPS['HYDRO'])

        col = 8
        for radius in [2,4,7,10]:
            lo=max(0,i-radius)
            hi=min(L,i+radius+1)
            w=seq[lo:hi]
            qq=q[lo:hi]

            vals = [
                sum(x in GROUPS['KR'] for x in w)/len(w),
                sum(x in GROUPS['KRH'] for x in w)/len(w),
                sum(x in GROUPS['DE'] for x in w)/len(w),
                sum(x in GROUPS['POLAR'] for x in w)/len(w),
                float(np.mean(qq)),
                sum(x in GROUPS['HYDRO'] for x in w)/len(w),
            ]

            out[i,col:col+6] = vals
            col += 6

    return out

def union_find_components(indices, coords, threshold=8.0):
    if len(indices)==0:
        return {}

    idx=np.asarray(indices,dtype=int)
    cc=np.asarray(coords,dtype=np.float32)

    parent=list(range(len(idx)))

    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:
            parent[rb]=ra

    if len(idx)>1:
        tree=cKDTree(cc)
        for a,b in tree.query_pairs(threshold):
            union(a,b)

    comp=defaultdict(list)
    for j,res_idx in enumerate(idx):
        comp[find(j)].append(int(res_idx))

    size={}
    for members in comp.values():
        s=len(members)
        for r in members:
            size[r]=s

    return size

def build_3d_blocks(pdb_path, seq):
    st=PDBParser(QUIET=True).get_structure('p',str(pdb_path))
    m=next(st.get_models())
    residues=standard_residues(m['A'])

    if residue_sequence(residues)!=seq:
        raise RuntimeError(f'{pdb_path}: sequence mismatch')

    coords=np.stack(
        [representative_coord(r) for r in residues],
        axis=0
    ).astype(np.float32)

    L=len(seq)
    tree=cKDTree(coords)
    q=np.array([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    basic_idx=[i for i,a in enumerate(seq) if a in GROUPS['KRH']]
    basic_comp=union_find_components(
        basic_idx,
        coords[basic_idx] if basic_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    patch=np.zeros((L,40),dtype=np.float32)
    hbond=np.zeros((L,16),dtype=np.float32)
    topo=np.zeros((L,12),dtype=np.float32)
    arom=np.zeros((L,8),dtype=np.float32)

    # Global spatial adjacency for topology.
    neigh8=[tree.query_ball_point(coords[i],8.0) for i in range(L)]
    neigh10=[tree.query_ball_point(coords[i],10.0) for i in range(L)]

    for i,a in enumerate(seq):
        # -------- own H-bond chemistry --------
        hbond[i,0]=float(a in GROUPS['HBD'])
        hbond[i,1]=float(a in GROUPS['HBA'])
        hbond[i,2]=float(a=='R')                 # guanidinium
        hbond[i,3]=float(a=='K')                 # amine
        hbond[i,4]=float(a=='H')                 # imidazole
        hbond[i,5]=float(a in set('NQ'))         # amide
        hbond[i,6]=float(a in set('STY'))        # hydroxyl
        hbond[i,7]=float(a in set('DE'))         # carboxylate

        # -------- 3D radius summaries --------
        col=0

        for radius in [6.0,8.0,10.0,12.0]:
            ids=np.asarray(tree.query_ball_point(coords[i],radius),dtype=int)
            ids=ids[ids!=i]

            if len(ids):
                aa=[seq[j] for j in ids]
                d=np.linalg.norm(coords[ids]-coords[i][None,:],axis=1)
                qn=q[ids]

                vals=[
                    min(len(ids),40)/40.0,
                    sum(x in GROUPS['KR'] for x in aa)/len(aa),
                    sum(x in GROUPS['KRH'] for x in aa)/len(aa),
                    sum(x in GROUPS['DE'] for x in aa)/len(aa),
                    sum(x in GROUPS['POLAR'] for x in aa)/len(aa),
                    sum(x in GROUPS['HYDRO'] for x in aa)/len(aa),
                    float(np.mean(qn)),
                    float(np.tanh(np.sum(qn/(d+1.0)))),
                ]
            else:
                vals=[0.0]*8

            patch[i,col:col+8]=vals
            col+=8

        # -------- local H-bond-capable environment --------
        col=8

        for radius in [6.0,10.0]:
            ids=np.asarray(tree.query_ball_point(coords[i],radius),dtype=int)
            ids=ids[ids!=i]
            aa=[seq[j] for j in ids]

            den=max(len(aa),1)

            hbond[i,col:col+4]=[
                sum(x in GROUPS['HBD'] for x in aa)/den,
                sum(x in GROUPS['HBA'] for x in aa)/den,
                sum(x in GROUPS['KRH'] for x in aa)/den,
                sum(x in GROUPS['POLAR'] for x in aa)/den,
            ]
            col+=4

        # -------- nearest chemistry distances --------
        def nearest(chars):
            ids=[j for j,x in enumerate(seq) if x in chars and j!=i]
            if not ids:
                return 1.0
            d=np.linalg.norm(coords[ids]-coords[i][None,:],axis=1)
            return min(float(d.min())/20.0,1.0)

        patch[i,32]=nearest(GROUPS['KR'])
        patch[i,33]=nearest(GROUPS['DE'])
        patch[i,34]=nearest(GROUPS['POLAR'])
        patch[i,35]=nearest(GROUPS['AROM'])

        # -------- higher-order basic component proximity --------
        basic_near=[
            j for j in tree.query_ball_point(coords[i],10.0)
            if j in basic_comp
        ]
        max_comp=max(
            [basic_comp[j] for j in basic_near],
            default=0
        )
        patch[i,36]=min(max_comp,20)/20.0

        def centroid_distance(chars):
            ids=[j for j,x in enumerate(seq) if x in chars and
                 np.linalg.norm(coords[j]-coords[i])<=10.0]
            if not ids:
                return 1.0
            c=coords[ids].mean(axis=0)
            return min(float(np.linalg.norm(coords[i]-c))/20.0,1.0)

        patch[i,37]=centroid_distance(GROUPS['KRH'])
        patch[i,38]=centroid_distance(GROUPS['POLAR'])

        ids=np.asarray(neigh10[i],dtype=int)
        ids=ids[ids!=i]
        if len(ids):
            patch[i,39]=min(
                float(np.mean(np.linalg.norm(
                    coords[ids]-coords[i][None,:],
                    axis=1
                )))/20.0,
                1.0
            )

        # -------- topology --------
        n8=[j for j in neigh8[i] if j!=i]
        n10=[j for j in neigh10[i] if j!=i]

        topo[i,0]=min(len(n8),30)/30.0
        topo[i,1]=min(len(n10),40)/40.0

        b8=[j for j in n8 if seq[j] in GROUPS['KRH']]
        p8=[j for j in n8 if seq[j] in GROUPS['POLAR']]
        d8=[j for j in n8 if seq[j] in GROUPS['DE']]

        topo[i,2]=min(len(b8),15)/15.0
        topo[i,3]=min(len(p8),20)/20.0
        topo[i,4]=min(len(d8),15)/15.0
        topo[i,5]=np.tanh((len(b8)-len(d8))/5.0)
        topo[i,6]=min(max_comp,20)/20.0
        topo[i,7]=min(len(b8)+len(d8),20)/20.0

        if n8:
            seqsep=np.abs(np.asarray(n8)-i)
            topo[i,8]=float(np.mean(seqsep>10))
            topo[i,9]=min(float(np.mean(seqsep))/50.0,1.0)

            # Local clustering coefficient among spatial neighbors.
            if len(n8)>=2:
                cc=coords[n8]
                t2=cKDTree(cc)
                n_edges=len(t2.query_pairs(8.0))
                possible=len(n8)*(len(n8)-1)/2
                topo[i,10]=n_edges/max(possible,1)

        if b8:
            bd=np.linalg.norm(coords[b8]-coords[i][None,:],axis=1)
            topo[i,11]=min(
                (len(b8)/(1.0+float(np.mean(bd))))/3.0,
                1.0
            )

        # -------- aromatic control --------
        arom[i,0]=float(a in GROUPS['AROM'])
        arom[i,1]=float(a in set('YH'))

        for col2,radius in zip([2,3,4],[6.0,8.0,10.0]):
            ids=[
                j for j in tree.query_ball_point(coords[i],radius)
                if j!=i
            ]
            arom[i,col2]=(
                sum(seq[j] in GROUPS['AROM'] for j in ids)/max(len(ids),1)
            )

        for col2,rad in zip([5,6],[4,10]):
            lo=max(0,i-rad)
            hi=min(L,i+rad+1)
            w=seq[lo:hi]
            arom[i,col2]=sum(x in GROUPS['AROM'] for x in w)/len(w)

        arom[i,7]=nearest(GROUPS['AROM'])

    return patch,hbond,topo,arom

def protein_context(seq):
    L=len(seq)
    q=np.asarray([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    p=np.array([
        math.log1p(L)/8.0,
        sum(a in GROUPS['KR'] for a in seq)/L,
        sum(a in GROUPS['KRH'] for a in seq)/L,
        sum(a in GROUPS['DE'] for a in seq)/L,
        sum(a in GROUPS['POLAR'] for a in seq)/L,
        sum(a in GROUPS['HYDRO'] for a in seq)/L,
        sum(a in GROUPS['AROM'] for a in seq)/L,
        float(np.mean(q)),
    ],dtype=np.float32)

    return np.repeat(p[None,:],L,axis=0)


In [ ]:
# ============================================================
# 7. GENERATE / CACHE / ATTACH NUCLEAR-SPECIFIC BLOCKS
# ============================================================
NEW_BLOCKS = {
    'NA_SEQELECTRO32':32,
    'NA_PATCH3D40':40,
    'NA_HBOND16':16,
    'NA_TOPO12':12,
    'NA_PROTCTX8':8,
    'NA_AROMCTRL8':8,
}

if RUN_FEATURE_GENERATION:
    for split,df in SPLIT_DF.items():
        outdir=FEATURE_CACHE/split
        outdir.mkdir(parents=True,exist_ok=True)

        for n,r in enumerate(df.itertuples(),start=1):
            fp=outdir/f'{r.prot_id}.npz'

            valid=False
            if fp.exists() and not FORCE_REBUILD_NA_FEATURES:
                try:
                    z=np.load(fp)
                    valid=all(
                        z[k].shape==(len(r.sequence),d)
                        for k,d in NEW_BLOCKS.items()
                    )
                except Exception:
                    valid=False

            if not valid:
                seqb=sequence_block(r.sequence)
                patch,hbond,topo,arom=build_3d_blocks(
                    PDB_NORM_ROOT/split/f'{r.prot_id}.pdb',
                    r.sequence
                )
                prot=protein_context(r.sequence)

                np.savez_compressed(
                    fp,
                    NA_SEQELECTRO32=seqb,
                    NA_PATCH3D40=patch,
                    NA_HBOND16=hbond,
                    NA_TOPO12=topo,
                    NA_PROTCTX8=prot,
                    NA_AROMCTRL8=arom,
                )

            if n%250==0:
                print(split,n,'/',len(df))

# Attach.
for split,df in SPLIT_DF.items():
    store=ctx.stores[STORE_KEY[split]]
    outdir=FEATURE_CACHE/split

    for r in df.itertuples():
        z=np.load(outdir/f'{r.prot_id}.npz')

        for k,d in NEW_BLOCKS.items():
            a=z[k].astype(np.float32,copy=False)
            if a.shape!=(len(r.sequence),d):
                raise RuntimeError(f'{split}/{r.prot_id}/{k}: {a.shape}')
            set_block(store,r.prot_id,k,a)

print('New nuclear blocks attached.')
display(reporting.dimension_table(ctx.stores['train']))

audit_blocks = [
    'ESM','ACCESS','ELEC3D','ENV','SEQMOTIF','SPATIAL',
    *NEW_BLOCKS.keys(),
]
if EVO_AVAILABLE:
    audit_blocks.append('EVO')
if ESMIF_AVAILABLE:
    audit_blocks.append('ESMIF')

display(reporting.validate_no_direct_label_features(
    ctx.stores,
    audit_blocks
))


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300
New nuclear blocks attached.


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
16,EVO,48
3,LOCAL,8


,split,block,missing_proteins,mean_finite_fraction
0,train,ESM,0,1.0
1,train,ACCESS,0,1.0
2,train,ELEC3D,0,1.0
3,train,ENV,0,1.0
4,train,SEQMOTIF,0,1.0
5,train,SPATIAL,0,1.0
6,train,NA_SEQELECTRO32,0,1.0
7,train,NA_PATCH3D40,0,1.0
8,train,NA_HBOND16,0,1.0
9,train,NA_TOPO12,0,1.0


# Part B — Same-MLP mechanism discovery

Feature discovery is deliberately separated from architecture discovery.

Every representation in the following stages uses the same GEAR MLP and the same official five protein folds.


In [ ]:
# ============================================================
# 8. STAGE A — SINGLE-FAMILY MLP SCREEN (DEV ONLY)
# ============================================================
FAMILY_GRID = {
    'ESM':['ESM'],

    # Existing audit-supported controls
    'ESM+ACCESS':['ESM','ACCESS'],
    'ESM+ELEC3D':['ESM','ELEC3D'],
    'ESM+ENV':['ESM','ENV'],
    'ESM+SEQMOTIF':['ESM','SEQMOTIF'],
    'ESM+Structure':['ESM','SPATIAL','LOCAL'],

    # New nuclear-specific blocks
    'ESM+NA_SEQELECTRO':['ESM','NA_SEQELECTRO32'],
    'ESM+NA_PATCH3D':['ESM','NA_PATCH3D40'],
    'ESM+NA_HBOND':['ESM','NA_HBOND16'],
    'ESM+NA_TOPO':['ESM','NA_TOPO12'],
    'ESM+NA_PROTCTX':['ESM','NA_PROTCTX8'],

    # Explicit control: aromatic signal must earn its place
    'ESM+NA_AROM_CTRL':['ESM','NA_AROMCTRL8'],
}

if EVO_AVAILABLE:
    FAMILY_GRID['ESM+EVO']=['ESM','EVO']

if ESMIF_AVAILABLE:
    FAMILY_GRID['ESM+ESMIF']=['ESM','ESMIF']

if RUN_FAMILY_SCREEN:
    family_cv,family_folds=suite.run_dev_cv_only(
        ctx,
        FAMILY_GRID,
        models=('MLP',),
        run_version='nuclear_v1_01_family_mlp',
    )

    family_cv.to_csv(RUN_ROOT/'family_cv.csv',index=False)
    family_folds.to_csv(RUN_ROOT/'family_folds.csv',index=False)
else:
    family_cv=pd.read_csv(RUN_ROOT/'family_cv.csv')
    family_folds=pd.read_csv(RUN_ROOT/'family_folds.csv')

base_score=float(
    family_cv.loc[
        family_cv.representation.eq('ESM'),
        'CV_AUPRC'
    ].iloc[0]
)

family_rank=family_cv.copy()
family_rank['delta_vs_ESM']=family_rank.CV_AUPRC-base_score
family_rank=family_rank.sort_values('CV_AUPRC',ascending=False)

display(family_rank)
family_rank.to_csv(RUN_ROOT/'family_ranked.csv',index=False)

print('ESM baseline:',base_score)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS 1
REUSE MLP ESM+ACCESS 2
REUSE MLP ESM+ACCESS 3
REUSE MLP ESM+ACCESS 4
REUSE MLP ESM+ACCESS 5
REUSE MLP ESM+ELEC3D 1
REUSE MLP ESM+ELEC3D 2
REUSE MLP ESM+ELEC3D 3
REUSE MLP ESM+ELEC3D 4
REUSE MLP ESM+ELEC3D 5
REUSE MLP ESM+ENV 1
REUSE MLP ESM+ENV 2
REUSE MLP ESM+ENV 3
REUSE MLP ESM+ENV 4
REUSE MLP ESM+ENV 5
REUSE MLP ESM+SEQMOTIF 1
REUSE MLP ESM+SEQMOTIF 2
REUSE MLP ESM+SEQMOTIF 3
REUSE MLP ESM+SEQMOTIF 4
REUSE MLP ESM+SEQMOTIF 5
REUSE MLP ESM+Structure 1
REUSE MLP ESM+Structure 2
REUSE MLP ESM+Structure 3
REUSE MLP ESM+Structure 4
REUSE MLP ESM+Structure 5
REUSE MLP ESM+NA_SEQELECTRO 1
REUSE MLP ESM+NA_SEQELECTRO 2
REUSE MLP ESM+NA_SEQELECTRO 3
REUSE MLP ESM+NA_SEQELECTRO 4
REUSE MLP ESM+NA_SEQELECTRO 5
REUSE MLP ESM+NA_PATCH3D 1
REUSE MLP ESM+NA_PATCH3D 2
REUSE MLP ESM+NA_PATCH3D 3
REUSE MLP ESM+NA_PATCH3D 4
REUSE MLP ESM+NA_PATCH3D 5
REUSE MLP ESM+NA_HBOND 1
REUSE MLP ESM+NA_HBOND 2


,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_ESM
7,MLP,ESM+NA_PATCH3D,0.472381,0.095097,0.952271,0.020195
8,MLP,ESM+NA_PROTCTX,0.466794,0.069785,0.949923,0.014608
1,MLP,ESM+ACCESS,0.463505,0.089183,0.950655,0.011319
10,MLP,ESM+NA_TOPO,0.462437,0.088621,0.949274,0.010250
12,MLP,ESM+Structure,0.462125,0.086578,0.949836,0.009939
4,MLP,ESM+EVO,0.459919,0.088635,0.945082,0.007732
6,MLP,ESM+NA_HBOND,0.459334,0.084958,0.949310,0.007147
2,MLP,ESM+ELEC3D,0.458180,0.080072,0.945972,0.005994
9,MLP,ESM+NA_SEQELECTRO,0.455958,0.082945,0.946915,0.003772
3,MLP,ESM+ENV,0.455212,0.081614,0.947952,0.003025


ESM baseline: 0.4521865239018867


In [ ]:
# ============================================================
# 9. STAGE B — AUDIT-DRIVEN MECHANISM GROUPS (DEV ONLY)
# ============================================================
MECHANISM_GRID = {
    'ESM':['ESM'],

    # Data-driven compact anchor from the audit.
    'ESM+ACCESS+ELEC3D':
        ['ESM','ACCESS','ELEC3D'],

    # Main nuclear mechanism groups.
    'ESM+BACKBONE_ATTRACTION':
        ['ESM','ACCESS','ELEC3D','NA_SEQELECTRO32'],

    'ESM+HBOND_RECOGNITION':
        ['ESM','NA_HBOND16'],

    'ESM+PATCH_ORGANIZATION':
        ['ESM','ACCESS','NA_PATCH3D40','NA_TOPO12'],

    'ESM+ELECTRO_PATCH':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
        ],

    'ESM+ELECTRO_PATCH_HBOND':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
            'NA_HBOND16',
        ],

    # Aromatic is a control, not a prior assumption.
    'ESM+ELECTRO_PATCH_HBOND+AROMCTRL':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
            'NA_HBOND16','NA_AROMCTRL8',
        ],
}

if EVO_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+EVO']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'EVO',
    ]

if ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+ESMIF']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'ESMIF',
    ]

if EVO_AVAILABLE and ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+EVO+ESMIF']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'EVO','ESMIF',
    ]

if RUN_COMBINATION_LADDER:
    combo_cv,combo_folds=suite.run_dev_cv_only(
        ctx,
        MECHANISM_GRID,
        models=('MLP',),
        run_version='nuclear_v1_02_mechanism_combo_mlp',
    )

    combo_cv.to_csv(RUN_ROOT/'mechanism_combo_cv.csv',index=False)
    combo_folds.to_csv(RUN_ROOT/'mechanism_combo_folds.csv',index=False)
else:
    combo_cv=pd.read_csv(RUN_ROOT/'mechanism_combo_cv.csv')
    combo_folds=pd.read_csv(RUN_ROOT/'mechanism_combo_folds.csv')

combo_rank=combo_cv.sort_values('CV_AUPRC',ascending=False).reset_index(drop=True)
display(combo_rank)

BEST_REP_NAME=str(combo_rank.iloc[0].representation)
BEST_REP_BLOCKS=list(MECHANISM_GRID[BEST_REP_NAME])
BEST_REP_SCORE=float(combo_rank.iloc[0].CV_AUPRC)

print('LOCKED REPRESENTATION:',BEST_REP_NAME)
print('Blocks:',BEST_REP_BLOCKS)
print('Dev CV AUPRC:',BEST_REP_SCORE)

(RUN_ROOT/'locked_representation.json').write_text(
    json.dumps({
        'representation':BEST_REP_NAME,
        'blocks':BEST_REP_BLOCKS,
        'dev_cv_auprc':BEST_REP_SCORE,
        'selection_source':'DevSet1010 only',
    },indent=2)
)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS+ELEC3D 1
REUSE MLP ESM+ACCESS+ELEC3D 2
REUSE MLP ESM+ACCESS+ELEC3D 3
REUSE MLP ESM+ACCESS+ELEC3D 4
REUSE MLP ESM+ACCESS+ELEC3D 5
REUSE MLP ESM+BACKBONE_ATTRACTION 1
REUSE MLP ESM+BACKBONE_ATTRACTION 2
REUSE MLP ESM+BACKBONE_ATTRACTION 3
REUSE MLP ESM+BACKBONE_ATTRACTION 4
REUSE MLP ESM+BACKBONE_ATTRACTION 5
REUSE MLP ESM+HBOND_RECOGNITION 1
REUSE MLP ESM+HBOND_RECOGNITION 2
REUSE MLP ESM+HBOND_RECOGNITION 3
REUSE MLP ESM+HBOND_RECOGNITION 4
REUSE MLP ESM+HBOND_RECOGNITION 5
REUSE MLP ESM+PATCH_ORGANIZATION 1
REUSE MLP ESM+PATCH_ORGANIZATION 2
REUSE MLP ESM+PATCH_ORGANIZATION 3
REUSE MLP ESM+PATCH_ORGANIZATION 4
REUSE MLP ESM+PATCH_ORGANIZATION 5
REUSE MLP ESM+ELECTRO_PATCH 1
REUSE MLP ESM+ELECTRO_PATCH 2
REUSE MLP ESM+ELECTRO_PATCH 3
REUSE MLP ESM+ELECTRO_PATCH 4
REUSE MLP ESM+ELECTRO_PATCH 5
REUSE MLP ESM+ELECTRO_PATCH_HBOND 1
REUSE MLP ESM+ELECTRO_PATCH_HBOND 2
REUSE MLP ESM+ELECTRO_P

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
0,MLP,ESM+PATCH_ORGANIZATION,0.474114,0.082243,0.953786
1,MLP,ESM+ELECTRO_PATCH_HBOND+AROMCTRL,0.472061,0.081735,0.951585
2,MLP,ESM+BACKBONE_ATTRACTION,0.471199,0.089175,0.951923
3,MLP,ESM+ACCESS+ELEC3D,0.468935,0.081552,0.950090
4,MLP,ESM+ELECTRO_PATCH_HBOND,0.468750,0.086151,0.953248
5,MLP,ESM+ELECTRO_PATCH_HBOND+EVO,0.466043,0.075069,0.953128
6,MLP,ESM+ELECTRO_PATCH,0.465813,0.082071,0.955128
7,MLP,ESM+HBOND_RECOGNITION,0.459334,0.084958,0.949310
8,MLP,ESM,0.452187,0.095384,0.950662


LOCKED REPRESENTATION: ESM+PATCH_ORGANIZATION
Blocks: ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12']
Dev CV AUPRC: 0.47411433584661966


208

# Part C — Multimodal architecture screen

After the feature representation is frozen, architecture is allowed to change.

Modalities are grouped semantically:

- **ESM** — PLM representation
- **ELECTRO** — ACCESS / ELEC3D / sequence electrostatics
- **PATCH** — 3D patch + patch topology
- **CHEM** — H-bond / aromatic-control chemistry
- **BIO** — EVO / motif / global protein context / learned structure when selected

Candidates:
- ConcatMLP
- ProjectedConcat
- GatedFusion
- ModalityAttention
- PatchResidualFusion
- LatentNAExpertMoE

The patch branch is residual, reflecting the Test46 observation that nuclear interfaces can be much smaller than those in Dev/Test300.


In [ ]:
# ============================================================
# 10. PACK LOCKED REPRESENTATION INTO SEMANTIC MODALITIES
# ============================================================
def modality_spec(blocks):
    blocks=list(blocks)
    spec={}

    if 'ESM' in blocks:
        spec['ESM']=['ESM']

    electro=[
        b for b in [
            'ACCESS','ELEC3D','NA_SEQELECTRO32'
        ]
        if b in blocks
    ]
    if electro:
        spec['ELECTRO']=electro

    patch=[
        b for b in [
            'NA_PATCH3D40','NA_TOPO12'
        ]
        if b in blocks
    ]
    if patch:
        spec['PATCH']=patch

    chem=[
        b for b in [
            'NA_HBOND16','NA_AROMCTRL8'
        ]
        if b in blocks
    ]
    if chem:
        spec['CHEM']=chem

    bio=[
        b for b in [
            'EVO','SEQMOTIF','NA_PROTCTX8',
            'SPATIAL','LOCAL','ESMIF'
        ]
        if b in blocks
    ]
    if bio:
        spec['BIO']=bio

    # Any selected block not yet assigned goes to OTHER.
    assigned={x for v in spec.values() for x in v}
    other=[b for b in blocks if b not in assigned]
    if other:
        spec['OTHER']=other

    return spec

MODALITY_SPEC=modality_spec(BEST_REP_BLOCKS)
print('MODALITY_SPEC =',MODALITY_SPEC)

def concat_blocks(store,pid,blocks):
    xs=[
        get_block(store,pid,b).astype(np.float32,copy=False)
        for b in blocks
    ]

    lens={x.shape[0] for x in xs}
    if len(lens)!=1:
        raise RuntimeError(f'{pid}: block length mismatch')

    return np.concatenate(xs,axis=1)

def build_pack(split):
    df=SPLIT_DF[split]
    store=ctx.stores[STORE_KEY[split]]

    mods={k:[] for k in MODALITY_SPEC}
    yy=[]
    pids=[]
    ridx=[]

    for r in df.itertuples():
        L=len(r.sequence)

        for m,blocks in MODALITY_SPEC.items():
            mods[m].append(
                concat_blocks(store,r.prot_id,blocks)
            )

        yy.append(np.asarray(r.nuclear_binding,dtype=np.float32))
        pids.extend([str(r.prot_id)]*L)
        ridx.extend(range(L))

    mods={
        k:np.concatenate(v,axis=0).astype(
            np.float16 if k in {'ESM','BIO'} else np.float32,
            copy=False
        )
        for k,v in mods.items()
    }

    return {
        'mods':mods,
        'y':np.concatenate(yy).astype(np.float32),
        'pids':np.asarray(pids,dtype=object),
        'ridx':np.asarray(ridx,dtype=np.int32),
    }

PACKS={s:build_pack(s) for s in SPLIT_DF}

DIMS={
    k:v.shape[1]
    for k,v in PACKS['DevSet1010']['mods'].items()
}

for split,p in PACKS.items():
    print(
        split,
        {k:v.shape for k,v in p['mods'].items()},
        'y=',p['y'].shape,
        'positive=',int(p['y'].sum())
    )

print('Modality dims:',DIMS)


MODALITY_SPEC = {'ESM': ['ESM'], 'ELECTRO': ['ACCESS'], 'PATCH': ['NA_PATCH3D40', 'NA_TOPO12']}
DevSet1010 {'ESM': (170181, 1280), 'ELECTRO': (170181, 10), 'PATCH': (170181, 52)} y= (170181,) positive= 2689
TestSet300 {'ESM': (62689, 1280), 'ELECTRO': (62689, 10), 'PATCH': (62689, 52)} y= (62689,) positive= 1470
TestSetNew46 {'ESM': (6227, 1280), 'ELECTRO': (6227, 10), 'PATCH': (6227, 52)} y= (6227,) positive= 77
Modality dims: {'ESM': 1280, 'ELECTRO': 10, 'PATCH': 52}


In [ ]:
# ============================================================
# 11. MULTIMODAL MODELS
# ============================================================
class ConcatMLP(nn.Module):
    def __init__(self,dims,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.norm=nn.ModuleDict({
            k:nn.LayerNorm(d)
            for k,d in dims.items()
        })
        d=sum(dims.values())

        self.head=nn.Sequential(
            nn.Linear(d,512),nn.GELU(),nn.Dropout(drop),
            nn.Linear(512,128),nn.GELU(),nn.Dropout(drop),
            nn.Linear(128,1),
        )

    def forward(self,x):
        z=torch.cat(
            [self.norm[k](x[k]) for k in self.keys],
            dim=-1
        )
        return self.head(z).squeeze(-1)

class ProjectedConcat(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.head=nn.Sequential(
            nn.Linear(latent*len(dims),256),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(256,1)
        )

    def forward(self,x):
        z=torch.cat(
            [self.enc[k](x[k]) for k in self.keys],
            dim=-1
        )
        return self.head(z).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def encode(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)

        return (Z*G).sum(dim=1),G.squeeze(-1)

    def forward(self,x):
        h,_=self.encode(x)
        return self.head(h).squeeze(-1)

class ModalityAttention(nn.Module):
    def __init__(self,dims,latent=192,heads=6,drop=0.2):
        super().__init__()
        assert latent%heads==0

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.attn=nn.MultiheadAttention(
            latent,
            heads,
            dropout=drop,
            batch_first=True
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        Z=torch.stack(
            [self.enc[k](x[k]) for k in self.keys],
            dim=1
        )
        A,_=self.attn(Z,Z,Z,need_weights=False)
        h=(Z+A).mean(dim=1)
        return self.head(h).squeeze(-1)

class PatchResidualFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        if 'PATCH' not in dims:
            raise ValueError('PatchResidualFusion requires PATCH modality')

        self.base_keys=sorted(
            [k for k in dims if k!='PATCH']
        )

        self.base_enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
            if k!='PATCH'
        })

        self.patch_enc=nn.Sequential(
            nn.LayerNorm(dims['PATCH']),
            nn.Linear(dims['PATCH'],latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
        )

        self.base_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in self.base_keys
        })

        # sigmoid(-2.944) ≈ 0.05
        self.alpha_logit=nn.Parameter(
            torch.tensor(-2.944,dtype=torch.float32)
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.base_keys:
            z=self.base_enc[k](x[k])
            zz.append(z)
            gg.append(self.base_gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)
        h=(Z*G).sum(dim=1)

        patch=self.patch_enc(x['PATCH'])
        alpha=torch.sigmoid(self.alpha_logit)

        h=h+alpha*patch
        return self.head(h).squeeze(-1)

class LatentNAExpertMoE(nn.Module):
    def __init__(self,dims,latent=192,experts=4,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.mod_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.shared=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU()
        )

        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
            for _ in range(experts)
        ])

        self.expert_gate=nn.Linear(latent,experts)

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.mod_gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)

        h=self.shared((Z*G).sum(dim=1))

        logits=torch.cat(
            [e(h) for e in self.experts],
            dim=1
        )
        weights=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (logits*weights).sum(dim=1)

ARCH_FACTORY = {
    'ConcatMLP':lambda dims:ConcatMLP(dims),
    'ProjectedConcat':lambda dims:ProjectedConcat(dims),
    'GatedFusion':lambda dims:GatedFusion(dims),
    'ModalityAttention':lambda dims:ModalityAttention(dims),
    'LatentNAExpertMoE':lambda dims:LatentNAExpertMoE(dims),
}

if 'PATCH' in DIMS:
    ARCH_FACTORY['PatchResidualFusion']=lambda dims:PatchResidualFusion(dims)

print('Architectures:',list(ARCH_FACTORY))


Architectures: ['ConcatMLP', 'ProjectedConcat', 'GatedFusion', 'ModalityAttention', 'LatentNAExpertMoE', 'PatchResidualFusion']


In [ ]:
# ============================================================
# 12. CUSTOM CV TRAINER — OFFICIAL PROTEIN FOLDS, RESUME SAFE
# ============================================================
class MultiArrayDataset(Dataset):
    def __init__(self,mods,y):
        self.mods=mods
        self.y=y
        self.keys=sorted(mods)

    def __len__(self):
        return len(self.y)

    def __getitem__(self,i):
        x={
            k:torch.from_numpy(
                np.asarray(self.mods[k][i],dtype=np.float32)
            )
            for k in self.keys
        }
        return x,torch.tensor(self.y[i],dtype=torch.float32)

def collate_multi(batch):
    keys=sorted(batch[0][0])
    x={
        k:torch.stack([b[0][k] for b in batch],dim=0)
        for k in keys
    }
    y=torch.stack([b[1] for b in batch])
    return x,y

def ap(y,p):
    return float(average_precision_score(y,p))

def predict_flat(model,dataset,indices=None,batch_size=8192):
    ds=dataset if indices is None else Subset(
        dataset,
        indices.tolist() if isinstance(indices,np.ndarray) else indices
    )

    loader=DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    out=[]
    model.eval()

    with torch.no_grad():
        for x,_ in loader:
            x={
                k:v.to(DEVICE,non_blocking=True)
                for k,v in x.items()
            }
            out.append(
                torch.sigmoid(model(x)).detach().cpu().numpy()
            )

    return np.concatenate(out)

CUSTOM_STATE_ROOT=RUN_ROOT/'custom_states'
CUSTOM_STATE_ROOT.mkdir(parents=True,exist_ok=True)

def train_one_arch_fold(arch,pack,train_idx,val_idx,fold):
    torch.manual_seed(SEED+fold)
    np.random.seed(SEED+fold)

    ds=MultiArrayDataset(pack['mods'],pack['y'])

    train_ds=Subset(ds,train_idx.tolist())
    val_ds=Subset(ds,val_idx.tolist())

    train_loader=DataLoader(
        train_ds,
        batch_size=CUSTOM_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    model=ARCH_FACTORY[arch](DIMS).to(DEVICE)

    ytr=pack['y'][train_idx]
    npos=max(float(ytr.sum()),1.0)
    nneg=max(float(len(ytr)-ytr.sum()),1.0)

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CUSTOM_LR,
        weight_decay=CUSTOM_WEIGHT_DECAY
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(1,CUSTOM_EPOCHS+1):
        model.train()
        losses=[]

        for x,y in train_loader:
            x={k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
            y=y.to(DEVICE,non_blocking=True)

            opt.zero_grad(set_to_none=True)

            logits=model(x)
            loss=criterion(logits,y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()

            losses.append(float(loss.detach().cpu()))

        vp=predict_flat(model,ds,val_idx)
        vy=pack['y'][val_idx]
        val_ap=ap(vy,vp)

        if val_ap>best_ap+1e-5:
            best_ap=val_ap
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad=0
        else:
            bad+=1

        if epoch==1 or epoch%5==0:
            print(
                arch,'fold',fold,'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5)
            )

        if bad>=CUSTOM_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_state

def run_arch_cv(arch):
    pack=PACKS['DevSet1010']
    pids=pack['pids']
    y=pack['y']

    ds=MultiArrayDataset(pack['mods'],pack['y'])
    oof=np.full(len(y),np.nan,dtype=np.float32)

    rows=[]
    states={}

    adir=CUSTOM_STATE_ROOT/arch
    adir.mkdir(parents=True,exist_ok=True)

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(val_ids)

        val_idx=np.flatnonzero(
            np.array([p in val_set for p in pids])
        )
        train_idx=np.flatnonzero(
            np.array([p not in val_set for p in pids])
        )

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        reuse=False

        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta=json.loads(meta_fp.read_text())
                pp=np.load(pred_fp)

                reuse=(
                    meta.get('representation')==BEST_REP_NAME
                    and meta.get('n_val')==len(val_idx)
                    and pp.shape==(len(val_idx),)
                    and np.all(np.isfinite(pp))
                )
            except Exception:
                reuse=False

        if reuse:
            print(arch,'fold',fold,'REUSE')
            state=torch.load(state_fp,map_location='cpu')
            pred=np.load(pred_fp).astype(np.float32)
        else:
            print(arch,'fold',fold,'TRAIN')
            model,state=train_one_arch_fold(
                arch,pack,train_idx,val_idx,fold
            )
            pred=predict_flat(model,ds,val_idx).astype(np.float32)

            torch.save(state,state_fp)
            np.save(pred_fp,pred)
            meta_fp.write_text(json.dumps({
                'representation':BEST_REP_NAME,
                'blocks':BEST_REP_BLOCKS,
                'n_val':int(len(val_idx)),
            },indent=2))

            del model
            torch.cuda.empty_cache()

        states[fold]=state
        oof[val_idx]=pred

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(y[val_idx],pred),
            'n_val':len(val_idx),
            'n_positive':int(y[val_idx].sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch}: incomplete OOF')

    f=pd.DataFrame(rows)

    return (
        {
            'architecture':arch,
            'CV_AUPRC_mean':float(f.AUPRC.mean()),
            'CV_AUPRC_std':float(f.AUPRC.std(ddof=1)),
            'OOF_AUPRC':ap(y,oof),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 13. RUN MULTIMODAL ARCHITECTURE SCREEN — DEV ONLY
# ============================================================
ARCH_RESULTS={}
ARCH_FOLDS=[]
ARCH_OOF={}
ARCH_STATES={}

if RUN_CUSTOM_ARCHITECTURES:
    for arch in ARCH_FACTORY:
        print('\n'+'='*80)
        print('ARCH:',arch)
        print('='*80)

        s,f,o,st=run_arch_cv(arch)

        ARCH_RESULTS[arch]=s
        ARCH_FOLDS.append(f)
        ARCH_OOF[arch]=o
        ARCH_STATES[arch]=st

arch_summary=pd.DataFrame(
    ARCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(drop=True)

arch_folds=pd.concat(
    ARCH_FOLDS,
    ignore_index=True
)

arch_summary.to_csv(RUN_ROOT/'architecture_cv.csv',index=False)
arch_folds.to_csv(RUN_ROOT/'architecture_folds.csv',index=False)

display(arch_summary)

concat_score=float(
    arch_summary.loc[
        arch_summary.architecture.eq('ConcatMLP'),
        'CV_AUPRC_mean'
    ].iloc[0]
)

best_arch=str(arch_summary.iloc[0].architecture)
best_arch_score=float(arch_summary.iloc[0].CV_AUPRC_mean)

if best_arch_score-concat_score >= MIN_MEAN_AUPRC_GAIN:
    LOCKED_ARCH=best_arch
else:
    LOCKED_ARCH='ConcatMLP'

LOCKED_ARCH_SCORE=float(
    arch_summary.loc[
        arch_summary.architecture.eq(LOCKED_ARCH),
        'CV_AUPRC_mean'
    ].iloc[0]
)

print('LOCKED ARCH:',LOCKED_ARCH,LOCKED_ARCH_SCORE)

(RUN_ROOT/'locked_architecture.json').write_text(
    json.dumps({
        'architecture':LOCKED_ARCH,
        'dev_cv_auprc':LOCKED_ARCH_SCORE,
        'concat_cv_auprc':concat_score,
        'best_raw_architecture':best_arch,
        'best_raw_cv_auprc':best_arch_score,
    },indent=2)
)



ARCH: ConcatMLP
ConcatMLP fold 1 REUSE
ConcatMLP fold 2 REUSE
ConcatMLP fold 3 REUSE
ConcatMLP fold 4 REUSE
ConcatMLP fold 5 REUSE

ARCH: ProjectedConcat
ProjectedConcat fold 1 REUSE
ProjectedConcat fold 2 REUSE
ProjectedConcat fold 3 REUSE
ProjectedConcat fold 4 REUSE
ProjectedConcat fold 5 REUSE

ARCH: GatedFusion
GatedFusion fold 1 REUSE
GatedFusion fold 2 REUSE
GatedFusion fold 3 REUSE
GatedFusion fold 4 REUSE
GatedFusion fold 5 REUSE

ARCH: ModalityAttention
ModalityAttention fold 1 REUSE
ModalityAttention fold 2 TRAIN
ModalityAttention fold 2 epoch 1 loss 0.365 val_AP 0.46319 best 0.46319
ModalityAttention fold 2 epoch 5 loss 0.0994 val_AP 0.52976 best 0.54368
ModalityAttention fold 2 epoch 10 loss 0.0382 val_AP 0.49383 best 0.54368
ModalityAttention fold 3 TRAIN
ModalityAttention fold 3 epoch 1 loss 0.3652 val_AP 0.30159 best 0.30159
ModalityAttention fold 3 epoch 5 loss 0.0816 val_AP 0.34843 best 0.38331
ModalityAttention fold 4 TRAIN
ModalityAttention fold 4 epoch 1 loss 0.33

,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,ProjectedConcat,0.467308,0.089055,0.474557
1,ConcatMLP,0.465391,0.082665,0.444775
2,ModalityAttention,0.463773,0.090579,0.453643
3,GatedFusion,0.461928,0.093732,0.459380
4,LatentNAExpertMoE,0.457301,0.093736,0.454949
5,PatchResidualFusion,0.455773,0.092193,0.426168


LOCKED ARCH: ConcatMLP 0.46539084222578725


204

# Part D — Nuclear surface-patch hypergraph

Generic residue graphs already underperformed in the previous nuclear experiments.

V1 therefore tests a task-specific graph unit:

```text
residues
   ↓
candidate cationic/polar 3D surface patches
   ↓
patch representation
   ↓
patch → residue residual message
```

Candidate patches are label-independent and seeded by K/R/H/N/Q/S/T/Y residues. All nearby residues within 10 Å become patch members.

The hypergraph correction starts with a learned residual weight near 0.05, so the patch branch cannot dominate the strong local predictor at initialization.


In [ ]:
# ============================================================
# 14. BUILD / CACHE NUCLEAR CANDIDATE PATCH HYPERGRAPHS
# ============================================================
PATCH_CACHE=CACHE_ROOT/'patch_hypergraph_v1'
PATCH_CACHE.mkdir(parents=True,exist_ok=True)

PATCH_SEEDS=set('KRHNQSTY')

def structure_coords(split,pid,seq):
    path=PDB_NORM_ROOT/split/f'{pid}.pdb'
    st=PDBParser(QUIET=True).get_structure('p',str(path))
    m=next(st.get_models())
    rr=standard_residues(m['A'])

    if residue_sequence(rr)!=seq:
        raise RuntimeError(f'{split}/{pid}: normalized PDB mismatch')

    return np.stack(
        [representative_coord(r) for r in rr],
        axis=0
    ).astype(np.float32)

def patch_static_features(seq,coords,members):
    members=np.asarray(members,dtype=int)
    aa=[seq[i] for i in members]
    cc=coords[members]
    n=len(members)

    q=np.asarray(
        [CHARGE.get(x,0.0) for x in aa],
        dtype=np.float32
    )

    centroid=cc.mean(axis=0)
    radial=np.linalg.norm(cc-centroid[None,:],axis=1)

    pair=[]
    if n>=2:
        for i in range(n):
            for j in range(i+1,n):
                pair.append(float(np.linalg.norm(cc[i]-cc[j])))

    pair=np.asarray(pair if pair else [0.0],dtype=np.float32)

    probs=[]
    cnt=Counter(aa)
    for x in cnt.values():
        probs.append(x/n)

    entropy=(
        -sum(p*math.log(p+1e-12) for p in probs)
        / math.log(20)
    )

    f=np.zeros(18,dtype=np.float32)

    f[0]=min(n,40)/40.0
    f[1]=sum(x in GROUPS['KR'] for x in aa)/n
    f[2]=sum(x in GROUPS['KRH'] for x in aa)/n
    f[3]=sum(x in GROUPS['DE'] for x in aa)/n
    f[4]=sum(x in GROUPS['POLAR'] for x in aa)/n
    f[5]=sum(x in GROUPS['AROM'] for x in aa)/n
    f[6]=sum(x in GROUPS['HYDRO'] for x in aa)/n
    f[7]=float(np.mean(q))
    f[8]=sum(x in GROUPS['HBD'] for x in aa)/n
    f[9]=sum(x in GROUPS['HBA'] for x in aa)/n

    f[10]=min(float(radial.mean())/12.0,1.0)
    f[11]=min(float(radial.std())/8.0,1.0)
    f[12]=min(float(radial.max())/15.0,1.0)

    f[13]=min((members.max()-members.min())/100.0,1.0)
    f[14]=min(float(pair.mean())/15.0,1.0)
    f[15]=min(float(pair.std())/10.0,1.0)
    f[16]=float(entropy)
    f[17]=float(np.tanh(
        (
            sum(x in GROUPS['KRH'] for x in aa)
            - sum(x in GROUPS['DE'] for x in aa)
        )/5.0
    ))

    return f

def build_patch_graph(split,pid,seq):
    coords=structure_coords(split,pid,seq)
    tree=cKDTree(coords)

    candidates={}

    for seed,a in enumerate(seq):
        if a not in PATCH_SEEDS:
            continue

        members=tuple(sorted(
            tree.query_ball_point(coords[seed],PATCH_RADIUS_A)
        ))

        if len(members)<4:
            continue

        if not any(seq[j] in GROUPS['KRH'] for j in members):
            continue

        f=patch_static_features(seq,coords,members)

        # Label-independent ranking score for cationic/polar recognition plausibility.
        score=(
            2.0*f[2]
            + 0.7*f[4]
            - 0.7*f[3]
            + 0.2*f[0]
        )

        old=candidates.get(members)
        if old is None or score>old[0]:
            candidates[members]=(score,f)

    ranked=sorted(
        candidates.items(),
        key=lambda kv:kv[1][0],
        reverse=True
    )[:PATCH_MAX_PER_PROTEIN]

    edge_res=[]
    edge_patch=[]
    patch_feat=[]

    for pidx,(members,(_,feat)) in enumerate(ranked):
        patch_feat.append(feat)

        for r in members:
            edge_res.append(int(r))
            edge_patch.append(int(pidx))

    return (
        np.asarray(edge_res,dtype=np.int64),
        np.asarray(edge_patch,dtype=np.int64),
        np.asarray(patch_feat,dtype=np.float32).reshape(-1,18)
    )

PATCH_RAM={}
PATCH_AUDIT=[]

for split,df in SPLIT_DF.items():
    outdir=PATCH_CACHE/split
    outdir.mkdir(parents=True,exist_ok=True)

    for n,r in enumerate(df.itertuples(),start=1):
        fp=outdir/f'{r.prot_id}.npz'

        valid=False

        if fp.exists() and not FORCE_REBUILD_PATCHES:
            try:
                z=np.load(fp)
                valid=(
                    z['patch_feat'].ndim==2
                    and z['patch_feat'].shape[1]==18
                    and len(z['edge_res'])==len(z['edge_patch'])
                )
            except Exception:
                valid=False

        if not valid:
            er,ep,pf=build_patch_graph(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.savez_compressed(
                fp,
                edge_res=er,
                edge_patch=ep,
                patch_feat=pf
            )

        z=np.load(fp)

        er=z['edge_res'].astype(np.int64)
        ep=z['edge_patch'].astype(np.int64)
        pf=z['patch_feat'].astype(np.float32)

        PATCH_RAM[(split,str(r.prot_id))]=(er,ep,pf)

        covered=len(np.unique(er)) if len(er) else 0

        PATCH_AUDIT.append({
            'dataset':split,
            'prot_id':str(r.prot_id),
            'n_residues':len(r.sequence),
            'n_candidate_patches':len(pf),
            'n_memberships':len(er),
            'covered_residue_fraction':
                covered/max(len(r.sequence),1),
        })

        if n%250==0:
            print(split,n,'/',len(df))

patch_audit=pd.DataFrame(PATCH_AUDIT)
patch_audit.to_csv(RUN_ROOT/'patch_audit.csv',index=False)

display(
    patch_audit.groupby('dataset').agg(
        proteins=('prot_id','nunique'),
        mean_patches=('n_candidate_patches','mean'),
        median_patches=('n_candidate_patches','median'),
        mean_memberships=('n_memberships','mean'),
        mean_covered_fraction=('covered_residue_fraction','mean'),
    ).reset_index()
)

print('Patch hypergraph cache: PASS')


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300


,dataset,proteins,mean_patches,median_patches,mean_memberships,mean_covered_fraction
0,DevSet1010,1010,52.591089,48.0,714.855446,0.921091
1,TestSet300,300,65.996667,60.5,954.406667,0.949175
2,TestSetNew46,46,41.543478,41.0,543.717391,0.917373


Patch hypergraph cache: PASS


In [ ]:
# ============================================================
# 15. PATCH-HYPERGRAPH DATASET / MODEL
# ============================================================
class PatchProteinDataset(Dataset):
    def __init__(self,split,df):
        self.split=split
        self.df=df.reset_index(drop=True)
        self.store=ctx.stores[STORE_KEY[split]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        r=self.df.iloc[i]
        pid=str(r.prot_id)

        mods={
            m:concat_blocks(
                self.store,
                pid,
                blocks
            ).astype(np.float32,copy=False)
            for m,blocks in MODALITY_SPEC.items()
        }

        er,ep,pf=PATCH_RAM[(self.split,pid)]

        return {
            'pid':pid,
            'mods':mods,
            'y':np.asarray(r.nuclear_binding,dtype=np.float32),
            'edge_res':er,
            'edge_patch':ep,
            'patch_feat':pf,
        }

def collate_patch(batch):
    keys=sorted(batch[0]['mods'])

    mods={
        k:torch.from_numpy(
            np.concatenate([b['mods'][k] for b in batch],axis=0)
        ).float()
        for k in keys
    }

    y=torch.from_numpy(
        np.concatenate([b['y'] for b in batch])
    ).float()

    patch_feat=[]
    edge_res=[]
    edge_patch=[]

    roff=0
    poff=0

    for b in batch:
        L=len(b['y'])
        P=len(b['patch_feat'])

        if P:
            patch_feat.append(
                torch.from_numpy(b['patch_feat']).float()
            )

        if len(b['edge_res']):
            edge_res.append(
                torch.from_numpy(b['edge_res']).long()+roff
            )
            edge_patch.append(
                torch.from_numpy(b['edge_patch']).long()+poff
            )

        roff+=L
        poff+=P

    patch_feat=(
        torch.cat(patch_feat,dim=0)
        if patch_feat
        else torch.zeros((0,18),dtype=torch.float32)
    )

    edge_res=(
        torch.cat(edge_res,dim=0)
        if edge_res
        else torch.zeros((0,),dtype=torch.long)
    )

    edge_patch=(
        torch.cat(edge_patch,dim=0)
        if edge_patch
        else torch.zeros((0,),dtype=torch.long)
    )

    return {
        'mods':mods,
        'y':y,
        'edge_res':edge_res,
        'edge_patch':edge_patch,
        'patch_feat':patch_feat,
    }

class PatchResidueEncoder(nn.Module):
    def __init__(self,dims,mode='gated',latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.mode=mode

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        if mode=='gated':
            self.gate=nn.ModuleDict({
                k:nn.Linear(latent,1)
                for k in dims
            })
        elif mode=='attention':
            self.attn=nn.MultiheadAttention(
                latent,
                6,
                dropout=drop,
                batch_first=True
            )
        else:
            raise ValueError(mode)

    def forward(self,x):
        Z=torch.stack(
            [self.enc[k](x[k]) for k in self.keys],
            dim=1
        )

        if self.mode=='gated':
            G=torch.cat(
                [
                    self.gate[k](Z[:,j,:])
                    for j,k in enumerate(self.keys)
                ],
                dim=1
            )
            G=torch.softmax(G,dim=1).unsqueeze(-1)
            return (Z*G).sum(dim=1)

        A,_=self.attn(Z,Z,Z,need_weights=False)
        return (Z+A).mean(dim=1)

class NuclearPatchHypergraph(nn.Module):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        moe=False,
        experts=4,
        drop=0.2,
    ):
        super().__init__()

        self.encoder=PatchResidueEncoder(
            dims,
            mode=mode,
            latent=latent,
            drop=drop
        )

        self.patch_static=nn.Sequential(
            nn.LayerNorm(18),
            nn.Linear(18,64),
            nn.GELU()
        )

        self.patch_mlp=nn.Sequential(
            nn.Linear(latent*2+64,latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
            nn.GELU(),
        )

        self.message_mlp=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent)
        )

        self.alpha_logit=nn.Parameter(
            torch.tensor(-2.944,dtype=torch.float32)
        )

        self.moe=moe

        if not moe:
            self.head=nn.Sequential(
                nn.LayerNorm(latent),
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
        else:
            self.experts=nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(latent),
                    nn.Linear(latent,128),
                    nn.GELU(),
                    nn.Dropout(drop),
                    nn.Linear(128,1)
                )
                for _ in range(experts)
            ])
            self.expert_gate=nn.Linear(latent,experts)

    def index_mean(self,x,index,n):
        out=x.new_zeros((n,x.shape[-1]))
        cnt=x.new_zeros((n,1))

        if len(index):
            out.index_add_(0,index,x)
            cnt.index_add_(
                0,index,
                x.new_ones((len(index),1))
            )

        return out/cnt.clamp_min(1.0)

    def index_max(self, x, index, n):
        if n == 0:
            return x.new_zeros(
                (0, x.shape[-1])
            )

        base = x.new_full(
            (n, x.shape[-1]),
            -1e9
        )

        if len(index):
            idx = index[:, None].expand(
                -1,
                x.shape[-1]
            )

            # IMPORTANT:
            # out-of-place scatter_reduce for autograd safety
            out = base.scatter_reduce(
                0,
                idx,
                x,
                reduce='amax',
                include_self=True
            )

        else:
            out = base

        # IMPORTANT:
        # no inplace modification after scatter_reduce
        out = torch.where(
            out < -1e8,
            torch.zeros_like(out),
            out
        )

        return out
    def forward(self,mods,edge_res,edge_patch,patch_feat):
        h=self.encoder(mods)

        nres=h.shape[0]
        npatch=patch_feat.shape[0]

        alpha=torch.sigmoid(self.alpha_logit)

        if npatch and edge_res.numel():
            mean=self.index_mean(
                h[edge_res],
                edge_patch,
                npatch
            )

            mx=self.index_max(
                h[edge_res],
                edge_patch,
                npatch
            )

            ps=self.patch_static(patch_feat)

            hp=self.patch_mlp(
                torch.cat([mean,mx,ps],dim=-1)
            )

            msg=self.index_mean(
                hp[edge_patch],
                edge_res,
                nres
            )

            incident=h.new_zeros((nres,1))
            incident.index_add_(
                0,
                edge_res,
                h.new_ones((len(edge_res),1))
            )

            delta=self.message_mlp(msg)
            delta=delta*(incident>0).float()

            h=h+alpha*delta

        if not self.moe:
            return self.head(h).squeeze(-1)

        logits=torch.cat(
            [e(h) for e in self.experts],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (logits*w).sum(dim=1)

PATCH_FACTORY={
    'NA_PatchHG_Gated':lambda dims:NuclearPatchHypergraph(
        dims,mode='gated',latent=PATCH_LATENT,moe=False
    ),
    'NA_PatchHG_Attention':lambda dims:NuclearPatchHypergraph(
        dims,mode='attention',latent=PATCH_LATENT,moe=False
    ),
    'NA_PatchHG_GatedMoE':lambda dims:NuclearPatchHypergraph(
        dims,mode='gated',latent=PATCH_LATENT,moe=True,experts=4
    ),
}

print('Patch-hypergraph architectures:',list(PATCH_FACTORY))


Patch-hypergraph architectures: ['NA_PatchHG_Gated', 'NA_PatchHG_Attention', 'NA_PatchHG_GatedMoE']


In [ ]:
# ============================================================
# 16. PATCH-HYPERGRAPH CV TRAINER — RESUME SAFE
# ============================================================
PATCH_STATE_ROOT=RUN_ROOT/'patch_states'
PATCH_STATE_ROOT.mkdir(parents=True,exist_ok=True)

def move_patch_batch(b):
    return {
        'mods':{
            k:v.to(DEVICE,non_blocking=True)
            for k,v in b['mods'].items()
        },
        'y':b['y'].to(DEVICE,non_blocking=True),
        'edge_res':b['edge_res'].to(DEVICE,non_blocking=True),
        'edge_patch':b['edge_patch'].to(DEVICE,non_blocking=True),
        'patch_feat':b['patch_feat'].to(DEVICE,non_blocking=True),
    }

def predict_patch(model,dataset):
    loader=DataLoader(
        dataset,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch
    )

    pp=[]
    yy=[]

    model.eval()

    with torch.no_grad():
        for b in loader:
            b=move_patch_batch(b)

            p=torch.sigmoid(
                model(
                    b['mods'],
                    b['edge_res'],
                    b['edge_patch'],
                    b['patch_feat']
                )
            )

            pp.append(p.detach().cpu().numpy())
            yy.append(b['y'].detach().cpu().numpy())

    return np.concatenate(pp),np.concatenate(yy)

def train_patch_fold(arch,tr_ds,va_ds,fold):
    torch.manual_seed(SEED+100+fold)
    np.random.seed(SEED+100+fold)

    model=PATCH_FACTORY[arch](DIMS).to(DEVICE)

    ytr=np.concatenate(
        tr_ds.df.nuclear_binding.apply(
            lambda x:np.asarray(x,dtype=np.float32)
        ).tolist()
    )

    npos=max(float(ytr.sum()),1.0)
    nneg=max(float(len(ytr)-ytr.sum()),1.0)

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=PATCH_LR,
        weight_decay=PATCH_WEIGHT_DECAY
    )

    loader=DataLoader(
        tr_ds,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(1,PATCH_EPOCHS+1):
        model.train()
        losses=[]

        for b in loader:
            b=move_patch_batch(b)

            opt.zero_grad(set_to_none=True)

            logits=model(
                b['mods'],
                b['edge_res'],
                b['edge_patch'],
                b['patch_feat']
            )

            loss=criterion(logits,b['y'])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()

            losses.append(float(loss.detach().cpu()))

        vp,vy=predict_patch(model,va_ds)
        val_ap=ap(vy,vp)

        if val_ap>best_ap+1e-5:
            best_ap=val_ap
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad=0
        else:
            bad+=1

        if epoch==1 or epoch%5==0:
            print(
                arch,'fold',fold,'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5),
                'alpha',round(
                    float(torch.sigmoid(model.alpha_logit).detach().cpu()),
                    4
                )
            )

        if bad>=PATCH_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_state

def run_patch_cv(arch):
    offsets={}
    cursor=0

    for r in train_df.itertuples():
        L=len(r.sequence)
        offsets[str(r.prot_id)]=(cursor,cursor+L)
        cursor+=L

    oof=np.full(cursor,np.nan,dtype=np.float32)

    rows=[]
    states={}

    adir=PATCH_STATE_ROOT/arch
    adir.mkdir(parents=True,exist_ok=True)

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(val_ids)

        tr_df=train_df[
            ~train_df.prot_id.isin(val_set)
        ].copy()

        va_df=train_df[
            train_df.prot_id.isin(val_set)
        ].copy()

        tr_ds=PatchProteinDataset('DevSet1010',tr_df)
        va_ds=PatchProteinDataset('DevSet1010',va_df)

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        expected=int(va_df.sequence.str.len().sum())

        reuse=False

        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta=json.loads(meta_fp.read_text())
                pp=np.load(pred_fp)

                reuse=(
                    meta.get('representation')==BEST_REP_NAME
                    and meta.get('n_val')==expected
                    and pp.shape==(expected,)
                    and np.all(np.isfinite(pp))
                )
            except Exception:
                reuse=False

        if reuse:
            print(arch,'fold',fold,'REUSE')
            state=torch.load(state_fp,map_location='cpu')
            pred=np.load(pred_fp).astype(np.float32)
        else:
            print(arch,'fold',fold,'TRAIN')
            model,state=train_patch_fold(
                arch,tr_ds,va_ds,fold
            )
            pred,_=predict_patch(model,va_ds)

            torch.save(state,state_fp)
            np.save(pred_fp,pred)

            meta_fp.write_text(json.dumps({
                'representation':BEST_REP_NAME,
                'blocks':BEST_REP_BLOCKS,
                'n_val':expected,
            },indent=2))

            del model
            torch.cuda.empty_cache()

        states[fold]=state

        pos=0

        for r in va_df.itertuples():
            L=len(r.sequence)
            a,b=offsets[str(r.prot_id)]
            oof[a:b]=pred[pos:pos+L]
            pos+=L

        yv=np.concatenate(
            va_df.nuclear_binding.apply(
                lambda x:np.asarray(x,dtype=np.float32)
            ).tolist()
        )

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(yv,pred),
            'n_val':len(pred),
            'n_positive':int(yv.sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch}: incomplete patch OOF')

    f=pd.DataFrame(rows)
    yall=np.concatenate(
        train_df.nuclear_binding.apply(
            lambda x:np.asarray(x,dtype=np.float32)
        ).tolist()
    )

    return (
        {
            'architecture':arch,
            'CV_AUPRC_mean':float(f.AUPRC.mean()),
            'CV_AUPRC_std':float(f.AUPRC.std(ddof=1)),
            'OOF_AUPRC':ap(yall,oof),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 17. RUN PATCH-HYPERGRAPH SCREEN — DEV ONLY
# ============================================================
PATCH_RESULTS={}
PATCH_FOLDS=[]
PATCH_OOF={}
PATCH_STATES={}

if RUN_PATCH_HYPERGRAPH:
    for arch in PATCH_FACTORY:
        print('\n'+'='*80)
        print('PATCH HYPERGRAPH:',arch)
        print('='*80)

        s,f,o,st=run_patch_cv(arch)

        PATCH_RESULTS[arch]=s
        PATCH_FOLDS.append(f)
        PATCH_OOF[arch]=o
        PATCH_STATES[arch]=st

patch_summary=pd.DataFrame(
    PATCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(drop=True)

patch_folds=pd.concat(
    PATCH_FOLDS,
    ignore_index=True
)

patch_summary.to_csv(RUN_ROOT/'patch_hypergraph_cv.csv',index=False)
patch_folds.to_csv(RUN_ROOT/'patch_hypergraph_folds.csv',index=False)

display(patch_summary)

BEST_PATCH=str(patch_summary.iloc[0].architecture)
BEST_PATCH_SCORE=float(patch_summary.iloc[0].CV_AUPRC_mean)

alphas=[]
for fold,state in PATCH_STATES[BEST_PATCH].items():
    if 'alpha_logit' in state:
        alphas.append(
            float(torch.sigmoid(state['alpha_logit']).item())
        )

print('BEST PATCH MODEL:',BEST_PATCH)
print('Dev CV AUPRC:',BEST_PATCH_SCORE)
print('Residual alpha:',alphas)
print('Mean alpha:',float(np.mean(alphas)) if alphas else None)

(RUN_ROOT/'locked_patch_model.json').write_text(
    json.dumps({
        'architecture':BEST_PATCH,
        'dev_cv_auprc':BEST_PATCH_SCORE,
        'alpha_by_fold':alphas,
        'alpha_mean':float(np.mean(alphas)) if alphas else None,
    },indent=2)
)



PATCH HYPERGRAPH: NA_PatchHG_Gated
NA_PatchHG_Gated fold 1 TRAIN
NA_PatchHG_Gated fold 1 epoch 1 loss 0.2682 val_AP 0.56766 best 0.56766 alpha 0.05
NA_PatchHG_Gated fold 1 epoch 5 loss 0.0747 val_AP 0.54811 best 0.58113 alpha 0.0501
NA_PatchHG_Gated fold 1 epoch 10 loss 0.0244 val_AP 0.57857 best 0.59117 alpha 0.0499
NA_PatchHG_Gated fold 2 TRAIN
NA_PatchHG_Gated fold 2 epoch 1 loss 0.2835 val_AP 0.53159 best 0.53159 alpha 0.0499
NA_PatchHG_Gated fold 2 epoch 5 loss 0.0756 val_AP 0.54316 best 0.54872 alpha 0.05
NA_PatchHG_Gated fold 2 epoch 10 loss 0.0261 val_AP 0.52924 best 0.56808 alpha 0.0499
NA_PatchHG_Gated fold 3 TRAIN
NA_PatchHG_Gated fold 3 epoch 1 loss 0.2655 val_AP 0.3829 best 0.3829 alpha 0.05
NA_PatchHG_Gated fold 3 epoch 5 loss 0.0743 val_AP 0.38361 best 0.38864 alpha 0.0499
NA_PatchHG_Gated fold 4 TRAIN
NA_PatchHG_Gated fold 4 epoch 1 loss 0.2693 val_AP 0.36792 best 0.36792 alpha 0.0502
NA_PatchHG_Gated fold 4 epoch 5 loss 0.07 val_AP 0.4873 best 0.50715 alpha 0.0504
NA_

,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,NA_PatchHG_Attention,0.501934,0.090728,0.473134
1,NA_PatchHG_GatedMoE,0.498398,0.100202,0.487782
2,NA_PatchHG_Gated,0.492446,0.091827,0.478074


BEST PATCH MODEL: NA_PatchHG_Attention
Dev CV AUPRC: 0.5019338304768628
Residual alpha: [0.05013297498226166, 0.05012139678001404, 0.050040267407894135, 0.05039290338754654, 0.050106484442949295]
Mean alpha: 0.050158805400133136


272

# Part E — Global Dev-only lock

The final headline family is chosen **before** external evaluation.

A more complex patch-hypergraph model replaces the best local multimodal model only if it improves mean Dev CV AUPRC by at least the configured complexity margin.


In [ ]:
# ============================================================
# 18. GLOBAL DEV-ONLY LOCK
# ============================================================
candidates=pd.DataFrame([
    {
        'family':'LOCAL_MULTIMODAL',
        'model':LOCKED_ARCH,
        'Dev_CV_AUPRC':LOCKED_ARCH_SCORE,
        'complexity_rank':1,
    },
    {
        'family':'PATCH_HYPERGRAPH',
        'model':BEST_PATCH,
        'Dev_CV_AUPRC':BEST_PATCH_SCORE,
        'complexity_rank':2,
    },
])

winner=candidates.sort_values('complexity_rank').iloc[0].to_dict()

for _,r in candidates.sort_values('complexity_rank').iloc[1:].iterrows():
    if (
        float(r.Dev_CV_AUPRC)
        - float(winner['Dev_CV_AUPRC'])
        >= GLOBAL_COMPLEXITY_MARGIN
    ):
        winner=r.to_dict()

GLOBAL_FAMILY=str(winner['family'])
GLOBAL_MODEL=str(winner['model'])
GLOBAL_SCORE=float(winner['Dev_CV_AUPRC'])

display(candidates)

print('GLOBAL LOCK:',GLOBAL_FAMILY,GLOBAL_MODEL,GLOBAL_SCORE)

(RUN_ROOT/'GLOBAL_LOCK.json').write_text(
    json.dumps({
        'family':GLOBAL_FAMILY,
        'model':GLOBAL_MODEL,
        'dev_cv_auprc':GLOBAL_SCORE,
        'complexity_margin':GLOBAL_COMPLEXITY_MARGIN,
    },indent=2)
)


,family,model,Dev_CV_AUPRC,complexity_rank
0,LOCAL_MULTIMODAL,ConcatMLP,0.465391,1
1,PATCH_HYPERGRAPH,NA_PatchHG_Attention,0.501934,2


GLOBAL LOCK: PATCH_HYPERGRAPH NA_PatchHG_Attention 0.5019338304768628


137

In [ ]:
# ============================================================
# 19. METRIC HELPERS + DEV OOF THRESHOLDS
# ============================================================
def binary_metrics(y,p,threshold):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    pred=(p>=threshold).astype(int)

    return {
        'AUPRC':float(average_precision_score(y,p)),
        'AUROC':float(roc_auc_score(y,p)),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'F1':float(f1_score(y,pred,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pred)),
    }

def best_mcc_threshold(y,p):
    best=(-999,None)

    for t in np.linspace(0.01,0.99,199):
        m=matthews_corrcoef(
            y,
            (p>=t).astype(int)
        )

        if m>best[0]:
            best=(m,float(t))

    return best[1],best[0]

dev_y=PACKS['DevSet1010']['y'].astype(int)

LOCAL_OOF=ARCH_OOF[LOCKED_ARCH]
LOCAL_THR,LOCAL_OOF_MCC=best_mcc_threshold(dev_y,LOCAL_OOF)

PATCH_OOF_LOCK=PATCH_OOF[BEST_PATCH]
PATCH_THR,PATCH_OOF_MCC=best_mcc_threshold(dev_y,PATCH_OOF_LOCK)

print('Local threshold:',LOCAL_THR,'OOF MCC:',LOCAL_OOF_MCC)
print('Patch threshold:',PATCH_THR,'OOF MCC:',PATCH_OOF_MCC)


Local threshold: 0.702929292929293 OOF MCC: 0.4863944921124474
Patch threshold: 0.7672727272727272 OOF MCC: 0.500818929385667


In [ ]:
# ============================================================
# 20. FINAL EXTERNAL EVALUATION — LOCKED FAMILY WINNERS
# ============================================================
def predict_local_external(split):
    pack=PACKS[split]
    ds=MultiArrayDataset(pack['mods'],pack['y'])
    preds=[]

    for fold,state in ARCH_STATES[LOCKED_ARCH].items():
        model=ARCH_FACTORY[LOCKED_ARCH](DIMS).to(DEVICE)
        model.load_state_dict(state)
        preds.append(predict_flat(model,ds))
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(preds),axis=0)

def predict_patch_external(split):
    ds=PatchProteinDataset(split,SPLIT_DF[split])
    preds=[]

    for fold,state in PATCH_STATES[BEST_PATCH].items():
        model=PATCH_FACTORY[BEST_PATCH](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p,_=predict_patch(model,ds)
        preds.append(p)
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(preds),axis=0)

rows=[]

if RUN_FINAL_EXTERNAL:
    for split in ['TestSet300','TestSetNew46']:
        y=PACKS[split]['y'].astype(int)

        lp=predict_local_external(split)
        pp=predict_patch_external(split)

        for family,model,p,thr,devscore in [
            (
                'LOCAL_MULTIMODAL',
                LOCKED_ARCH,
                lp,
                LOCAL_THR,
                LOCKED_ARCH_SCORE
            ),
            (
                'PATCH_HYPERGRAPH',
                BEST_PATCH,
                pp,
                PATCH_THR,
                BEST_PATCH_SCORE
            ),
        ]:
            m05=binary_metrics(y,p,0.5)
            moo=binary_metrics(y,p,thr)

            row={
                'family':family,
                'model':model,
                'dataset':split,
                'Dev_CV_AUPRC':devscore,
                'Dev_OOF_MCC_threshold':thr,
            }

            for k,v in m05.items():
                row[f'{k}_fixed0.5']=v

            for k,v in moo.items():
                row[f'{k}_OOFthr']=v

            rows.append(row)

        pd.DataFrame({
            'prot_id':PACKS[split]['pids'],
            'residue_index':PACKS[split]['ridx'],
            'y_true':y,
            'local_probability':lp,
            'patch_probability':pp,
        }).to_csv(
            RUN_ROOT/f'predictions_{split}.csv.gz',
            index=False
        )

external_df=pd.DataFrame(rows)
external_df.to_csv(RUN_ROOT/'external_family_winners.csv',index=False)

display(external_df)

headline=external_df[
    external_df.family.eq(GLOBAL_FAMILY)
    & external_df.model.eq(GLOBAL_MODEL)
].copy()

headline.to_csv(RUN_ROOT/'FINAL_HEADLINE_EXTERNAL.csv',index=False)

print('HEADLINE:')
display(headline)


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,LOCAL_MULTIMODAL,ConcatMLP,TestSet300,0.465391,0.702929,0.416222,0.939848,0.545345,0.306803,0.392686,0.398762,0.416222,0.939848,0.615385,0.163265,0.258065,0.309496
1,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSet300,0.501934,0.767273,0.432287,0.941411,0.548712,0.333333,0.414727,0.417265,0.432287,0.941411,0.635659,0.167347,0.264943,0.318856
2,LOCAL_MULTIMODAL,ConcatMLP,TestSetNew46,0.465391,0.702929,0.118199,0.939255,0.150000,0.506494,0.231454,0.259975,0.118199,0.939255,0.093960,0.181818,0.123894,0.115603
3,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSetNew46,0.501934,0.767273,0.122551,0.936822,0.125413,0.493506,0.200000,0.231350,0.122551,0.936822,0.082759,0.155844,0.108108,0.098353


HEADLINE:


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
1,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSet300,0.501934,0.767273,0.432287,0.941411,0.548712,0.333333,0.414727,0.417265,0.432287,0.941411,0.635659,0.167347,0.264943,0.318856
3,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSetNew46,0.501934,0.767273,0.122551,0.936822,0.125413,0.493506,0.200000,0.231350,0.122551,0.936822,0.082759,0.155844,0.108108,0.098353


# Part F — Read-only comparison with published-reproduction baselines and prior GEAR controls

No baseline inference is rerun.


In [ ]:
# ============================================================
# 21. BASELINE / HISTORICAL CONTROL TABLES
# ============================================================
baseline_frames=[]

for fp in [
    BASELINE_WORK/'results'/'FULL_DIRECT_BASELINE_METRICS_300_46.csv',
]:
    if fp.exists():
        z=pd.read_csv(fp)

        if 'label' in z.columns:
            z=z[z.label.eq('nuclear_binding')].copy()

        if 'operating_point' in z.columns:
            z=z[z.operating_point.eq('fixed_0.5')].copy()

        baseline_frames.append(z)

baselines=(
    pd.concat(baseline_frames,ignore_index=True,sort=False)
    if baseline_frames
    else pd.DataFrame()
)

if len(baselines):
    cols=[
        c for c in [
            'method','dataset','label',
            'AP_AUPRC','AUPRC','AUROC',
            'global_positive_F1','global_MCC',
            'protein_mean_F1','protein_mean_MCC',
        ]
        if c in baselines.columns
    ]

    print('LOCKED PUBLISHED-REPRODUCTION BASELINES')
    display(baselines[cols])

    baselines.to_csv(
        RUN_ROOT/'locked_nuclear_baselines.csv',
        index=False
    )

# Prior GEAR nuclear controls, read-only.
historic_runs=[
    'ultimate_other_generic_core_v1',
    'ultimate_other_standardized13_transfer_v1',
    'ultimate_other_metal_mechanistic_transfer_v1',
    'ultimate_other_evo_full_v1',
    'ultimate_other_generic_graph_v1',
    'ultimate_other_evo_graph_v1',
]

historic=[]

for run in historic_runs:
    root=BASE/'results'/run/'nuclear_binding'
    fp=root/'cv_summary.csv'

    if fp.exists():
        z=pd.read_csv(fp)
        z.insert(0,'run_version',run)
        historic.append(z)

historic_cv=(
    pd.concat(historic,ignore_index=True,sort=False)
    if historic
    else pd.DataFrame()
)

if len(historic_cv):
    print('PRIOR NUCLEAR DEV CONTROLS')
    display(
        historic_cv.sort_values(
            'CV_AUPRC',
            ascending=False
        ).head(30)
    )

    historic_cv.to_csv(
        RUN_ROOT/'prior_nuclear_cv_controls.csv',
        index=False
    )


LOCKED PUBLISHED-REPRODUCTION BASELINES


,method,dataset,label,AP_AUPRC,AUROC,global_positive_F1,global_MCC,protein_mean_F1,protein_mean_MCC
0,bindEmbed21DL,TestSet300,nuclear_binding,0.371532,0.921621,0.378126,0.379045,0.238822,0.224839
1,bindEmbed21DL,TestSetNew46,nuclear_binding,0.094095,0.927154,0.198582,0.239288,0.193721,0.192283
2,bindNode24,TestSet300,nuclear_binding,0.400676,0.943253,0.383162,0.386394,0.261493,0.248807
3,bindNode24,TestSetNew46,nuclear_binding,0.124657,0.948962,0.221198,0.272449,0.250441,0.254545


PRIOR NUCLEAR DEV CONTROLS


,run_version,model,representation,folds,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,CV_F1_at_0_5,CV_MCC_at_0_5,mean_best_epoch,n_params,mean_seconds,fold_AUPRC,OOF_threshold,mean_alpha
17,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Access,5.0,0.468560,0.088412,0.950651,0.378434,0.420452,3.2,357505.0,20.657980,NaN,NaN,NaN
18,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Elec3D,5.0,0.467955,0.089302,0.950274,0.374336,0.415830,3.4,356993.0,21.373925,NaN,NaN,NaN
30,ultimate_other_evo_full_v1,MLP,ESM+Structure+EVO,5.0,0.466832,0.085209,0.947550,0.389136,0.424311,4.0,387969.0,25.643859,NaN,NaN,NaN
19,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Env,5.0,0.464981,0.090594,0.947197,0.378886,0.417191,4.2,356993.0,22.723710,NaN,NaN,NaN
20,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+ALL9,5.0,0.464579,0.088768,0.949157,0.379556,0.419420,3.8,378497.0,22.948221,NaN,NaN,NaN
21,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+MECH1,5.0,0.464341,0.085201,0.949126,0.369691,0.413587,3.4,363649.0,21.909971,NaN,NaN,NaN
4,ultimate_other_standardized13_transfer_v1,MLP,ESM+Structure+Atomic,5.0,0.464094,0.091454,0.946928,0.384151,0.422047,3.2,379521.0,21.691321,NaN,NaN,NaN
5,ultimate_other_standardized13_transfer_v1,MLP,ESM+Structure+Chem,5.0,0.462884,0.088831,0.949563,0.363047,0.407773,2.8,381825.0,20.814985,NaN,NaN,NaN
0,ultimate_other_generic_core_v1,MLP,ESM+Structure,5.0,0.462125,0.086578,0.949836,0.383998,0.423414,3.4,375681.0,21.929307,NaN,NaN,NaN
31,ultimate_other_evo_full_v1,MLP,ESM+Structure,5.0,0.462125,0.086578,0.949836,0.383998,0.423414,3.4,375681.0,23.568707,NaN,NaN,NaN


# Part G — Automatic insight report

The final section explains whether the data support:

- electrostatic surface recognition;
- H-bond chemistry;
- patch organization;
- evolutionary context;
- aromatic control;
- multimodal fusion;
- full patch-hypergraph reasoning.

It also explicitly reports when extra complexity is not justified.


In [ ]:
# ============================================================
# 22. GAIN DECOMPOSITION / HYPOTHESIS VERDICTS
# ============================================================
def score_rep(df,name):
    x=df.loc[
        df.representation.eq(name),
        'CV_AUPRC'
    ]
    return float(x.iloc[0]) if len(x) else np.nan

mechanism_rows=[]

for name in MECHANISM_GRID:
    s=score_rep(combo_cv,name)

    mechanism_rows.append({
        'representation':name,
        'Dev_CV_AUPRC':s,
        'delta_vs_ESM':s-base_score,
    })

mechanism_gain=pd.DataFrame(mechanism_rows).sort_values(
    'Dev_CV_AUPRC',
    ascending=False
)

display(mechanism_gain)

mechanism_gain.to_csv(
    RUN_ROOT/'INSIGHT_mechanism_gain.csv',
    index=False
)

arch_gain=LOCKED_ARCH_SCORE-concat_score
patch_gain=BEST_PATCH_SCORE-LOCKED_ARCH_SCORE

def verdict(gain):
    if gain>=0.01:
        return 'STRONG'
    if gain>=0.005:
        return 'CLEAR'
    if gain>=MIN_MEAN_AUPRC_GAIN:
        return 'MODEST_BUT_REAL'
    if gain>0:
        return 'WEAK'
    return 'NO_GAIN'

summary={
    'ESM_baseline':base_score,

    'best_representation':
        BEST_REP_NAME,

    'best_representation_dev_auprc':
        BEST_REP_SCORE,

    'best_representation_gain_vs_ESM':
        BEST_REP_SCORE-base_score,

    'locked_multimodal_architecture':
        LOCKED_ARCH,

    'locked_multimodal_dev_auprc':
        LOCKED_ARCH_SCORE,

    'architecture_gain_vs_concat':
        arch_gain,

    'architecture_verdict':
        verdict(arch_gain),

    'best_patch_hypergraph':
        BEST_PATCH,

    'best_patch_dev_auprc':
        BEST_PATCH_SCORE,

    'patch_gain_vs_locked_local':
        patch_gain,

    'patch_verdict':
        verdict(patch_gain),

    'patch_alpha_mean':
        float(np.mean(alphas))
        if alphas else None,

    'global_family':
        GLOBAL_FAMILY,

    'global_model':
        GLOBAL_MODEL,

    'global_dev_auprc':
        GLOBAL_SCORE,

    'EVO_available':
        EVO_AVAILABLE,

    'ESMIF_available':
        ESMIF_AVAILABLE,
}

(RUN_ROOT/'INSIGHT_summary.json').write_text(
    json.dumps(summary,indent=2)
)

print(json.dumps(summary,indent=2))

print('\n'+'='*96)
print('NUCLEAR ULTIMATE V1 — INTERPRETATION')
print('='*96)

# Main mechanism comparisons.
for key in [
    'ESM+ACCESS+ELEC3D',
    'ESM+BACKBONE_ATTRACTION',
    'ESM+PATCH_ORGANIZATION',
    'ESM+ELECTRO_PATCH',
    'ESM+ELECTRO_PATCH_HBOND',
    'ESM+ELECTRO_PATCH_HBOND+AROMCTRL',
    'ESM+ELECTRO_PATCH_HBOND+EVO',
]:
    if key in set(combo_cv.representation):
        s=score_rep(combo_cv,key)
        print(
            f'{key:42s} '
            f'{s:.5f} '
            f'({s-base_score:+.5f} vs ESM)'
        )

print('\nArchitecture:')
print(
    LOCKED_ARCH,
    f'{LOCKED_ARCH_SCORE:.5f}',
    'gain vs concat =',
    f'{arch_gain:+.5f}',
    verdict(arch_gain)
)

print('\nPatch hypergraph:')
print(
    BEST_PATCH,
    f'{BEST_PATCH_SCORE:.5f}',
    'gain vs local =',
    f'{patch_gain:+.5f}',
    verdict(patch_gain)
)

if alphas:
    print(
        'Learned patch residual alpha:',
        [round(x,4) for x in alphas],
        'mean=',
        round(float(np.mean(alphas)),4)
    )

print('\nGLOBAL DEV-LOCKED WINNER:')
print(
    GLOBAL_FAMILY,
    '/',
    GLOBAL_MODEL,
    '/',
    f'{GLOBAL_SCORE:.5f}'
)

print('\nExternal headline:')
display(headline)

print('\nArtifacts:')
print(RUN_ROOT)
print('='*96)


,representation,Dev_CV_AUPRC,delta_vs_ESM
4,ESM+PATCH_ORGANIZATION,0.474114,0.021928
7,ESM+ELECTRO_PATCH_HBOND+AROMCTRL,0.472061,0.019874
2,ESM+BACKBONE_ATTRACTION,0.471199,0.019013
1,ESM+ACCESS+ELEC3D,0.468935,0.016749
6,ESM+ELECTRO_PATCH_HBOND,0.468750,0.016563
8,ESM+ELECTRO_PATCH_HBOND+EVO,0.466043,0.013857
5,ESM+ELECTRO_PATCH,0.465813,0.013626
3,ESM+HBOND_RECOGNITION,0.459334,0.007147
0,ESM,0.452187,0.000000


{
  "ESM_baseline": 0.4521865239018867,
  "best_representation": "ESM+PATCH_ORGANIZATION",
  "best_representation_dev_auprc": 0.47411433584661966,
  "best_representation_gain_vs_ESM": 0.021927811944732978,
  "locked_multimodal_architecture": "ConcatMLP",
  "locked_multimodal_dev_auprc": 0.46539084222578725,
  "architecture_gain_vs_concat": 0.0,
  "architecture_verdict": "NO_GAIN",
  "best_patch_hypergraph": "NA_PatchHG_Attention",
  "best_patch_dev_auprc": 0.5019338304768628,
  "patch_gain_vs_locked_local": 0.03654298825107555,
  "patch_verdict": "STRONG",
  "patch_alpha_mean": 0.050158805400133136,
  "global_family": "PATCH_HYPERGRAPH",
  "global_model": "NA_PatchHG_Attention",
  "global_dev_auprc": 0.5019338304768628,
  "EVO_available": true,
  "ESMIF_available": false
}

NUCLEAR ULTIMATE V1 — INTERPRETATION
ESM+ACCESS+ELEC3D                          0.46894 (+0.01675 vs ESM)
ESM+BACKBONE_ATTRACTION                    0.47120 (+0.01901 vs ESM)
ESM+PATCH_ORGANIZATION                  

,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
1,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSet300,0.501934,0.767273,0.432287,0.941411,0.548712,0.333333,0.414727,0.417265,0.432287,0.941411,0.635659,0.167347,0.264943,0.318856
3,PATCH_HYPERGRAPH,NA_PatchHG_Attention,TestSetNew46,0.501934,0.767273,0.122551,0.936822,0.125413,0.493506,0.200000,0.231350,0.122551,0.936822,0.082759,0.155844,0.108108,0.098353



Artifacts:
/content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH


In [ ]:
# ============================================================
# 23. FINAL MANIFEST
# ============================================================
manifest={
    'gear_version':GEAR_VERSION,
    'python':platform.python_version(),
    'torch':torch.__version__,
    'device':str(DEVICE),

    'task':'nuclear_binding',

    'development_set':
        'DevSet1010 official bindNode24 5-fold split',

    'external_sets':[
        'TestSet300',
        'TestSetNew46',
    ],

    'selection_metric':
        'residue-level Dev CV AUPRC',

    'audit_driven_design':{
        'core_existing':['ACCESS','ELEC3D'],
        'new_blocks':NEW_BLOCKS,
        'aromatic_role':'control, not core prior',
        'patch_reasoning':'soft residual due Test46 interface-size shift',
    },

    'best_representation':BEST_REP_NAME,
    'best_representation_blocks':BEST_REP_BLOCKS,
    'locked_architecture':LOCKED_ARCH,
    'best_patch_model':BEST_PATCH,

    'global_family':GLOBAL_FAMILY,
    'global_model':GLOBAL_MODEL,
    'global_dev_auprc':GLOBAL_SCORE,

    'evo_available':EVO_AVAILABLE,
    'esmif_available':ESMIF_AVAILABLE,

    'summary':summary,
}

(RUN_ROOT/'FINAL_MANIFEST.json').write_text(
    json.dumps(manifest,indent=2)
)

print(json.dumps(manifest,indent=2))

print('\n'+'='*96)
print('NUCLEAR ULTIMATE V1 COMPLETE')
print('='*96)
print('Headline:',GLOBAL_FAMILY,'/',GLOBAL_MODEL)
print('Dev CV AUPRC:',GLOBAL_SCORE)
print('Saved to:',RUN_ROOT)
print('='*96)


{
  "gear_version": "2026.08-other-bindings-v1",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "device": "cuda",
  "task": "nuclear_binding",
  "development_set": "DevSet1010 official bindNode24 5-fold split",
  "external_sets": [
    "TestSet300",
    "TestSetNew46"
  ],
  "selection_metric": "residue-level Dev CV AUPRC",
  "audit_driven_design": {
    "core_existing": [
      "ACCESS",
      "ELEC3D"
    ],
    "new_blocks": {
      "NA_SEQELECTRO32": 32,
      "NA_PATCH3D40": 40,
      "NA_HBOND16": 16,
      "NA_TOPO12": 12,
      "NA_PROTCTX8": 8,
      "NA_AROMCTRL8": 8
    },
    "aromatic_role": "control, not core prior",
    "patch_reasoning": "soft residual due Test46 interface-size shift"
  },
  "best_representation": "ESM+PATCH_ORGANIZATION",
  "best_representation_blocks": [
    "ESM",
    "ACCESS",
    "NA_PATCH3D40",
    "NA_TOPO12"
  ],
  "locked_architecture": "ConcatMLP",
  "best_patch_model": "NA_PatchHG_Attention",
  "global_family": "PATCH_HYPERGRAPH",
  "glo

In [ ]:
# ============================================================
# 24. FINAL CONSOLIDATED PERFORMANCE COMPARISON
#     Read-only: NO training / NO model selection
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 110)
print("NUCLEAR ULTIMATE V1 — FINAL CONSOLIDATED COMPARISON")
print("=" * 110)


# ============================================================
# A. DEV REPRESENTATION LADDER
#    IMPORTANT:
#    These rows use the SAME GEAR MLP framework and are directly
#    appropriate for feature/representation ablation.
# ============================================================

dev_rep_rows = []

# ESM baseline
dev_rep_rows.append({
    "stage": "Representation",
    "method": "ESM baseline",
    "framework": "GEAR MLP",
    "Dev_CV_AUPRC": float(base_score),
    "delta_vs_ESM": 0.0,
})

# Existing audit-supported compact representation
for rep_name in [
    "ESM+ACCESS+ELEC3D",
    "ESM+BACKBONE_ATTRACTION",
    "ESM+PATCH_ORGANIZATION",
    "ESM+ELECTRO_PATCH",
    "ESM+ELECTRO_PATCH_HBOND",
    "ESM+ELECTRO_PATCH_HBOND+AROMCTRL",
    "ESM+ELECTRO_PATCH_HBOND+EVO",
]:
    if rep_name in set(combo_cv["representation"]):
        score = float(
            combo_cv.loc[
                combo_cv["representation"].eq(rep_name),
                "CV_AUPRC"
            ].iloc[0]
        )

        dev_rep_rows.append({
            "stage": "Representation",
            "method": rep_name,
            "framework": "GEAR MLP",
            "Dev_CV_AUPRC": score,
            "delta_vs_ESM": score - float(base_score),
        })

dev_rep = (
    pd.DataFrame(dev_rep_rows)
    .sort_values("Dev_CV_AUPRC", ascending=False)
    .reset_index(drop=True)
)

print("\n[A] DEV REPRESENTATION / FEATURE ABLATION")
display(dev_rep)

dev_rep.to_csv(
    RUN_ROOT / "FINAL_COMPARISON_DEV_REPRESENTATION.csv",
    index=False
)


# ============================================================
# B. DEV ARCHITECTURE COMPARISON
#
# NOTE:
# Custom fusion / hypergraph models use the custom trainer.
# Therefore:
#
#   ConcatMLP  vs ProjectedConcat/Gated/etc. = matched comparison
#   ConcatMLP  vs Patch Hypergraph           = matched-enough custom family
#
# Do NOT interpret:
#   GEAR MLP best representation 0.xxx
#   vs custom ConcatMLP 0.xxx
# as pure architecture gain, because trainers/objectives differ.
# ============================================================

dev_arch_rows = []

for _, r in arch_summary.iterrows():
    dev_arch_rows.append({
        "family": "LOCAL_MULTIMODAL",
        "method": str(r["architecture"]),
        "Dev_CV_AUPRC_mean": float(r["CV_AUPRC_mean"]),
        "Dev_CV_AUPRC_SD": float(r["CV_AUPRC_std"]),
        "OOF_AUPRC": float(r["OOF_AUPRC"]),
        "delta_vs_ConcatMLP":
            float(r["CV_AUPRC_mean"]) - float(concat_score),
    })

for _, r in patch_summary.iterrows():
    dev_arch_rows.append({
        "family": "PATCH_HYPERGRAPH",
        "method": str(r["architecture"]),
        "Dev_CV_AUPRC_mean": float(r["CV_AUPRC_mean"]),
        "Dev_CV_AUPRC_SD": float(r["CV_AUPRC_std"]),
        "OOF_AUPRC": float(r["OOF_AUPRC"]),
        "delta_vs_ConcatMLP":
            float(r["CV_AUPRC_mean"]) - float(concat_score),
    })

dev_arch = (
    pd.DataFrame(dev_arch_rows)
    .sort_values("Dev_CV_AUPRC_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n[B] DEV MATCHED ARCHITECTURE COMPARISON")
display(dev_arch)

dev_arch.to_csv(
    RUN_ROOT / "FINAL_COMPARISON_DEV_ARCHITECTURE.csv",
    index=False
)


# ============================================================
# C. LOAD LOCKED PUBLISHED-REPRODUCTION BASELINES
# ============================================================

baseline_fp = (
    BASELINE_WORK
    / "results"
    / "FULL_DIRECT_BASELINE_METRICS_300_46.csv"
)

if not baseline_fp.exists():
    raise FileNotFoundError(
        f"Baseline file not found: {baseline_fp}"
    )

b = pd.read_csv(baseline_fp)

if "label" in b.columns:
    b = b[
        b["label"].eq("nuclear_binding")
    ].copy()

if "operating_point" in b.columns:
    b = b[
        b["operating_point"].eq("fixed_0.5")
    ].copy()

# Normalize dataset names.
dataset_map = {
    "TestSet300": "TestSet300",
    "TestSetNew46": "TestSetNew46",
    "Test300": "TestSet300",
    "Test46": "TestSetNew46",
}

if "dataset" in b.columns:
    b["dataset"] = b["dataset"].replace(dataset_map)


# ============================================================
# D. BUILD EXTERNAL BENCHMARK TABLE
#    Same test sets, nuclear_binding only.
# ============================================================

benchmark_rows = []

# ---------- OUR LOCKED MODEL ----------
for _, r in headline.iterrows():

    benchmark_rows.append({
        "method": f"Our {GLOBAL_MODEL}",
        "family": GLOBAL_FAMILY,
        "dataset": str(r["dataset"]),

        "AUPRC": float(r["AUPRC_fixed0.5"]),
        "AUROC": float(r["AUROC_fixed0.5"]),
        "F1": float(r["F1_fixed0.5"]),
        "MCC": float(r["MCC_fixed0.5"]),

        "precision": float(r["precision_fixed0.5"]),
        "recall": float(r["recall_fixed0.5"]),

        "threshold": 0.5,
        "source": "Current nuclear V1",
    })


# ---------- DIRECT REPRODUCED BASELINES ----------
for _, r in b.iterrows():

    method = str(r["method"])

    if method not in {
        "bindEmbed21DL",
        "bindNode24",
    }:
        continue

    auprc_col = (
        "AP_AUPRC"
        if "AP_AUPRC" in r.index
        else "AUPRC"
    )

    benchmark_rows.append({
        "method": method,
        "family": "PUBLISHED_BASELINE",
        "dataset": str(r["dataset"]),

        "AUPRC": float(r[auprc_col]),
        "AUROC": float(r["AUROC"]),
        "F1": float(r["global_positive_F1"]),
        "MCC": float(r["global_MCC"]),

        "precision": np.nan,
        "recall": np.nan,

        "threshold": 0.5,
        "source": "Locked direct reproduction",
    })


benchmark = pd.DataFrame(benchmark_rows)

dataset_order = {
    "TestSet300": 0,
    "TestSetNew46": 1,
}

method_order = {
    f"Our {GLOBAL_MODEL}": 0,
    "bindNode24": 1,
    "bindEmbed21DL": 2,
}

benchmark["_ds_order"] = benchmark["dataset"].map(
    dataset_order
).fillna(99)

benchmark["_method_order"] = benchmark["method"].map(
    method_order
).fillna(99)

benchmark = (
    benchmark
    .sort_values(
        ["_ds_order", "_method_order"]
    )
    .drop(
        columns=[
            "_ds_order",
            "_method_order",
        ]
    )
    .reset_index(drop=True)
)

print("\n[C] EXTERNAL BENCHMARK — FIXED THRESHOLD 0.5")
display(
    benchmark[
        [
            "dataset",
            "method",
            "AUPRC",
            "AUROC",
            "F1",
            "MCC",
        ]
    ]
)

benchmark.to_csv(
    RUN_ROOT / "FINAL_BENCHMARK_COMPARISON.csv",
    index=False
)


# ============================================================
# E. DELTA OF OUR MODEL VS EACH BASELINE
# ============================================================

our_name = f"Our {GLOBAL_MODEL}"

delta_rows = []

for dataset in [
    "TestSet300",
    "TestSetNew46",
]:

    ours = benchmark[
        benchmark["dataset"].eq(dataset)
        & benchmark["method"].eq(our_name)
    ]

    if len(ours) != 1:
        continue

    ours = ours.iloc[0]

    for baseline_name in [
        "bindNode24",
        "bindEmbed21DL",
    ]:

        ref = benchmark[
            benchmark["dataset"].eq(dataset)
            & benchmark["method"].eq(baseline_name)
        ]

        if len(ref) != 1:
            continue

        ref = ref.iloc[0]

        delta_rows.append({
            "dataset": dataset,
            "comparison":
                f"{our_name} - {baseline_name}",

            "delta_AUPRC":
                float(ours["AUPRC"] - ref["AUPRC"]),

            "delta_AUROC":
                float(ours["AUROC"] - ref["AUROC"]),

            "delta_F1":
                float(ours["F1"] - ref["F1"]),

            "delta_MCC":
                float(ours["MCC"] - ref["MCC"]),
        })

delta_df = pd.DataFrame(delta_rows)

print("\n[D] DELTA OF OUR LOCKED MODEL VS BASELINES")
display(delta_df)

delta_df.to_csv(
    RUN_ROOT / "FINAL_DELTA_VS_BASELINES.csv",
    index=False
)


# ============================================================
# F. COMPACT SCIENTIFIC LADDER
# ============================================================

best_rep_row = dev_rep.iloc[0]

best_local_row = dev_arch[
    dev_arch["family"].eq(
        "LOCAL_MULTIMODAL"
    )
].sort_values(
    "Dev_CV_AUPRC_mean",
    ascending=False
).iloc[0]

best_patch_row = dev_arch[
    dev_arch["family"].eq(
        "PATCH_HYPERGRAPH"
    )
].sort_values(
    "Dev_CV_AUPRC_mean",
    ascending=False
).iloc[0]

ladder = pd.DataFrame([
    {
        "step": 1,
        "stage": "Sequence baseline",
        "model": "ESM",
        "Dev_AUPRC": float(base_score),
        "comparison_scope": "GEAR MLP representation",
    },

    {
        "step": 2,
        "stage": "Best explicit representation",
        "model": str(best_rep_row["method"]),
        "Dev_AUPRC":
            float(best_rep_row["Dev_CV_AUPRC"]),
        "comparison_scope": "GEAR MLP representation",
    },

    {
        "step": 3,
        "stage": "Matched local custom baseline",
        "model": "ConcatMLP",
        "Dev_AUPRC": float(concat_score),
        "comparison_scope": "Custom architecture",
    },

    {
        "step": 4,
        "stage": "Best local custom architecture",
        "model": str(best_local_row["method"]),
        "Dev_AUPRC":
            float(best_local_row["Dev_CV_AUPRC_mean"]),
        "comparison_scope": "Custom architecture",
    },

    {
        "step": 5,
        "stage": "Best patch hypergraph",
        "model": str(best_patch_row["method"]),
        "Dev_AUPRC":
            float(best_patch_row["Dev_CV_AUPRC_mean"]),
        "comparison_scope": "Custom architecture",
    },
])

print("\n[E] SCIENTIFIC PERFORMANCE LADDER")
display(ladder)

ladder.to_csv(
    RUN_ROOT / "FINAL_SCIENTIFIC_LADDER.csv",
    index=False
)


# ============================================================
# G. AUTOMATIC KEY NUMBERS / VERDICT
# ============================================================

rep_gain = (
    float(BEST_REP_SCORE)
    - float(base_score)
)

patch_gain_matched = (
    float(BEST_PATCH_SCORE)
    - float(concat_score)
)

test300_delta_bn24 = delta_df.loc[
    delta_df["dataset"].eq("TestSet300")
    & delta_df["comparison"].str.endswith(
        "bindNode24"
    ),
    "delta_AUPRC"
]

test46_delta_bn24 = delta_df.loc[
    delta_df["dataset"].eq("TestSetNew46")
    & delta_df["comparison"].str.endswith(
        "bindNode24"
    ),
    "delta_AUPRC"
]

test300_delta_bn24 = (
    float(test300_delta_bn24.iloc[0])
    if len(test300_delta_bn24)
    else np.nan
)

test46_delta_bn24 = (
    float(test46_delta_bn24.iloc[0])
    if len(test46_delta_bn24)
    else np.nan
)

comparison_summary = {
    "task": "nuclear_binding",

    "ESM_Dev_CV_AUPRC":
        float(base_score),

    "best_explicit_representation":
        BEST_REP_NAME,

    "best_explicit_representation_Dev_CV_AUPRC":
        float(BEST_REP_SCORE),

    "explicit_representation_gain_vs_ESM":
        rep_gain,

    "matched_custom_baseline":
        "ConcatMLP",

    "ConcatMLP_Dev_CV_AUPRC":
        float(concat_score),

    "best_patch_hypergraph":
        BEST_PATCH,

    "best_patch_hypergraph_Dev_CV_AUPRC":
        float(BEST_PATCH_SCORE),

    "patch_gain_vs_ConcatMLP":
        patch_gain_matched,

    "Test300_delta_AUPRC_vs_bindNode24":
        test300_delta_bn24,

    "Test46_delta_AUPRC_vs_bindNode24":
        test46_delta_bn24,

    "global_locked_family":
        GLOBAL_FAMILY,

    "global_locked_model":
        GLOBAL_MODEL,
}

(RUN_ROOT / "FINAL_COMPARISON_SUMMARY.json").write_text(
    json.dumps(
        comparison_summary,
        indent=2
    )
)

print("\n[F] KEY VERDICT")
print("-" * 110)

print(
    f"ESM -> best explicit representation: "
    f"{base_score:.5f} -> {BEST_REP_SCORE:.5f} "
    f"({rep_gain:+.5f})"
)

print(
    f"ConcatMLP -> best patch hypergraph: "
    f"{concat_score:.5f} -> {BEST_PATCH_SCORE:.5f} "
    f"({patch_gain_matched:+.5f})"
)

print(
    f"TestSet300 ΔAUPRC vs bindNode24: "
    f"{test300_delta_bn24:+.5f}"
)

print(
    f"TestSetNew46 ΔAUPRC vs bindNode24: "
    f"{test46_delta_bn24:+.5f}"
)

if (
    test300_delta_bn24 > 0
    and test46_delta_bn24 >= 0
):
    ext_verdict = (
        "OUTPERFORMS bindNode24 on both external sets."
    )

elif (
    test300_delta_bn24 > 0
    and test46_delta_bn24 > -0.005
):
    ext_verdict = (
        "CLEAR Test300 improvement; essentially competitive "
        "with bindNode24 on shifted Test46."
    )

elif test300_delta_bn24 > 0:
    ext_verdict = (
        "CLEAR Test300 improvement, but the gain does not fully "
        "transfer to shifted Test46."
    )

else:
    ext_verdict = (
        "No consistent external advantage over bindNode24."
    )

print(
    "External verdict:",
    ext_verdict
)

print("-" * 110)

print(
    "\nIMPORTANT INTERPRETATION NOTE:"
)

print(
    "The GEAR-MLP representation ladder and the custom architecture "
    "ladder use different trainers/objectives. "
    "Therefore, do NOT interpret BEST_REP_SCORE -> ConcatMLP "
    "as a pure architecture effect."
)

print(
    "The clean architecture comparison is ConcatMLP vs the other "
    "custom fusion / patch-hypergraph models."
)

print("\nSaved consolidated outputs to:")
print(RUN_ROOT)

print("=" * 110)

NUCLEAR ULTIMATE V1 — FINAL CONSOLIDATED COMPARISON

[A] DEV REPRESENTATION / FEATURE ABLATION


,stage,method,framework,Dev_CV_AUPRC,delta_vs_ESM
0,Representation,ESM+PATCH_ORGANIZATION,GEAR MLP,0.474114,0.021928
1,Representation,ESM+ELECTRO_PATCH_HBOND+AROMCTRL,GEAR MLP,0.472061,0.019874
2,Representation,ESM+BACKBONE_ATTRACTION,GEAR MLP,0.471199,0.019013
3,Representation,ESM+ACCESS+ELEC3D,GEAR MLP,0.468935,0.016749
4,Representation,ESM+ELECTRO_PATCH_HBOND,GEAR MLP,0.468750,0.016563
5,Representation,ESM+ELECTRO_PATCH_HBOND+EVO,GEAR MLP,0.466043,0.013857
6,Representation,ESM+ELECTRO_PATCH,GEAR MLP,0.465813,0.013626
7,Representation,ESM baseline,GEAR MLP,0.452187,0.000000



[B] DEV MATCHED ARCHITECTURE COMPARISON


,family,method,Dev_CV_AUPRC_mean,Dev_CV_AUPRC_SD,OOF_AUPRC,delta_vs_ConcatMLP
0,PATCH_HYPERGRAPH,NA_PatchHG_Attention,0.501934,0.090728,0.473134,0.036543
1,PATCH_HYPERGRAPH,NA_PatchHG_GatedMoE,0.498398,0.100202,0.487782,0.033007
2,PATCH_HYPERGRAPH,NA_PatchHG_Gated,0.492446,0.091827,0.478074,0.027055
3,LOCAL_MULTIMODAL,ProjectedConcat,0.467308,0.089055,0.474557,0.001918
4,LOCAL_MULTIMODAL,ConcatMLP,0.465391,0.082665,0.444775,0.000000
5,LOCAL_MULTIMODAL,ModalityAttention,0.463773,0.090579,0.453643,-0.001618
6,LOCAL_MULTIMODAL,GatedFusion,0.461928,0.093732,0.459380,-0.003463
7,LOCAL_MULTIMODAL,LatentNAExpertMoE,0.457301,0.093736,0.454949,-0.008090
8,LOCAL_MULTIMODAL,PatchResidualFusion,0.455773,0.092193,0.426168,-0.009618



[C] EXTERNAL BENCHMARK — FIXED THRESHOLD 0.5


...


[D] DELTA OF OUR LOCKED MODEL VS BASELINES


,dataset,comparison,delta_AUPRC,delta_AUROC,delta_F1,delta_MCC
0,TestSet300,Our NA_PatchHG_Attention - bindNode24,0.031610,-0.001842,0.031566,0.030871
1,TestSet300,Our NA_PatchHG_Attention - bindEmbed21DL,0.060755,0.019790,0.036601,0.038220
2,TestSetNew46,Our NA_PatchHG_Attention - bindNode24,-0.002106,-0.012140,-0.021198,-0.041099
3,TestSetNew46,Our NA_PatchHG_Attention - bindEmbed21DL,0.028456,0.009667,0.001418,-0.007939



[E] SCIENTIFIC PERFORMANCE LADDER


,step,stage,model,Dev_AUPRC,comparison_scope
0,1,Sequence baseline,ESM,0.452187,GEAR MLP representation
1,2,Best explicit representation,ESM+PATCH_ORGANIZATION,0.474114,GEAR MLP representation
2,3,Matched local custom baseline,ConcatMLP,0.465391,Custom architecture
3,4,Best local custom architecture,ProjectedConcat,0.467308,Custom architecture
4,5,Best patch hypergraph,NA_PatchHG_Attention,0.501934,Custom architecture



[F] KEY VERDICT
--------------------------------------------------------------------------------------------------------------
ESM -> best explicit representation: 0.45219 -> 0.47411 (+0.02193)
ConcatMLP -> best patch hypergraph: 0.46539 -> 0.50193 (+0.03654)
TestSet300 ΔAUPRC vs bindNode24: +0.03161
TestSetNew46 ΔAUPRC vs bindNode24: -0.00211
External verdict: CLEAR Test300 improvement; essentially competitive with bindNode24 on shifted Test46.
--------------------------------------------------------------------------------------------------------------

IMPORTANT INTERPRETATION NOTE:
The GEAR-MLP representation ladder and the custom architecture ladder use different trainers/objectives. Therefore, do NOT interpret BEST_REP_SCORE -> ConcatMLP as a pure architecture effect.
The clean architecture comparison is ConcatMLP vs the other custom fusion / patch-hypergraph models.

Saved consolidated outputs to:
/content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_ULTIMATE_V1_DATA_DRIVE

In [ ]:
# ============================================================
# NUCLEAR V2-A. Surface-topology feature block (Dev-selected)
# Append AFTER V1 has finished. Does not overwrite V1.
# ============================================================
from pathlib import Path
from collections import defaultdict, Counter
import json, math, numpy as np, pandas as pd
from scipy.spatial import cKDTree

V2_RUN_ROOT = BASE/'results'/'NUCLEAR_V2_SELECTIVE_MULTISCALE'
V2_RUN_ROOT.mkdir(parents=True, exist_ok=True)

V2_CACHE_ROOT = BASE/'cache'/'nuclear_v2_selective_multiscale'
V2_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

V2_FORCE_FEATURES = False

V2_CHARGE = {'K':1.0,'R':1.0,'H':0.10,'D':-1.0,'E':-1.0}
V2_KR  = set('KR')
V2_KRH = set('KRH')
V2_DE  = set('DE')
V2_POL = set('NQSTY')
V2_HBD = set('KRHNQSTYW')

V2_BACKBONE = {'N','CA','C','O','OXT'}

def v2_rep_coord(res):
    side = [
        np.asarray(a.coord, dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in V2_BACKBONE
    ]

    if side:
        return np.mean(np.stack(side), axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord, dtype=np.float32)

    atoms = [
        np.asarray(a.coord, dtype=np.float32)
        for a in res
    ]

    return np.mean(np.stack(atoms), axis=0)


def v2_coords(split, pid, seq):
    p = PDB_NORM_ROOT / split / f'{pid}.pdb'

    st = PDBParser(QUIET=True).get_structure(
        'v2',
        str(p)
    )

    m = next(st.get_models())
    rr = standard_residues(m['A'])

    if residue_sequence(rr) != seq:
        raise RuntimeError(
            f'{split}/{pid}: PDB/sequence mismatch'
        )

    return np.stack(
        [v2_rep_coord(r) for r in rr],
        axis=0
    ).astype(np.float32)


def v2_exposure_proxy(coords):
    """
    Label-free geometric exposure proxy.

    Fewer residues within 10 Å = more surface exposed.
    Returns [0,1], ranked within each protein.
    """

    tree = cKDTree(coords)

    n10 = np.array(
        [
            max(
                len(
                    tree.query_ball_point(
                        coords[i],
                        10.0
                    )
                ) - 1,
                0
            )
            for i in range(len(coords))
        ],
        dtype=np.float32
    )

    if len(coords) <= 1:
        return (
            np.ones(
                len(coords),
                dtype=np.float32
            ),
            n10
        )

    order = np.argsort(
        np.argsort(n10)
    )

    dense_rank = (
        order.astype(np.float32)
        / (len(coords) - 1)
    )

    exposure = 1.0 - dense_rank

    return (
        exposure.astype(np.float32),
        n10
    )


def v2_component_sizes(
    indices,
    coords,
    threshold=8.0
):
    if len(indices) == 0:
        return {}

    idx = np.asarray(
        indices,
        dtype=int
    )

    cc = np.asarray(
        coords,
        dtype=np.float32
    )

    parent = list(
        range(len(idx))
    )

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)

        if ra != rb:
            parent[rb] = ra

    if len(idx) > 1:
        for a, b in cKDTree(
            cc
        ).query_pairs(
            threshold
        ):
            union(a, b)

    groups = defaultdict(list)

    for j, rid in enumerate(idx):
        groups[
            find(j)
        ].append(
            int(rid)
        )

    out = {}

    for g in groups.values():
        for rid in g:
            out[rid] = len(g)

    return out


def build_na_surftopo16(
    split,
    pid,
    seq
):
    coords = v2_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    exposure, n10 = v2_exposure_proxy(
        coords
    )

    L = len(seq)

    q = np.array(
        [
            V2_CHARGE.get(
                a,
                0.0
            )
            for a in seq
        ],
        dtype=np.float32
    )

    # top ~50% geometrically exposed residues
    surface_mask = (
        exposure >= 0.50
    )

    surf_basic_idx = [
        i
        for i, a in enumerate(seq)
        if (
            surface_mask[i]
            and a in V2_KRH
        )
    ]

    comp = v2_component_sizes(
        surf_basic_idx,
        (
            coords[surf_basic_idx]
            if surf_basic_idx
            else np.zeros(
                (0, 3),
                np.float32
            )
        ),
        threshold=8.0
    )

    out = np.zeros(
        (L, 16),
        dtype=np.float32
    )

    for i in range(L):

        ids8 = np.asarray(
            tree.query_ball_point(
                coords[i],
                8.0
            ),
            dtype=int
        )

        ids8 = ids8[
            ids8 != i
        ]

        surf8 = (
            ids8[
                surface_mask[ids8]
            ]
            if len(ids8)
            else np.array(
                [],
                dtype=int
            )
        )

        # 0: geometric exposure
        out[i, 0] = exposure[i]

        # 1: packing density proxy
        out[i, 1] = min(
            float(n10[i]) / 40.0,
            1.0
        )

        # 2: fraction of local neighbors that are surface-like
        out[i, 2] = (
            len(surf8)
            / max(
                len(ids8),
                1
            )
        )

        if len(surf8):

            aa = [
                seq[j]
                for j in surf8
            ]

            qq = q[
                surf8
            ]

            dd = np.linalg.norm(
                coords[surf8]
                - coords[i][None, :],
                axis=1
            )

            # 3-6: surface chemistry
            out[i, 3] = (
                sum(
                    a in V2_KRH
                    for a in aa
                )
                / len(aa)
            )

            out[i, 4] = (
                sum(
                    a in V2_KR
                    for a in aa
                )
                / len(aa)
            )

            out[i, 5] = (
                sum(
                    a in V2_POL
                    for a in aa
                )
                / len(aa)
            )

            out[i, 6] = (
                sum(
                    a in V2_DE
                    for a in aa
                )
                / len(aa)
            )

            # 7: mean surface charge
            out[i, 7] = float(
                np.mean(qq)
            )

            # 8: distance-weighted electrostatic proxy
            out[i, 8] = float(
                np.tanh(
                    np.sum(
                        qq / (dd + 1.0)
                    )
                )
            )

            # 12: charge heterogeneity
            out[i, 12] = float(
                np.std(qq)
            )

            # 13: basic-acidic surface imbalance
            out[i, 13] = float(
                np.tanh(
                    (
                        sum(
                            a in V2_KRH
                            for a in aa
                        )
                        -
                        sum(
                            a in V2_DE
                            for a in aa
                        )
                    )
                    / 5.0
                )
            )

            # 14: H-bond donor-rich surface
            out[i, 14] = (
                sum(
                    a in V2_HBD
                    for a in aa
                )
                / len(aa)
            )

            # 15: tertiary / nonlocal surface contacts
            out[i, 15] = float(
                np.mean(
                    np.abs(
                        surf8 - i
                    ) > 10
                )
            )

        # 9: nearest exposed basic residue distance
        basic = [
            j
            for j in surf_basic_idx
            if j != i
        ]

        if basic:
            d = np.linalg.norm(
                coords[basic]
                - coords[i][None, :],
                axis=1
            )

            out[i, 9] = min(
                float(d.min()) / 20.0,
                1.0
            )

        else:
            out[i, 9] = 1.0

        # 10: exposed basic connected-component size
        out[i, 10] = min(
            comp.get(
                i,
                0
            ),
            20
        ) / 20.0

        # 11: distance to exposed positive centroid
        ids12 = np.asarray(
            tree.query_ball_point(
                coords[i],
                12.0
            ),
            dtype=int
        )

        pos = [
            j
            for j in ids12
            if (
                surface_mask[j]
                and seq[j] in V2_KRH
            )
        ]

        if pos:

            cent = coords[
                pos
            ].mean(
                axis=0
            )

            out[i, 11] = min(
                float(
                    np.linalg.norm(
                        coords[i] - cent
                    )
                )
                / 20.0,
                1.0
            )

        else:
            out[i, 11] = 1.0

    return out


V2_SURF_CACHE = (
    V2_CACHE_ROOT
    / 'surftopo16'
)

for split, df in SPLIT_DF.items():

    od = (
        V2_SURF_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )

    store = ctx.stores[
        STORE_KEY[split]
    ]

    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npy'
        )

        good = False

        if (
            fp.exists()
            and not V2_FORCE_FEATURES
        ):

            try:

                a = np.load(
                    fp,
                    mmap_mode='r'
                )

                good = (
                    a.shape
                    ==
                    (
                        len(r.sequence),
                        16
                    )
                )

            except Exception:
                good = False

        if not good:

            a = build_na_surftopo16(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.save(
                fp,
                a.astype(
                    np.float32
                )
            )

        else:

            a = np.load(
                fp
            ).astype(
                np.float32
            )

        set_block(
            store,
            r.prot_id,
            'NA_SURFTOPO16',
            a
        )

        if n % 250 == 0:
            print(
                split,
                n,
                '/',
                len(df)
            )

print(
    'NA_SURFTOPO16 attached: PASS'
)

DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300
NA_SURFTOPO16 attached: PASS


In [ ]:
# ============================================================
# NUCLEAR V2-B. Dev-only representation re-lock
# ============================================================

V2_REP_GRID = {

    'V2_ESM': [
        'ESM'
    ],

    # exact V1 winner/control
    'V2_V1_PATCHORG': [
        'ESM',
        'ACCESS',
        'NA_PATCH3D40',
        'NA_TOPO12'
    ],

    # new feature alone
    'V2_ESM_SURFTOPO': [
        'ESM',
        'NA_SURFTOPO16'
    ],

    # V1 winner + new topology
    'V2_PATCHORG_SURFTOPO': [
        'ESM',
        'ACCESS',
        'NA_PATCH3D40',
        'NA_TOPO12',
        'NA_SURFTOPO16'
    ],

    # see whether explicit electrostatics comes back
    # once surface topology is available
    'V2_PATCHORG_SURFTOPO_ELEC': [
        'ESM',
        'ACCESS',
        'ELEC3D',
        'NA_PATCH3D40',
        'NA_TOPO12',
        'NA_SURFTOPO16'
    ],
}


v2_rep_cv, v2_rep_folds = (
    suite.run_dev_cv_only(
        ctx,
        V2_REP_GRID,
        models=('MLP',),
        run_version='nuclear_v2_01_surftopo_rep',
    )
)


v2_rep_cv.to_csv(
    V2_RUN_ROOT
    / 'representation_cv.csv',
    index=False
)

v2_rep_folds.to_csv(
    V2_RUN_ROOT
    / 'representation_folds.csv',
    index=False
)


display(
    v2_rep_cv.sort_values(
        'CV_AUPRC',
        ascending=False
    )
)


V2_REP_NAME = str(
    v2_rep_cv
    .sort_values(
        'CV_AUPRC',
        ascending=False
    )
    .iloc[0]
    .representation
)


V2_BLOCKS = list(
    V2_REP_GRID[
        V2_REP_NAME
    ]
)


V2_REP_SCORE = float(
    v2_rep_cv.loc[
        v2_rep_cv
        .representation
        .eq(
            V2_REP_NAME
        ),
        'CV_AUPRC'
    ].iloc[0]
)


print(
    'V2 LOCKED REPRESENTATION:',
    V2_REP_NAME
)

print(
    'V2 BLOCKS:',
    V2_BLOCKS
)

print(
    'V2 Dev CV AUPRC:',
    V2_REP_SCORE
)


def v2_modality_spec(
    blocks
):

    spec = {}

    if 'ESM' in blocks:
        spec['ESM'] = [
            'ESM'
        ]

    surface = [
        b
        for b in [
            'ACCESS',
            'NA_SURFTOPO16'
        ]
        if b in blocks
    ]

    if surface:
        spec[
            'SURFACE'
        ] = surface

    patch = [
        b
        for b in [
            'NA_PATCH3D40',
            'NA_TOPO12'
        ]
        if b in blocks
    ]

    if patch:
        spec[
            'PATCH_LOCAL'
        ] = patch

    electro = [
        b
        for b in [
            'ELEC3D',
            'NA_SEQELECTRO32'
        ]
        if b in blocks
    ]

    if electro:
        spec[
            'ELECTRO'
        ] = electro

    chem = [
        b
        for b in [
            'NA_HBOND16'
        ]
        if b in blocks
    ]

    if chem:
        spec[
            'CHEM'
        ] = chem

    assigned = {
        x
        for v in spec.values()
        for x in v
    }

    other = [
        b
        for b in blocks
        if b not in assigned
    ]

    if other:
        spec[
            'OTHER'
        ] = other

    return spec


V2_MODALITY_SPEC = (
    v2_modality_spec(
        V2_BLOCKS
    )
)


def v2_concat_blocks(
    store,
    pid,
    blocks
):

    xs = [
        get_block(
            store,
            pid,
            b
        ).astype(
            np.float32,
            copy=False
        )

        for b in blocks
    ]

    return np.concatenate(
        xs,
        axis=1
    )


V2_DIMS = {}


for m, blocks in (
    V2_MODALITY_SPEC.items()
):

    sample_pid = str(
        train_df
        .iloc[0]
        .prot_id
    )

    V2_DIMS[m] = (
        v2_concat_blocks(
            ctx.stores[
                'train'
            ],
            sample_pid,
            blocks
        )
        .shape[1]
    )


print(
    'V2 modalities:',
    V2_MODALITY_SPEC
)

print(
    'V2 dims:',
    V2_DIMS
)


(
    V2_RUN_ROOT
    / 'locked_representation.json'
).write_text(
    json.dumps(
        {
            'representation':
                V2_REP_NAME,

            'blocks':
                V2_BLOCKS,

            'modalities':
                V2_MODALITY_SPEC,

            'dev_cv_auprc':
                V2_REP_SCORE,
        },
        indent=2
    )
)

DONE MLP V2_ESM 1 AUPRC 0.5687764675463045
DONE MLP V2_ESM 2 AUPRC 0.5289289634922406
DONE MLP V2_ESM 3 AUPRC 0.35337136222706256
DONE MLP V2_ESM 4 AUPRC 0.44265220942829486
DONE MLP V2_ESM 5 AUPRC 0.36631481935526583
DONE MLP V2_V1_PATCHORG 1 AUPRC 0.5786001232286393
DONE MLP V2_V1_PATCHORG 2 AUPRC 0.5457085837258228
DONE MLP V2_V1_PATCHORG 3 AUPRC 0.39943837567302537
DONE MLP V2_V1_PATCHORG 4 AUPRC 0.455046927064814
DONE MLP V2_V1_PATCHORG 5 AUPRC 0.3864226329863582
DONE MLP V2_ESM_SURFTOPO 1 AUPRC 0.5607369990575767
DONE MLP V2_ESM_SURFTOPO 2 AUPRC 0.5272236133734354
DONE MLP V2_ESM_SURFTOPO 3 AUPRC 0.3833503654733994
DONE MLP V2_ESM_SURFTOPO 4 AUPRC 0.44497983016005666
DONE MLP V2_ESM_SURFTOPO 5 AUPRC 0.38688357986478755
DONE MLP V2_PATCHORG_SURFTOPO 1 AUPRC 0.5717049472101969
DONE MLP V2_PATCHORG_SURFTOPO 2 AUPRC 0.545250332405171
DONE MLP V2_PATCHORG_SURFTOPO 3 AUPRC 0.3819680832934487
DONE MLP V2_PATCHORG_SURFTOPO 4 AUPRC 0.44068368667078883
DONE MLP V2_PATCHORG_SURFTOPO 5 AUPRC

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
3,MLP,V2_PATCHORG_SURFTOPO_ELEC,0.474367,0.084129,0.953692
4,MLP,V2_V1_PATCHORG,0.473043,0.086121,0.953731
2,MLP,V2_PATCHORG_SURFTOPO,0.468113,0.085681,0.953569
1,MLP,V2_ESM_SURFTOPO,0.460635,0.080796,0.952197
0,MLP,V2_ESM,0.452009,0.095802,0.948969


V2 LOCKED REPRESENTATION: V2_PATCHORG_SURFTOPO_ELEC
V2 BLOCKS: ['ESM', 'ACCESS', 'ELEC3D', 'NA_PATCH3D40', 'NA_TOPO12', 'NA_SURFTOPO16']
V2 Dev CV AUPRC: 0.4743665569718883
V2 modalities: {'ESM': ['ESM'], 'SURFACE': ['ACCESS', 'NA_SURFTOPO16'], 'PATCH_LOCAL': ['NA_PATCH3D40', 'NA_TOPO12'], 'ELECTRO': ['ELEC3D']}
V2 dims: {'ESM': 1280, 'SURFACE': 26, 'PATCH_LOCAL': 52, 'ELECTRO': 8}


427

In [ ]:
# ============================================================
# NUCLEAR V2-C. Selective multi-scale surface patches
# ============================================================

V2_RADII = (
    6.0,
    8.0,
    10.0,
    12.0
)

V2_PATCH_MAX = 48

V2_JACCARD_NMS = 0.65

V2_FORCE_PATCH = False


V2_PATCH_CACHE = (
    V2_CACHE_ROOT
    / 'selective_multiscale_patches'
)

V2_PATCH_CACHE.mkdir(
    parents=True,
    exist_ok=True
)


def v2_patch_feature(
    seq,
    coords,
    exposure,
    members,
    radius
):

    members = np.asarray(
        members,
        dtype=int
    )

    aa = [
        seq[j]
        for j in members
    ]

    cc = coords[
        members
    ]

    qq = np.asarray(
        [
            V2_CHARGE.get(
                a,
                0.0
            )
            for a in aa
        ],
        dtype=np.float32
    )

    n = len(
        members
    )


    cent = cc.mean(
        axis=0
    )

    radial = np.linalg.norm(
        cc
        - cent[None, :],
        axis=1
    )


    pair = []
    seqsep = []

    for x in range(n):

        for y in range(
            x + 1,
            n
        ):

            pair.append(
                float(
                    np.linalg.norm(
                        cc[x]
                        -
                        cc[y]
                    )
                )
            )

            seqsep.append(
                abs(
                    int(
                        members[x]
                    )
                    -
                    int(
                        members[y]
                    )
                )
            )


    pair = np.asarray(
        (
            pair
            if pair
            else [0.0]
        ),
        dtype=np.float32
    )

    seqsep = np.asarray(
        (
            seqsep
            if seqsep
            else [0.0]
        ),
        dtype=np.float32
    )


    comp = Counter(
        aa
    )

    probs = np.asarray(
        [
            c / n
            for c in comp.values()
        ],
        dtype=np.float32
    )


    entropy = float(
        -np.sum(
            probs
            *
            np.log(
                probs
                + 1e-12
            )
        )
        /
        math.log(20)
    )


    f = np.zeros(
        24,
        dtype=np.float32
    )


    # scale
    f[0] = (
        radius
        / 12.0
    )

    # size
    f[1] = (
        min(
            n,
            40
        )
        / 40.0
    )

    # surface exposure
    f[2] = float(
        np.mean(
            exposure[
                members
            ]
        )
    )

    f[3] = float(
        np.min(
            exposure[
                members
            ]
        )
    )

    # chemistry
    f[4] = (
        sum(
            a in V2_KR
            for a in aa
        )
        / n
    )

    f[5] = (
        sum(
            a in V2_KRH
            for a in aa
        )
        / n
    )

    f[6] = (
        sum(
            a in V2_POL
            for a in aa
        )
        / n
    )

    f[7] = (
        sum(
            a in V2_DE
            for a in aa
        )
        / n
    )

    f[8] = (
        sum(
            a in V2_HBD
            for a in aa
        )
        / n
    )

    f[9] = float(
        np.mean(qq)
    )

    f[10] = float(
        np.std(qq)
    )

    f[11] = float(
        np.tanh(
            (
                sum(
                    a in V2_KRH
                    for a in aa
                )
                -
                sum(
                    a in V2_DE
                    for a in aa
                )
            )
            / 5.0
        )
    )

    # geometry
    f[12] = min(
        float(
            radial.mean()
        )
        / 12.0,
        1.0
    )

    f[13] = min(
        float(
            radial.std()
        )
        / 8.0,
        1.0
    )

    f[14] = min(
        float(
            radial.max()
        )
        / 15.0,
        1.0
    )

    f[15] = min(
        float(
            pair.mean()
        )
        / 15.0,
        1.0
    )

    f[16] = min(
        float(
            pair.std()
        )
        / 10.0,
        1.0
    )

    # tertiary contacts
    f[17] = float(
        np.mean(
            seqsep > 10
        )
    )

    f[18] = min(
        float(
            seqsep.mean()
        )
        / 50.0,
        1.0
    )

    # chemistry diversity
    f[19] = entropy


    if n >= 3:

        z = (
            cc
            -
            cc.mean(
                axis=0,
                keepdims=True
            )
        )

        cov = (
            z.T
            @ z
            / max(
                n - 1,
                1
            )
        )

        eig = np.sort(
            np.clip(
                np.linalg.eigvalsh(
                    cov
                ),
                0,
                None
            )
        )[::-1]

        total = float(
            eig.sum()
            + 1e-8
        )

        # linearity
        f[20] = np.clip(
            (
                eig[0]
                -
                eig[1]
            )
            /
            (
                eig[0]
                + 1e-8
            ),
            0,
            1
        )

        # planarity
        f[21] = np.clip(
            (
                eig[1]
                -
                eig[2]
            )
            /
            (
                eig[0]
                + 1e-8
            ),
            0,
            1
        )

        # isotropy
        f[22] = np.clip(
            eig[2]
            / total,
            0,
            1
        )


    # broad sequence dispersion
    f[23] = float(
        np.mean(
            np.abs(
                members
                -
                int(
                    np.median(
                        members
                    )
                )
            )
            > 10
        )
    )


    return f


def v2_jaccard(
    a,
    b
):

    a = set(a)
    b = set(b)

    return (
        len(
            a & b
        )
        /
        max(
            len(
                a | b
            ),
            1
        )
    )


def build_v2_patch_graph(
    split,
    pid,
    seq
):

    coords = v2_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    exposure, _ = (
        v2_exposure_proxy(
            coords
        )
    )


    candidates = []


    for seed, a in enumerate(
        seq
    ):

        # Basic seeds may be only moderately exposed.
        if a in V2_KRH:

            if exposure[
                seed
            ] < 0.35:

                continue

        # Polar-only seeds must be strongly exposed.
        elif a in V2_POL:

            if exposure[
                seed
            ] < 0.65:

                continue

        else:
            continue


        for radius in V2_RADII:

            raw = (
                tree.query_ball_point(
                    coords[seed],
                    radius
                )
            )

            # mostly surface / near-surface residues
            members = tuple(
                sorted(
                    j
                    for j in raw
                    if exposure[j] >= 0.25
                )
            )


            if len(
                members
            ) < 4:
                continue


            if not any(
                seq[j] in V2_KRH
                for j in members
            ):
                continue


            feat = v2_patch_feature(
                seq,
                coords,
                exposure,
                members,
                radius
            )


            score = (
                2.0
                * feat[5]

                + 0.70
                * feat[6]

                - 0.70
                * feat[7]

                + 1.20
                * feat[2]

                + 0.25
                * feat[17]

                - 0.10
                * feat[0]
            )


            candidates.append(
                (
                    float(
                        score
                    ),
                    members,
                    feat
                )
            )


    # exact duplicate removal
    unique = {}


    for (
        score,
        members,
        feat
    ) in candidates:

        if (
            members not in unique
            or score
            >
            unique[
                members
            ][0]
        ):

            unique[
                members
            ] = (
                score,
                feat
            )


    ranked = sorted(
        [
            (
                s,
                m,
                f
            )
            for m, (
                s,
                f
            )
            in unique.items()
        ],
        key=lambda x:
            x[0],
        reverse=True
    )


    # Jaccard non-maximum suppression
    selected = []


    for (
        score,
        members,
        feat
    ) in ranked:

        if any(
            v2_jaccard(
                members,
                m2
            )
            >=
            V2_JACCARD_NMS

            for _, m2, _
            in selected
        ):
            continue


        selected.append(
            (
                score,
                members,
                feat
            )
        )


        if (
            len(
                selected
            )
            >=
            V2_PATCH_MAX
        ):
            break


    edge_res = []
    edge_patch = []
    patch_feat = []


    for pidx, (
        _,
        members,
        feat
    ) in enumerate(
        selected
    ):

        patch_feat.append(
            feat
        )


        for rid in members:

            edge_res.append(
                int(rid)
            )

            edge_patch.append(
                int(pidx)
            )


    return (
        np.asarray(
            edge_res,
            dtype=np.int64
        ),

        np.asarray(
            edge_patch,
            dtype=np.int64
        ),

        np.asarray(
            patch_feat,
            dtype=np.float32
        ).reshape(
            -1,
            24
        )
    )


V2_PATCH_RAM = {}

v2_audit = []


for split, df in SPLIT_DF.items():

    od = (
        V2_PATCH_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )


    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npz'
        )


        good = False


        if (
            fp.exists()
            and not V2_FORCE_PATCH
        ):

            try:

                z = np.load(
                    fp
                )

                good = (
                    z[
                        'patch_feat'
                    ].ndim
                    == 2

                    and
                    z[
                        'patch_feat'
                    ].shape[1]
                    == 24

                    and
                    len(
                        z[
                            'edge_res'
                        ]
                    )
                    ==
                    len(
                        z[
                            'edge_patch'
                        ]
                    )
                )

            except Exception:
                good = False


        if not good:

            er, ep, pf = (
                build_v2_patch_graph(
                    split,
                    str(
                        r.prot_id
                    ),
                    r.sequence
                )
            )


            np.savez_compressed(
                fp,
                edge_res=er,
                edge_patch=ep,
                patch_feat=pf
            )


        z = np.load(
            fp
        )


        er = z[
            'edge_res'
        ].astype(
            np.int64
        )


        ep = z[
            'edge_patch'
        ].astype(
            np.int64
        )


        pf = z[
            'patch_feat'
        ].astype(
            np.float32
        )


        V2_PATCH_RAM[
            (
                split,
                str(
                    r.prot_id
                )
            )
        ] = (
            er,
            ep,
            pf
        )


        covered = (
            len(
                np.unique(
                    er
                )
            )
            if len(er)
            else 0
        )


        v2_audit.append(
            {
                'dataset':
                    split,

                'prot_id':
                    str(
                        r.prot_id
                    ),

                'n_residues':
                    len(
                        r.sequence
                    ),

                'n_patches':
                    len(pf),

                'n_memberships':
                    len(er),

                'coverage':
                    covered
                    /
                    max(
                        len(
                            r.sequence
                        ),
                        1
                    ),
            }
        )


        if n % 250 == 0:

            print(
                split,
                n,
                '/',
                len(df)
            )


v2_patch_audit = pd.DataFrame(
    v2_audit
)


v2_patch_audit.to_csv(
    V2_RUN_ROOT
    / 'patch_audit.csv',
    index=False
)


display(
    v2_patch_audit
    .groupby(
        'dataset'
    )
    .agg(
        mean_patches=(
            'n_patches',
            'mean'
        ),

        median_patches=(
            'n_patches',
            'median'
        ),

        mean_coverage=(
            'coverage',
            'mean'
        ),

        median_coverage=(
            'coverage',
            'median'
        ),

        mean_memberships=(
            'n_memberships',
            'mean'
        ),
    )
    .reset_index()
)

DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300


,dataset,mean_patches,median_patches,mean_coverage,median_coverage,mean_memberships
0,DevSet1010,39.814851,48.0,0.599709,0.621822,335.160396
1,TestSet300,44.046667,48.0,0.575903,0.585326,371.463333
2,TestSetNew46,36.826087,47.0,0.629532,0.668627,310.130435


In [ ]:
# ============================================================
# NUCLEAR V2-D. Matched residue<->patch cross-attention model
# ============================================================

class V2PatchDataset(
    Dataset
):

    def __init__(
        self,
        split,
        df
    ):

        self.split = split

        self.df = df.reset_index(
            drop=True
        )

        self.store = ctx.stores[
            STORE_KEY[
                split
            ]
        ]


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        i
    ):

        r = self.df.iloc[
            i
        ]

        pid = str(
            r.prot_id
        )


        mods = {

            m:
                v2_concat_blocks(
                    self.store,
                    pid,
                    blocks
                )
                .astype(
                    np.float32,
                    copy=False
                )

            for m, blocks
            in V2_MODALITY_SPEC.items()
        }


        er, ep, pf = (
            V2_PATCH_RAM[
                (
                    self.split,
                    pid
                )
            ]
        )


        return {
            'pid':
                pid,

            'mods':
                mods,

            'y':
                np.asarray(
                    r.nuclear_binding,
                    dtype=np.float32
                ),

            'edge_res':
                er,

            'edge_patch':
                ep,

            'patch_feat':
                pf,
        }


def v2_collate(
    batch
):

    keys = sorted(
        batch[0][
            'mods'
        ]
    )


    mods = {

        k:
            torch.from_numpy(
                np.concatenate(
                    [
                        b[
                            'mods'
                        ][k]
                        for b in batch
                    ],
                    axis=0
                )
            )
            .float()

        for k in keys
    }


    y = torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()


    er_all = []
    ep_all = []
    pf_all = []
    protein_index = []


    roff = 0
    poff = 0


    for pidx, b in enumerate(
        batch
    ):

        L = len(
            b['y']
        )

        P = len(
            b[
                'patch_feat'
            ]
        )


        protein_index.append(
            torch.full(
                (L,),
                pidx,
                dtype=torch.long
            )
        )


        if P:

            pf_all.append(
                torch.from_numpy(
                    b[
                        'patch_feat'
                    ]
                )
                .float()
            )


        if len(
            b[
                'edge_res'
            ]
        ):

            er_all.append(
                torch.from_numpy(
                    b[
                        'edge_res'
                    ]
                )
                .long()
                + roff
            )

            ep_all.append(
                torch.from_numpy(
                    b[
                        'edge_patch'
                    ]
                )
                .long()
                + poff
            )


        roff += L
        poff += P


    patch_feat = (

        torch.cat(
            pf_all,
            0
        )

        if pf_all

        else torch.zeros(
            (
                0,
                24
            ),
            dtype=torch.float32
        )
    )


    edge_res = (

        torch.cat(
            er_all,
            0
        )

        if er_all

        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )


    edge_patch = (

        torch.cat(
            ep_all,
            0
        )

        if ep_all

        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )


    return {

        'mods':
            mods,

        'y':
            y,

        'protein_index':
            torch.cat(
                protein_index,
                0
            ),

        'edge_res':
            edge_res,

        'edge_patch':
            edge_patch,

        'patch_feat':
            patch_feat,
    }


class V2ResidueEncoder(
    nn.Module
):

    def __init__(
        self,
        dims,
        latent=192,
        heads=6,
        drop=0.20
    ):

        super().__init__()


        assert (
            latent
            % heads
            == 0
        )


        self.keys = sorted(
            dims
        )


        self.enc = nn.ModuleDict(
            {

                k:
                    nn.Sequential(
                        nn.LayerNorm(
                            d
                        ),
                        nn.Linear(
                            d,
                            latent
                        ),
                        nn.GELU(),
                        nn.Dropout(
                            drop
                        )
                    )

                for k, d
                in dims.items()
            }
        )


        self.attn = (
            nn.MultiheadAttention(
                latent,
                heads,
                dropout=drop,
                batch_first=True
            )
        )


    def forward(
        self,
        x
    ):

        z = torch.stack(
            [
                self.enc[k](
                    x[k]
                )
                for k in self.keys
            ],
            dim=1
        )


        a, _ = self.attn(
            z,
            z,
            z,
            need_weights=False
        )


        return (
            z + a
        ).mean(
            dim=1
        )


class NuclearPatchV2(
    nn.Module
):

    def __init__(
        self,
        dims,
        latent=192,
        heads=6,
        drop=0.20,
        use_patch=True
    ):

        super().__init__()


        self.use_patch = (
            use_patch
        )

        self.latent = (
            latent
        )


        self.residue_encoder = (
            V2ResidueEncoder(
                dims,
                latent=latent,
                heads=heads,
                drop=drop
            )
        )


        self.patch_static = (
            nn.Sequential(
                nn.LayerNorm(
                    24
                ),
                nn.Linear(
                    24,
                    64
                ),
                nn.GELU()
            )
        )


        self.patch_encoder = (
            nn.Sequential(
                nn.Linear(
                    latent * 2
                    + 64,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU(),
            )
        )


        self.q = nn.Linear(
            latent,
            latent,
            bias=False
        )

        self.k = nn.Linear(
            latent,
            latent,
            bias=False
        )

        self.v = nn.Linear(
            latent,
            latent,
            bias=False
        )


        self.delta = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                ),
            )
        )


        # per-residue learnable patch gate
        self.gate = (
            nn.Sequential(
                nn.LayerNorm(
                    latent * 2
                ),
                nn.Linear(
                    latent * 2,
                    1
                )
            )
        )


        # initial gate ~= sigmoid(-1.5) ~= 0.18
        nn.init.zeros_(
            self.gate[
                -1
            ].weight
        )

        nn.init.constant_(
            self.gate[
                -1
            ].bias,
            -1.5
        )


        self.head = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    128
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    128,
                    1
                )
            )
        )


    def index_mean(
        self,
        x,
        index,
        n
    ):

        out = x.new_zeros(
            (
                n,
                x.shape[-1]
            )
        )

        cnt = x.new_zeros(
            (
                n,
                1
            )
        )


        if index.numel():

            out = out.index_add(
                0,
                index,
                x
            )

            cnt = cnt.index_add(
                0,
                index,
                x.new_ones(
                    (
                        len(index),
                        1
                    )
                )
            )


        return (
            out
            /
            cnt.clamp_min(
                1.0
            )
        )


    def index_max(
        self,
        x,
        index,
        n
    ):

        if n == 0:

            return x.new_zeros(
                (
                    0,
                    x.shape[-1]
                )
            )


        base = x.new_full(
            (
                n,
                x.shape[-1]
            ),
            -1e9
        )


        if index.numel():

            idx = index[
                :,
                None
            ].expand(
                -1,
                x.shape[-1]
            )


            # OUT-OF-PLACE:
            # safe for autograd
            out = (
                base.scatter_reduce(
                    0,
                    idx,
                    x,
                    reduce='amax',
                    include_self=True
                )
            )

        else:

            out = base


        return torch.where(
            out < -1e8,
            torch.zeros_like(
                out
            ),
            out
        )


    def forward(
        self,
        mods,
        edge_res,
        edge_patch,
        patch_feat,
        return_aux=False
    ):

        h = self.residue_encoder(
            mods
        )


        gate_mean = h.new_tensor(
            0.0
        )


        if (
            self.use_patch
            and patch_feat.shape[0] > 0
            and edge_res.numel() > 0
        ):

            npatch = (
                patch_feat.shape[0]
            )

            nres = (
                h.shape[0]
            )


            mean = self.index_mean(
                h[
                    edge_res
                ],
                edge_patch,
                npatch
            )


            mx = self.index_max(
                h[
                    edge_res
                ],
                edge_patch,
                npatch
            )


            stat = (
                self.patch_static(
                    patch_feat
                )
            )


            hp = (
                self.patch_encoder(
                    torch.cat(
                        [
                            mean,
                            mx,
                            stat
                        ],
                        dim=-1
                    )
                )
            )


            # residue-to-patch cross-attention
            q = self.q(
                h[
                    edge_res
                ]
            )

            k = self.k(
                hp[
                    edge_patch
                ]
            )


            edge_logits = (
                (
                    q * k
                )
                .sum(
                    dim=-1
                )
                /
                math.sqrt(
                    self.latent
                )
            )


            # stable softmax over incident patches
            # separately for each residue
            base = (
                edge_logits
                .new_full(
                    (
                        nres,
                    ),
                    -1e9
                )
            )


            max_r = (
                base.scatter_reduce(
                    0,
                    edge_res,
                    edge_logits,
                    reduce='amax',
                    include_self=True
                )
            )


            max_r = torch.where(
                max_r < -1e8,
                torch.zeros_like(
                    max_r
                ),
                max_r
            )


            ew = torch.exp(
                edge_logits
                -
                max_r[
                    edge_res
                ]
            )


            denom = (
                ew
                .new_zeros(
                    (
                        nres,
                        1
                    )
                )
                .index_add(
                    0,
                    edge_res,
                    ew[
                        :,
                        None
                    ]
                )
            )


            weighted = (
                self.v(
                    hp[
                        edge_patch
                    ]
                )
                *
                ew[
                    :,
                    None
                ]
            )


            msg = (
                h
                .new_zeros(
                    (
                        nres,
                        self.latent
                    )
                )
                .index_add(
                    0,
                    edge_res,
                    weighted
                )
            )


            msg = (
                msg
                /
                denom.clamp_min(
                    1e-8
                )
            )


            # residue-specific patch gate
            g = torch.sigmoid(
                self.gate(
                    torch.cat(
                        [
                            h,
                            msg
                        ],
                        dim=-1
                    )
                )
            )


            incident = (
                h
                .new_zeros(
                    (
                        nres,
                        1
                    )
                )
                .index_add(
                    0,
                    edge_res,
                    h.new_ones(
                        (
                            len(
                                edge_res
                            ),
                            1
                        )
                    )
                )
            )


            g = (
                g
                *
                (
                    incident > 0
                ).float()
            )


            h = (
                h
                +
                g
                *
                self.delta(
                    msg
                )
            )


            active = (
                incident
                .squeeze(
                    -1
                )
                > 0
            )


            gate_mean = (

                g[
                    active
                ].mean()

                if active.any()

                else h.new_tensor(
                    0.0
                )
            )


        logits = (
            self.head(
                h
            )
            .squeeze(
                -1
            )
        )


        if return_aux:

            return (
                logits,
                {
                    'gate_mean':
                        gate_mean
                }
            )


        return logits

In [ ]:
# ============================================================
# NUCLEAR V2-E.
# Protein-balanced BCE + hard-negative ranking
# 2x2 MATCHED ABLATION
# ============================================================

V2_BATCH = 4

V2_EPOCHS = 36

V2_PATIENCE = 6

V2_LR = 3e-4

V2_WD = 1e-4

V2_HARDNEG_LAMBDA = 0.20

V2_HARDNEG_MARGIN = 0.50

V2_CODE_VERSION = (
    'nuclear_v2_selective_multiscale_r1'
)


V2_STATE_ROOT = (
    V2_RUN_ROOT
    / 'states'
)

V2_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


V2_VARIANTS = {

    'NoPatch_BCE': {
        'use_patch':
            False,

        'hardneg':
            False
    },

    'NoPatch_HardNeg': {
        'use_patch':
            False,

        'hardneg':
            True
    },

    'Patch_BCE': {
        'use_patch':
            True,

        'hardneg':
            False
    },

    'Patch_HardNeg': {
        'use_patch':
            True,

        'hardneg':
            True
    },
}


def v2_move(
    b
):

    return {

        'mods': {
            k:
                v.to(
                    DEVICE,
                    non_blocking=True
                )

            for k, v
            in b[
                'mods'
            ].items()
        },

        'y':
            b[
                'y'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'protein_index':
            b[
                'protein_index'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_res':
            b[
                'edge_res'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_patch':
            b[
                'edge_patch'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'patch_feat':
            b[
                'patch_feat'
            ].to(
                DEVICE,
                non_blocking=True
            ),
    }


def v2_loss(
    logits,
    y,
    protein_index,
    pos_weight,
    use_hardneg
):

    # ----------------------------------
    # protein-balanced weighted BCE
    # ----------------------------------

    raw = (
        F.binary_cross_entropy_with_logits(
            logits,
            y,
            pos_weight=pos_weight,
            reduction='none'
        )
    )


    nprot = (
        int(
            protein_index
            .max()
            .item()
        )
        + 1
    )


    sums = (
        raw
        .new_zeros(
            (
                nprot,
            )
        )
        .index_add(
            0,
            protein_index,
            raw
        )
    )


    counts = (
        raw
        .new_zeros(
            (
                nprot,
            )
        )
        .index_add(
            0,
            protein_index,
            raw.new_ones(
                len(raw)
            )
        )
    )


    base = (
        sums
        /
        counts.clamp_min(
            1.0
        )
    ).mean()


    if not use_hardneg:

        return (
            base,
            raw.new_tensor(
                0.0
            )
        )


    # ----------------------------------
    # within-protein hard-negative rank
    # ----------------------------------

    rank_terms = []


    for p in range(
        nprot
    ):

        mask = (
            protein_index
            == p
        )


        lp = logits[
            mask
        ]


        yp = (
            y[
                mask
            ]
            > 0.5
        )


        # need both positives and negatives
        if (
            not yp.any()
            or
            (
                ~yp
            ).sum()
            == 0
        ):

            continue


        pos_logits = (
            lp[
                yp
            ]
        )


        neg_logits = (
            lp[
                ~yp
            ]
        )


        k = min(
            max(
                4,
                int(
                    2
                    *
                    len(
                        pos_logits
                    )
                )
            ),
            32,
            len(
                neg_logits
            )
        )


        hard_neg = (
            torch.topk(
                neg_logits,
                k=k
            )
            .values
        )


        # Positive interface anchor.
        pos_anchor = (
            pos_logits.mean()
        )


        rank_terms.append(
            F.softplus(
                hard_neg
                -
                pos_anchor
                +
                V2_HARDNEG_MARGIN
            )
            .mean()
        )


    rank = (

        torch.stack(
            rank_terms
        ).mean()

        if rank_terms

        else base.new_tensor(
            0.0
        )
    )


    return (
        base
        +
        V2_HARDNEG_LAMBDA
        *
        rank,
        rank
    )


def v2_predict(
    model,
    ds
):

    loader = DataLoader(
        ds,
        batch_size=V2_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == 'cuda'
        ),
        collate_fn=v2_collate
    )


    pp = []
    yy = []
    gates = []


    model.eval()


    with torch.no_grad():

        for b in loader:

            b = v2_move(
                b
            )


            logits, aux = model(
                b[
                    'mods'
                ],
                b[
                    'edge_res'
                ],
                b[
                    'edge_patch'
                ],
                b[
                    'patch_feat'
                ],
                return_aux=True
            )


            pp.append(
                torch.sigmoid(
                    logits
                )
                .cpu()
                .numpy()
            )


            yy.append(
                b[
                    'y'
                ]
                .cpu()
                .numpy()
            )


            gates.append(
                float(
                    aux[
                        'gate_mean'
                    ]
                    .detach()
                    .cpu()
                )
            )


    return (
        np.concatenate(
            pp
        ),
        np.concatenate(
            yy
        ),
        float(
            np.mean(
                gates
            )
        )
    )


def v2_train_fold(
    variant,
    tr_ds,
    va_ds,
    fold
):

    cfg = V2_VARIANTS[
        variant
    ]


    torch.manual_seed(
        SEED
        + 700
        + fold
    )


    np.random.seed(
        SEED
        + 700
        + fold
    )


    model = NuclearPatchV2(
        V2_DIMS,
        latent=192,
        heads=6,
        drop=0.20,
        use_patch=cfg[
            'use_patch'
        ]
    ).to(
        DEVICE
    )


    ytr = np.concatenate(
        tr_ds
        .df
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        )
        .tolist()
    )


    npos = max(
        float(
            ytr.sum()
        ),
        1.0
    )


    nneg = max(
        float(
            len(ytr)
            -
            ytr.sum()
        ),
        1.0
    )


    pos_weight = torch.tensor(
        np.sqrt(
            nneg
            / npos
        ),
        dtype=torch.float32,
        device=DEVICE
    )


    opt = torch.optim.AdamW(
        model.parameters(),
        lr=V2_LR,
        weight_decay=V2_WD
    )


    loader = DataLoader(
        tr_ds,
        batch_size=V2_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == 'cuda'
        ),
        collate_fn=v2_collate
    )


    best_ap = -1.0

    best_state = None

    bad = 0


    for epoch in range(
        1,
        V2_EPOCHS + 1
    ):

        model.train()

        ls = []
        rs = []


        for b in loader:

            b = v2_move(
                b
            )


            opt.zero_grad(
                set_to_none=True
            )


            logits = model(
                b[
                    'mods'
                ],
                b[
                    'edge_res'
                ],
                b[
                    'edge_patch'
                ],
                b[
                    'patch_feat'
                ]
            )


            loss, rank = (
                v2_loss(
                    logits,
                    b[
                        'y'
                    ],
                    b[
                        'protein_index'
                    ],
                    pos_weight,
                    cfg[
                        'hardneg'
                    ]
                )
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )


            opt.step()


            ls.append(
                float(
                    loss
                    .detach()
                    .cpu()
                )
            )


            rs.append(
                float(
                    rank
                    .detach()
                    .cpu()
                )
            )


        vp, vy, vg = (
            v2_predict(
                model,
                va_ds
            )
        )


        vap = float(
            average_precision_score(
                vy,
                vp
            )
        )


        if (
            vap
            >
            best_ap
            + 1e-5
        ):

            best_ap = vap


            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model
                .state_dict()
                .items()
            }


            bad = 0


        else:

            bad += 1


        if (
            epoch == 1
            or
            epoch % 5 == 0
        ):

            print(
                variant,
                'fold',
                fold,
                'epoch',
                epoch,
                'loss',
                round(
                    float(
                        np.mean(
                            ls
                        )
                    ),
                    4
                ),
                'rank',
                round(
                    float(
                        np.mean(
                            rs
                        )
                    ),
                    4
                ),
                'val_AP',
                round(
                    vap,
                    5
                ),
                'gate',
                round(
                    vg,
                    4
                )
            )


        if bad >= V2_PATIENCE:
            break


    model.load_state_dict(
        best_state
    )


    return (
        model,
        best_state
    )


def v2_run_cv(
    variant
):

    offsets = {}

    cursor = 0


    for r in train_df.itertuples():

        L = len(
            r.sequence
        )

        offsets[
            str(
                r.prot_id
            )
        ] = (
            cursor,
            cursor + L
        )

        cursor += L


    oof = np.full(
        cursor,
        np.nan,
        dtype=np.float32
    )


    rows = []

    states = {}


    adir = (
        V2_STATE_ROOT
        / variant
    )

    adir.mkdir(
        parents=True,
        exist_ok=True
    )


    for fold, val_ids in (
        OFFICIAL_FOLDS.items()
    ):

        val_set = set(
            val_ids
        )


        tr_df = train_df[
            ~train_df
            .prot_id
            .isin(
                val_set
            )
        ].copy()


        va_df = train_df[
            train_df
            .prot_id
            .isin(
                val_set
            )
        ].copy()


        tr_ds = V2PatchDataset(
            'DevSet1010',
            tr_df
        )


        va_ds = V2PatchDataset(
            'DevSet1010',
            va_df
        )


        state_fp = (
            adir
            / f'fold{fold}.pt'
        )


        pred_fp = (
            adir
            / f'fold{fold}_val.npy'
        )


        meta_fp = (
            adir
            / f'fold{fold}.json'
        )


        expected = int(
            va_df
            .sequence
            .str
            .len()
            .sum()
        )


        reuse = False


        if (
            state_fp.exists()
            and
            pred_fp.exists()
            and
            meta_fp.exists()
        ):

            try:

                meta = json.loads(
                    meta_fp
                    .read_text()
                )


                pred = np.load(
                    pred_fp
                )


                reuse = (

                    meta.get(
                        'code_version'
                    )
                    ==
                    V2_CODE_VERSION

                    and

                    meta.get(
                        'representation'
                    )
                    ==
                    V2_REP_NAME

                    and

                    meta.get(
                        'n_val'
                    )
                    ==
                    expected

                    and

                    pred.shape
                    ==
                    (
                        expected,
                    )

                    and

                    np.all(
                        np.isfinite(
                            pred
                        )
                    )
                )


            except Exception:

                reuse = False


        if reuse:

            print(
                variant,
                'fold',
                fold,
                'REUSE'
            )


            state = torch.load(
                state_fp,
                map_location='cpu'
            )


            pred = np.load(
                pred_fp
            ).astype(
                np.float32
            )


            model = NuclearPatchV2(
                V2_DIMS,
                use_patch=V2_VARIANTS[
                    variant
                ][
                    'use_patch'
                ]
            ).to(
                DEVICE
            )


            model.load_state_dict(
                state
            )


            _, _, gate = (
                v2_predict(
                    model,
                    va_ds
                )
            )


            del model


        else:

            print(
                variant,
                'fold',
                fold,
                'TRAIN'
            )


            model, state = (
                v2_train_fold(
                    variant,
                    tr_ds,
                    va_ds,
                    fold
                )
            )


            pred, _, gate = (
                v2_predict(
                    model,
                    va_ds
                )
            )


            torch.save(
                state,
                state_fp
            )


            np.save(
                pred_fp,
                pred.astype(
                    np.float32
                )
            )


            meta_fp.write_text(
                json.dumps(
                    {
                        'code_version':
                            V2_CODE_VERSION,

                        'representation':
                            V2_REP_NAME,

                        'blocks':
                            V2_BLOCKS,

                        'n_val':
                            expected,
                    },
                    indent=2
                )
            )


            del model


            if DEVICE.type == 'cuda':

                torch.cuda.empty_cache()


        states[
            fold
        ] = state


        pos = 0


        for r in va_df.itertuples():

            L = len(
                r.sequence
            )


            a, b = offsets[
                str(
                    r.prot_id
                )
            ]


            oof[
                a:b
            ] = pred[
                pos:
                pos + L
            ]


            pos += L


        yv = np.concatenate(
            va_df
            .nuclear_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.float32
                    )
            )
            .tolist()
        )


        rows.append(
            {
                'variant':
                    variant,

                'fold':
                    fold,

                'AUPRC':
                    float(
                        average_precision_score(
                            yv,
                            pred
                        )
                    ),

                'n_positive':
                    int(
                        yv.sum()
                    ),

                'gate_mean':
                    gate,
            }
        )


    f = pd.DataFrame(
        rows
    )


    yall = np.concatenate(
        train_df
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        )
        .tolist()
    )


    return (
        {
            'variant':
                variant,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                float(
                    average_precision_score(
                        yall,
                        oof
                    )
                ),

            'gate_mean':
                float(
                    f.gate_mean.mean()
                ),
        },

        f,

        oof,

        states
    )


V2_RESULTS = {}

V2_FOLDS = []

V2_OOF = {}

V2_STATES = {}


for variant in V2_VARIANTS:

    print(
        '\n'
        + '=' * 88
    )

    print(
        'V2:',
        variant
    )

    print(
        '=' * 88
    )


    s, f, o, st = (
        v2_run_cv(
            variant
        )
    )


    V2_RESULTS[
        variant
    ] = s

    V2_FOLDS.append(
        f
    )

    V2_OOF[
        variant
    ] = o

    V2_STATES[
        variant
    ] = st


v2_summary = (
    pd.DataFrame(
        V2_RESULTS.values()
    )
    .sort_values(
        'CV_AUPRC_mean',
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


v2_folds = pd.concat(
    V2_FOLDS,
    ignore_index=True
)


v2_summary.to_csv(
    V2_RUN_ROOT
    / 'v2_cv_summary.csv',
    index=False
)


v2_folds.to_csv(
    V2_RUN_ROOT
    / 'v2_cv_folds.csv',
    index=False
)


display(
    v2_summary
)


def v2score(
    name
):

    return float(
        v2_summary.loc[
            v2_summary
            .variant
            .eq(
                name
            ),
            'CV_AUPRC_mean'
        ]
        .iloc[0]
    )


# ============================================================
# MATCHED CAUSAL/MECHANISTIC DELTAS
# ============================================================

v2_ablation = pd.DataFrame(
    [

        {
            'effect':
                'Hard-negative loss without patches',

            'delta_AUPRC':
                v2score(
                    'NoPatch_HardNeg'
                )
                -
                v2score(
                    'NoPatch_BCE'
                )
        },

        {
            'effect':
                'Patch reasoning under BCE',

            'delta_AUPRC':
                v2score(
                    'Patch_BCE'
                )
                -
                v2score(
                    'NoPatch_BCE'
                )
        },

        {
            'effect':
                'Patch reasoning under HardNeg',

            'delta_AUPRC':
                v2score(
                    'Patch_HardNeg'
                )
                -
                v2score(
                    'NoPatch_HardNeg'
                )
        },

        {
            'effect':
                'Hard-negative loss with patches',

            'delta_AUPRC':
                v2score(
                    'Patch_HardNeg'
                )
                -
                v2score(
                    'Patch_BCE'
                )
        },
    ]
)


display(
    v2_ablation
)


v2_ablation.to_csv(
    V2_RUN_ROOT
    / 'matched_ablation.csv',
    index=False
)


# ============================================================
# Complexity-aware Dev lock
# ============================================================

complexity = {

    'NoPatch_BCE':
        0,

    'NoPatch_HardNeg':
        1,

    'Patch_BCE':
        2,

    'Patch_HardNeg':
        3,
}


ordered = sorted(
    V2_VARIANTS,
    key=lambda x:
        complexity[x]
)


winner = (
    ordered[0]
)


for name in ordered[1:]:

    if (
        v2score(
            name
        )
        -
        v2score(
            winner
        )
        >= 0.002
    ):

        winner = name


V2_LOCKED = (
    winner
)


V2_LOCKED_SCORE = (
    v2score(
        winner
    )
)


print(
    'V2 LOCKED:',
    V2_LOCKED,
    V2_LOCKED_SCORE
)


V2: NoPatch_BCE
NoPatch_BCE fold 1 TRAIN
NoPatch_BCE fold 1 epoch 1 loss 0.2899 rank 0.0 val_AP 0.51087 gate 0.0
NoPatch_BCE fold 1 epoch 5 loss 0.0959 rank 0.0 val_AP 0.54233 gate 0.0
NoPatch_BCE fold 1 epoch 10 loss 0.0302 rank 0.0 val_AP 0.53327 gate 0.0
NoPatch_BCE fold 2 TRAIN
NoPatch_BCE fold 2 epoch 1 loss 0.2792 rank 0.0 val_AP 0.54354 gate 0.0
NoPatch_BCE fold 2 epoch 5 loss 0.0867 rank 0.0 val_AP 0.49911 gate 0.0
NoPatch_BCE fold 3 TRAIN
NoPatch_BCE fold 3 epoch 1 loss 0.3034 rank 0.0 val_AP 0.3689 gate 0.0
NoPatch_BCE fold 3 epoch 5 loss 0.0905 rank 0.0 val_AP 0.38857 gate 0.0
NoPatch_BCE fold 3 epoch 10 loss 0.0247 rank 0.0 val_AP 0.35807 gate 0.0
NoPatch_BCE fold 4 TRAIN
NoPatch_BCE fold 4 epoch 1 loss 0.318 rank 0.0 val_AP 0.39154 gate 0.0
NoPatch_BCE fold 4 epoch 5 loss 0.0967 rank 0.0 val_AP 0.47219 gate 0.0
NoPatch_BCE fold 4 epoch 10 loss 0.0229 rank 0.0 val_AP 0.3782 gate 0.0
NoPatch_BCE fold 5 TRAIN
NoPatch_BCE fold 5 epoch 1 loss 0.2908 rank 0.0 val_AP 0.34149 gat

,variant,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,gate_mean
0,Patch_BCE,0.483032,0.084115,0.423223,0.151903
1,NoPatch_BCE,0.477271,0.090648,0.425337,0.000000
2,Patch_HardNeg,0.468608,0.095257,0.462364,0.162406
3,NoPatch_HardNeg,0.458039,0.093593,0.432353,0.000000


,effect,delta_AUPRC
0,Hard-negative loss without patches,-0.019232
1,Patch reasoning under BCE,0.005761
2,Patch reasoning under HardNeg,0.010569
3,Hard-negative loss with patches,-0.014424


V2 LOCKED: Patch_BCE 0.48303182428242275


In [ ]:
# ============================================================
# NUCLEAR V2-F.
# External confirmation of DEV-LOCKED V2 only
# ============================================================

def v2_best_mcc_threshold(
    y,
    p
):

    best = (
        -999,
        None
    )


    for t in np.linspace(
        0.01,
        0.99,
        199
    ):

        m = matthews_corrcoef(
            y,
            (
                p >= t
            ).astype(
                int
            )
        )


        if m > best[0]:

            best = (
                m,
                float(t)
            )


    return best[1]


v2_dev_y = np.concatenate(
    train_df
    .nuclear_binding
    .apply(
        lambda x:
            np.asarray(
                x,
                dtype=np.int8
            )
    )
    .tolist()
)


V2_THR = (
    v2_best_mcc_threshold(
        v2_dev_y,
        V2_OOF[
            V2_LOCKED
        ]
    )
)


def v2_predict_external(
    split,
    variant
):

    ds = V2PatchDataset(
        split,
        SPLIT_DF[
            split
        ]
    )


    preds = []


    for fold, state in (
        V2_STATES[
            variant
        ].items()
    ):

        model = NuclearPatchV2(
            V2_DIMS,
            latent=192,
            heads=6,
            drop=0.20,
            use_patch=V2_VARIANTS[
                variant
            ][
                'use_patch'
            ]
        ).to(
            DEVICE
        )


        model.load_state_dict(
            state
        )


        p, _, _ = (
            v2_predict(
                model,
                ds
            )
        )


        preds.append(
            p
        )


        del model


        if DEVICE.type == 'cuda':

            torch.cuda.empty_cache()


    return np.mean(
        np.stack(
            preds
        ),
        axis=0
    )


def v2_metrics(
    y,
    p,
    t=0.5
):

    pred = (
        p >= t
    ).astype(
        int
    )


    return {

        'AUPRC':
            float(
                average_precision_score(
                    y,
                    p
                )
            ),

        'AUROC':
            float(
                roc_auc_score(
                    y,
                    p
                )
            ),

        'F1':
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'MCC':
            float(
                matthews_corrcoef(
                    y,
                    pred
                )
            ),

        'precision':
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'recall':
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),
    }


ext_rows = []


for split in [
    'TestSet300',
    'TestSetNew46'
]:

    y = np.concatenate(
        SPLIT_DF[
            split
        ]
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    p = v2_predict_external(
        split,
        V2_LOCKED
    )


    m05 = v2_metrics(
        y,
        p,
        0.5
    )


    mdev = v2_metrics(
        y,
        p,
        V2_THR
    )


    row = {

        'dataset':
            split,

        'variant':
            V2_LOCKED,

        'Dev_CV_AUPRC':
            V2_LOCKED_SCORE,

        'Dev_OOF_MCC_threshold':
            V2_THR,
    }


    row.update(
        {
            f'{k}_fixed0.5':
                v

            for k, v
            in m05.items()
        }
    )


    row.update(
        {
            f'{k}_OOFthr':
                v

            for k, v
            in mdev.items()
        }
    )


    ext_rows.append(
        row
    )


    pd.DataFrame(
        {

            'prot_id':
                np.concatenate(
                    [
                        np.repeat(
                            str(
                                r.prot_id
                            ),
                            len(
                                r.sequence
                            )
                        )

                        for r
                        in SPLIT_DF[
                            split
                        ].itertuples()
                    ]
                ),

            'residue_index':
                np.concatenate(
                    [
                        np.arange(
                            len(
                                r.sequence
                            )
                        )

                        for r
                        in SPLIT_DF[
                            split
                        ].itertuples()
                    ]
                ),

            'y_true':
                y,

            'probability':
                p,
        }
    ).to_csv(
        V2_RUN_ROOT
        / f'predictions_{split}.csv.gz',
        index=False
    )


v2_external = pd.DataFrame(
    ext_rows
)


display(
    v2_external
)


v2_external.to_csv(
    V2_RUN_ROOT
    / 'FINAL_V2_EXTERNAL.csv',
    index=False
)


# ============================================================
# V2 vs V1 vs direct baselines
# ============================================================

comparison_rows = []


for _, r in (
    v2_external.iterrows()
):

    comparison_rows.append(
        {

            'dataset':
                r[
                    'dataset'
                ],

            'method':
                f'V2 {V2_LOCKED}',

            'AUPRC':
                r[
                    'AUPRC_fixed0.5'
                ],

            'AUROC':
                r[
                    'AUROC_fixed0.5'
                ],

            'F1':
                r[
                    'F1_fixed0.5'
                ],

            'MCC':
                r[
                    'MCC_fixed0.5'
                ],
        }
    )


# V1 current headline
if (
    'headline'
    in globals()
    and len(
        headline
    )
):

    for _, r in (
        headline.iterrows()
    ):

        comparison_rows.append(
            {

                'dataset':
                    r[
                        'dataset'
                    ],

                'method':
                    f'V1 {GLOBAL_MODEL}',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


# direct baselines
baseline_fp = (
    BASELINE_WORK
    / 'results'
    / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
)


if baseline_fp.exists():

    bb = pd.read_csv(
        baseline_fp
    )


    if 'label' in bb.columns:

        bb = bb[
            bb.label.eq(
                'nuclear_binding'
            )
        ].copy()


    if (
        'operating_point'
        in bb.columns
    ):

        bb = bb[
            bb.operating_point.eq(
                'fixed_0.5'
            )
        ].copy()


    for _, r in bb.iterrows():

        if str(
            r.method
        ) not in {
            'bindNode24',
            'bindEmbed21DL'
        }:

            continue


        apcol = (

            'AP_AUPRC'

            if (
                'AP_AUPRC'
                in bb.columns
            )

            else 'AUPRC'
        )


        comparison_rows.append(
            {

                'dataset':
                    str(
                        r.dataset
                    ),

                'method':
                    str(
                        r.method
                    ),

                'AUPRC':
                    float(
                        r[
                            apcol
                        ]
                    ),

                'AUROC':
                    float(
                        r.AUROC
                    ),

                'F1':
                    float(
                        r.global_positive_F1
                    ),

                'MCC':
                    float(
                        r.global_MCC
                    ),
            }
        )


v2_compare = pd.DataFrame(
    comparison_rows
)


display(
    v2_compare
    .sort_values(
        [
            'dataset',
            'AUPRC'
        ],
        ascending=[
            True,
            False
        ]
    )
)


v2_compare.to_csv(
    V2_RUN_ROOT
    / 'FINAL_V2_COMPARISON.csv',
    index=False
)


print(
    '\nV2 RESULT DIRECTORY:'
)

print(
    V2_RUN_ROOT
)

,dataset,variant,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,F1_fixed0.5,MCC_fixed0.5,precision_fixed0.5,recall_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,F1_OOFthr,MCC_OOFthr,precision_OOFthr,recall_OOFthr
0,TestSet300,Patch_BCE,0.483032,0.59899,0.418014,0.936092,0.398628,0.402372,0.538818,0.316327,0.418014,0.936092,0.355869,0.375451,0.574242,0.257823
1,TestSetNew46,Patch_BCE,0.483032,0.59899,0.110123,0.926090,0.166667,0.180483,0.108108,0.363636,0.110123,0.926090,0.146667,0.150482,0.098655,0.285714


,dataset,method,AUPRC,AUROC,F1,MCC
2,TestSet300,V1 NA_PatchHG_Attention,0.432287,0.941411,0.414727,0.417265
0,TestSet300,V2 Patch_BCE,0.418014,0.936092,0.398628,0.402372
6,TestSet300,bindNode24,0.400676,0.943253,0.383162,0.386394
4,TestSet300,bindEmbed21DL,0.371532,0.921621,0.378126,0.379045
7,TestSetNew46,bindNode24,0.124657,0.948962,0.221198,0.272449
3,TestSetNew46,V1 NA_PatchHG_Attention,0.122551,0.936822,0.200000,0.231350
1,TestSetNew46,V2 Patch_BCE,0.110123,0.926090,0.166667,0.180483
5,TestSetNew46,bindEmbed21DL,0.094095,0.927154,0.198582,0.239288



V2 RESULT DIRECTORY:
/content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_V2_SELECTIVE_MULTISCALE


In [ ]:
# ============================================================
# NUCLEAR FINAL EXPERIMENT
# V1 Attention architecture + V2 selective multi-scale patches
# One-cell, Dev-lock first, then external evaluation
# ============================================================
from pathlib import Path
import json, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    matthews_corrcoef, precision_score, recall_score
)

assert 'V2_PATCH_RAM' in globals(), \
    "Run Nuclear V2-C first to create V2_PATCH_RAM."

assert 'NuclearPatchHypergraph' in globals()
assert 'PatchProteinDataset' in globals()
assert 'run_patch_cv' in globals()

NUC_MS_ROOT = (
    BASE
    / 'results'
    / 'NUCLEAR_V1_ATTENTION_V2_MULTISCALE_PATCH'
)

NUC_MS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Same V1 hypergraph architecture.
# ONLY patch static dimension changes 18 -> 24
# because V2 multiscale patches have 24 features.
# ============================================================

class NuclearPatchHypergraphMS(
    NuclearPatchHypergraph
):
    def __init__(
        self,
        dims,
        mode='attention',
        latent=192,
        drop=0.2
    ):
        super().__init__(
            dims,
            mode=mode,
            latent=latent,
            moe=False,
            drop=drop
        )

        self.patch_static = nn.Sequential(
            nn.LayerNorm(24),
            nn.Linear(24, 64),
            nn.GELU()
        )


def collate_patch_ms(batch):

    keys = sorted(
        batch[0]['mods']
    )

    mods = {
        k: torch.from_numpy(
            np.concatenate(
                [
                    b['mods'][k]
                    for b in batch
                ],
                axis=0
            )
        ).float()

        for k in keys
    }

    y = torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()

    patch_feat = []
    edge_res = []
    edge_patch = []

    roff = 0
    poff = 0

    for b in batch:

        L = len(
            b['y']
        )

        P = len(
            b['patch_feat']
        )

        if P:
            patch_feat.append(
                torch.from_numpy(
                    b['patch_feat']
                ).float()
            )

        if len(
            b['edge_res']
        ):

            edge_res.append(
                torch.from_numpy(
                    b['edge_res']
                ).long()
                + roff
            )

            edge_patch.append(
                torch.from_numpy(
                    b['edge_patch']
                ).long()
                + poff
            )

        roff += L
        poff += P

    patch_feat = (
        torch.cat(
            patch_feat,
            dim=0
        )
        if patch_feat
        else torch.zeros(
            (0, 24),
            dtype=torch.float32
        )
    )

    edge_res = (
        torch.cat(
            edge_res,
            dim=0
        )
        if edge_res
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    edge_patch = (
        torch.cat(
            edge_patch,
            dim=0
        )
        if edge_patch
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    return {
        'mods': mods,
        'y': y,
        'edge_res': edge_res,
        'edge_patch': edge_patch,
        'patch_feat': patch_feat,
    }


# ============================================================
# Save current V1 globals.
# Temporarily substitute ONLY:
#   patch bank
#   patch-static dimensionality
# ============================================================

_OLD_PATCH_RAM = PATCH_RAM
_OLD_PATCH_FACTORY = PATCH_FACTORY
_OLD_PATCH_STATE_ROOT = PATCH_STATE_ROOT
_OLD_COLLATE_PATCH = collate_patch


PATCH_RAM = V2_PATCH_RAM

PATCH_FACTORY = {
    'NA_PatchHG_Attention_MS':
        lambda dims:
            NuclearPatchHypergraphMS(
                dims,
                mode='attention',
                latent=PATCH_LATENT,
                drop=0.2
            )
}

PATCH_STATE_ROOT = (
    NUC_MS_ROOT
    / 'states'
)

PATCH_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

collate_patch = collate_patch_ms


try:

    print("=" * 100)
    print(
        "NUCLEAR: V1 ATTENTION "
        "+ V2 SELECTIVE MULTI-SCALE PATCHES"
    )
    print("=" * 100)

    # ========================================================
    # DEV CV
    # ========================================================

    (
        ms_summary,
        ms_folds,
        ms_oof,
        ms_states
    ) = run_patch_cv(
        'NA_PatchHG_Attention_MS'
    )

    ms_summary_df = pd.DataFrame(
        [ms_summary]
    )

    display(
        ms_summary_df
    )

    display(
        ms_folds
    )

    ms_summary_df.to_csv(
        NUC_MS_ROOT
        / 'dev_cv_summary.csv',
        index=False
    )

    ms_folds.to_csv(
        NUC_MS_ROOT
        / 'dev_cv_folds.csv',
        index=False
    )


    MS_DEV = float(
        ms_summary[
            'CV_AUPRC_mean'
        ]
    )


    # V1 original Attention performance
    V1_DEV = float(
        patch_summary.loc[
            patch_summary
            .architecture
            .eq(
                'NA_PatchHG_Attention'
            ),
            'CV_AUPRC_mean'
        ].iloc[0]
    )


    print(
        f"\nV1 Attention Dev mean AUPRC : "
        f"{V1_DEV:.6f}"
    )

    print(
        f"MS Attention Dev mean AUPRC : "
        f"{MS_DEV:.6f}"
    )

    print(
        f"Delta MS - V1               : "
        f"{MS_DEV - V1_DEV:+.6f}"
    )


    # Complexity-aware stop rule
    MS_WINS = (
        MS_DEV
        - V1_DEV
        >= 0.002
    )

    print(
        "MS becomes new headline?     :",
        MS_WINS
    )


    # ========================================================
    # Frozen DEV OOF threshold
    # ========================================================

    dev_y = np.concatenate(
        train_df
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    (
        ms_thr,
        ms_oof_mcc
    ) = best_mcc_threshold(
        dev_y,
        ms_oof
    )


    print(
        "Frozen Dev threshold:",
        ms_thr
    )

    print(
        "Dev OOF MCC:",
        ms_oof_mcc
    )


    # ========================================================
    # External prediction
    # ========================================================

    def _predict_ms_external(
        split
    ):

        ds = PatchProteinDataset(
            split,
            SPLIT_DF[
                split
            ]
        )

        preds = []

        for fold, state in (
            ms_states.items()
        ):

            model = PATCH_FACTORY[
                'NA_PatchHG_Attention_MS'
            ](
                DIMS
            ).to(
                DEVICE
            )

            model.load_state_dict(
                state
            )

            p, _ = predict_patch(
                model,
                ds
            )

            preds.append(
                p
            )

            del model

            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

        return np.mean(
            np.stack(
                preds
            ),
            axis=0
        )


    ext_rows = []


    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:

        y = np.concatenate(
            SPLIT_DF[
                split
            ]
            .nuclear_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.int8
                    )
            )
            .tolist()
        )

        p = _predict_ms_external(
            split
        )


        m05 = binary_metrics(
            y,
            p,
            0.5
        )

        mdev = binary_metrics(
            y,
            p,
            ms_thr
        )


        row = {
            'dataset':
                split,

            'method':
                'NA_PatchHG_Attention_MS',

            'Dev_CV_AUPRC':
                MS_DEV,

            'Dev_OOF_MCC_threshold':
                ms_thr,
        }


        row.update(
            {
                f'{k}_fixed0.5':
                    v

                for k, v
                in m05.items()
            }
        )


        row.update(
            {
                f'{k}_OOFthr':
                    v

                for k, v
                in mdev.items()
            }
        )


        ext_rows.append(
            row
        )


    ms_external = pd.DataFrame(
        ext_rows
    )


    display(
        ms_external
    )


    ms_external.to_csv(
        NUC_MS_ROOT
        / 'external_metrics.csv',
        index=False
    )


    # ========================================================
    # FINAL comparison:
    # new MS vs V1 vs bindNode24 vs bindEmbed21DL
    # ========================================================

    cmp_rows = []


    for _, r in (
        ms_external.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r['dataset'],

                'method':
                    'MS selective PatchHG Attention',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    # Original V1
    for _, r in (
        headline.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r['dataset'],

                'method':
                    f'V1 {GLOBAL_MODEL}',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    baseline_fp = (
        BASELINE_WORK
        / 'results'
        / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
    )


    if baseline_fp.exists():

        bb = pd.read_csv(
            baseline_fp
        )


        if 'label' in bb.columns:

            bb = bb[
                bb.label.eq(
                    'nuclear_binding'
                )
            ].copy()


        if (
            'operating_point'
            in bb.columns
        ):

            bb = bb[
                bb.operating_point.eq(
                    'fixed_0.5'
                )
            ].copy()


        for _, r in (
            bb.iterrows()
        ):

            if str(
                r.method
            ) not in {
                'bindNode24',
                'bindEmbed21DL'
            }:
                continue


            apcol = (
                'AP_AUPRC'
                if 'AP_AUPRC'
                in bb.columns
                else 'AUPRC'
            )


            cmp_rows.append(
                {
                    'dataset':
                        str(
                            r.dataset
                        ),

                    'method':
                        str(
                            r.method
                        ),

                    'AUPRC':
                        float(
                            r[
                                apcol
                            ]
                        ),

                    'AUROC':
                        float(
                            r.AUROC
                        ),

                    'F1':
                        float(
                            r.global_positive_F1
                        ),

                    'MCC':
                        float(
                            r.global_MCC
                        ),
                }
            )


    ms_compare = (
        pd.DataFrame(
            cmp_rows
        )
        .sort_values(
            [
                'dataset',
                'AUPRC'
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print(
        "\nFINAL NUCLEAR COMPARISON"
    )

    display(
        ms_compare
    )


    ms_compare.to_csv(
        NUC_MS_ROOT
        / 'FINAL_COMPARISON.csv',
        index=False
    )


finally:

    # Restore V1 globals.
    PATCH_RAM = (
        _OLD_PATCH_RAM
    )

    PATCH_FACTORY = (
        _OLD_PATCH_FACTORY
    )

    PATCH_STATE_ROOT = (
        _OLD_PATCH_STATE_ROOT
    )

    collate_patch = (
        _OLD_COLLATE_PATCH
    )


print(
    "\nSaved:",
    NUC_MS_ROOT
)

NUCLEAR: V1 ATTENTION + V2 SELECTIVE MULTI-SCALE PATCHES
NA_PatchHG_Attention_MS fold 1 TRAIN
NA_PatchHG_Attention_MS fold 1 epoch 1 loss 0.2644 val_AP 0.55324 best 0.55324 alpha 0.0498
NA_PatchHG_Attention_MS fold 1 epoch 5 loss 0.081 val_AP 0.60518 best 0.60518 alpha 0.05
NA_PatchHG_Attention_MS fold 1 epoch 10 loss 0.0272 val_AP 0.55564 best 0.60518 alpha 0.05
NA_PatchHG_Attention_MS fold 2 TRAIN
NA_PatchHG_Attention_MS fold 2 epoch 1 loss 0.2712 val_AP 0.53549 best 0.53549 alpha 0.05
NA_PatchHG_Attention_MS fold 2 epoch 5 loss 0.0798 val_AP 0.50684 best 0.54264 alpha 0.0503
NA_PatchHG_Attention_MS fold 3 TRAIN
NA_PatchHG_Attention_MS fold 3 epoch 1 loss 0.2591 val_AP 0.38882 best 0.38882 alpha 0.0501
NA_PatchHG_Attention_MS fold 3 epoch 5 loss 0.0785 val_AP 0.3747 best 0.39463 alpha 0.0503
NA_PatchHG_Attention_MS fold 4 TRAIN
NA_PatchHG_Attention_MS fold 4 epoch 1 loss 0.2785 val_AP 0.40405 best 0.40405 alpha 0.0499
NA_PatchHG_Attention_MS fold 4 epoch 5 loss 0.0803 val_AP 0.48109 

,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,NA_PatchHG_Attention_MS,0.484679,0.091594,0.4671


,architecture,fold,AUPRC,n_val,n_positive
0,NA_PatchHG_Attention_MS,1,0.605184,32977,630
1,NA_PatchHG_Attention_MS,2,0.542638,34703,554
2,NA_PatchHG_Attention_MS,3,0.394629,34034,512
3,NA_PatchHG_Attention_MS,4,0.483249,33622,556
4,NA_PatchHG_Attention_MS,5,0.397694,34845,437



V1 Attention Dev mean AUPRC : 0.501934
MS Attention Dev mean AUPRC : 0.484679
Delta MS - V1               : -0.017255
MS becomes new headline?     : False
Frozen Dev threshold: 0.6682828282828283
Dev OOF MCC: 0.47749055714596816


,dataset,method,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,TestSet300,NA_PatchHG_Attention_MS,0.484679,0.668283,0.418942,0.935185,0.491597,0.397959,0.439850,0.430340,0.418942,0.935185,0.564168,0.302041,0.393443,0.402899
1,TestSetNew46,NA_PatchHG_Attention_MS,0.484679,0.668283,0.128783,0.936511,0.135294,0.597403,0.220624,0.267326,0.128783,0.936511,0.134387,0.441558,0.206061,0.227228



FINAL NUCLEAR COMPARISON


,dataset,method,AUPRC,AUROC,F1,MCC
2,TestSet300,V1 NA_PatchHG_Attention,0.432287,0.941411,0.414727,0.417265
0,TestSet300,MS selective PatchHG Attention,0.418942,0.935185,0.439850,0.430340
6,TestSet300,bindNode24,0.400676,0.943253,0.383162,0.386394
4,TestSet300,bindEmbed21DL,0.371532,0.921621,0.378126,0.379045
1,TestSetNew46,MS selective PatchHG Attention,0.128783,0.936511,0.220624,0.267326
7,TestSetNew46,bindNode24,0.124657,0.948962,0.221198,0.272449
3,TestSetNew46,V1 NA_PatchHG_Attention,0.122551,0.936822,0.200000,0.231350
5,TestSetNew46,bindEmbed21DL,0.094095,0.927154,0.198582,0.239288



Saved: /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_V1_ATTENTION_V2_MULTISCALE_PATCH


In [ ]:
# ============================================================
# NUCLEAR FINAL DEV-LOCKED ENSEMBLE
# V1 broad PatchHG + selective multi-scale PatchHG
# ============================================================

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

ENS_ROOT = (
    BASE
    / 'results'
    / 'NUCLEAR_FINAL_V1_MS_ENSEMBLE'
)

ENS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# 1. Get aligned Dev OOF predictions
# ============================================================

V1_OOF = np.asarray(
    PATCH_OOF[
        'NA_PatchHG_Attention'
    ],
    dtype=np.float32
)

MS_OOF = np.asarray(
    ms_oof,
    dtype=np.float32
)

assert (
    V1_OOF.shape
    ==
    MS_OOF.shape
)

assert np.all(
    np.isfinite(
        V1_OOF
    )
)

assert np.all(
    np.isfinite(
        MS_OOF
    )
)


DEV_Y = np.concatenate(
    train_df
    .nuclear_binding
    .apply(
        lambda x:
            np.asarray(
                x,
                dtype=np.int8
            )
    )
    .tolist()
)

assert len(
    DEV_Y
) == len(
    V1_OOF
)


# ============================================================
# 2. Build residue indices for each official protein fold
# ============================================================

offsets = {}

cursor = 0

for r in train_df.itertuples():

    L = len(
        r.sequence
    )

    offsets[
        str(
            r.prot_id
        )
    ] = (
        cursor,
        cursor + L
    )

    cursor += L


FOLD_RES_IDX = {}

for fold, ids in OFFICIAL_FOLDS.items():

    idx = []

    for pid in ids:

        a, b = offsets[
            str(
                pid
            )
        ]

        idx.extend(
            range(
                a,
                b
            )
        )

    FOLD_RES_IDX[
        fold
    ] = np.asarray(
        idx,
        dtype=np.int64
    )


# ============================================================
# 3. DEV-ONLY weight search
#
# w = fraction V1
# 1-w = fraction MS
#
# Selection criterion:
# mean official 5-fold AUPRC
# ============================================================

rows = []

for w in np.linspace(
    0.0,
    1.0,
    21
):

    pred = (
        w
        * V1_OOF
        +
        (
            1.0 - w
        )
        * MS_OOF
    )

    fold_ap = []

    for fold in sorted(
        FOLD_RES_IDX
    ):

        idx = FOLD_RES_IDX[
            fold
        ]

        fold_ap.append(
            float(
                average_precision_score(
                    DEV_Y[
                        idx
                    ],
                    pred[
                        idx
                    ]
                )
            )
        )

    rows.append(
        {
            'w_V1':
                float(
                    w
                ),

            'w_MS':
                float(
                    1.0 - w
                ),

            'CV_AUPRC_mean':
                float(
                    np.mean(
                        fold_ap
                    )
                ),

            'CV_AUPRC_std':
                float(
                    np.std(
                        fold_ap,
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                float(
                    average_precision_score(
                        DEV_Y,
                        pred
                    )
                ),

            **{
                f'fold{f}_AUPRC':
                    fold_ap[
                        i
                    ]

                for i, f
                in enumerate(
                    sorted(
                        FOLD_RES_IDX
                    )
                )
            }
        }
    )


weight_search = pd.DataFrame(
    rows
)


display(
    weight_search.sort_values(
        'CV_AUPRC_mean',
        ascending=False
    ).head(
        10
    )
)


weight_search.to_csv(
    ENS_ROOT
    / 'dev_weight_search.csv',
    index=False
)


best = (
    weight_search
    .sort_values(
        'CV_AUPRC_mean',
        ascending=False
    )
    .iloc[0]
)


ENS_W_V1 = float(
    best[
        'w_V1'
    ]
)

ENS_W_MS = float(
    best[
        'w_MS'
    ]
)

ENS_DEV = float(
    best[
        'CV_AUPRC_mean'
    ]
)


V1_DEV = float(
    patch_summary.loc[
        patch_summary
        .architecture
        .eq(
            'NA_PatchHG_Attention'
        ),
        'CV_AUPRC_mean'
    ].iloc[0]
)


print(
    '\n'
    + '=' * 90
)

print(
    'DEV-LOCKED ENSEMBLE'
)

print(
    '=' * 90
)

print(
    'V1 weight:',
    ENS_W_V1
)

print(
    'MS weight:',
    ENS_W_MS
)

print(
    'V1 Dev mean AUPRC:',
    V1_DEV
)

print(
    'Ensemble Dev mean AUPRC:',
    ENS_DEV
)

print(
    'Delta:',
    ENS_DEV - V1_DEV
)


ENSEMBLE_PASS = (
    ENS_DEV
    -
    V1_DEV
    >= 0.002
)


print(
    'Promote ensemble?',
    ENSEMBLE_PASS
)


# ============================================================
# 4. Frozen Dev threshold for ensemble
# ============================================================

ENS_OOF = (
    ENS_W_V1
    * V1_OOF
    +
    ENS_W_MS
    * MS_OOF
)


ENS_THR, ENS_MCC = (
    best_mcc_threshold(
        DEV_Y,
        ENS_OOF
    )
)


print(
    'Frozen Dev threshold:',
    ENS_THR
)

print(
    'Dev OOF MCC:',
    ENS_MCC
)


# ============================================================
# 5. External helpers
# ============================================================

def _metric(
    y,
    p,
    t
):

    pred = (
        p >= t
    ).astype(
        int
    )

    return {
        'AUPRC':
            float(
                average_precision_score(
                    y,
                    p
                )
            ),

        'AUROC':
            float(
                roc_auc_score(
                    y,
                    p
                )
            ),

        'precision':
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'recall':
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'F1':
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'MCC':
            float(
                matthews_corrcoef(
                    y,
                    pred
                )
            ),
    }


# ============================================================
# 6. Recreate MS prediction safely
# ============================================================

def predict_ms_again(
    split
):

    global PATCH_RAM
    global PATCH_FACTORY
    global PATCH_STATE_ROOT
    global collate_patch

    old_ram = (
        PATCH_RAM
    )

    old_factory = (
        PATCH_FACTORY
    )

    old_state_root = (
        PATCH_STATE_ROOT
    )

    old_collate = (
        collate_patch
    )


    PATCH_RAM = (
        V2_PATCH_RAM
    )

    PATCH_FACTORY = {
        'NA_PatchHG_Attention_MS':
            lambda dims:
                NuclearPatchHypergraphMS(
                    dims,
                    mode='attention',
                    latent=PATCH_LATENT,
                    drop=0.2
                )
    }

    collate_patch = (
        collate_patch_ms
    )


    try:

        ds = PatchProteinDataset(
            split,
            SPLIT_DF[
                split
            ]
        )

        preds = []

        for fold, state in (
            ms_states.items()
        ):

            model = (
                PATCH_FACTORY[
                    'NA_PatchHG_Attention_MS'
                ](
                    DIMS
                )
                .to(
                    DEVICE
                )
            )

            model.load_state_dict(
                state
            )

            p, _ = predict_patch(
                model,
                ds
            )

            preds.append(
                p
            )

            del model

            if DEVICE.type == 'cuda':

                torch.cuda.empty_cache()


        return np.mean(
            np.stack(
                preds
            ),
            axis=0
        )


    finally:

        PATCH_RAM = (
            old_ram
        )

        PATCH_FACTORY = (
            old_factory
        )

        PATCH_STATE_ROOT = (
            old_state_root
        )

        collate_patch = (
            old_collate
        )


# ============================================================
# 7. External evaluation
# ============================================================

external_rows = []


for split in [
    'TestSet300',
    'TestSetNew46'
]:

    y = np.concatenate(
        SPLIT_DF[
            split
        ]
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    # Original V1 fold ensemble
    p_v1 = predict_patch_external(
        split
    )


    # Multi-scale fold ensemble
    p_ms = predict_ms_again(
        split
    )


    p_ens = (
        ENS_W_V1
        * p_v1
        +
        ENS_W_MS
        * p_ms
    )


    for name, p, thr in [

        (
            'V1_Attention',
            p_v1,
            PATCH_THR
        ),

        (
            'MS_Attention',
            p_ms,
            ms_thr
        ),

        (
            'DEV_LOCKED_ENSEMBLE',
            p_ens,
            ENS_THR
        ),
    ]:

        m05 = _metric(
            y,
            p,
            0.5
        )

        mdev = _metric(
            y,
            p,
            thr
        )


        row = {
            'dataset':
                split,

            'method':
                name,

            'AUPRC':
                m05[
                    'AUPRC'
                ],

            'AUROC':
                m05[
                    'AUROC'
                ],

            'precision_fixed0.5':
                m05[
                    'precision'
                ],

            'recall_fixed0.5':
                m05[
                    'recall'
                ],

            'F1_fixed0.5':
                m05[
                    'F1'
                ],

            'MCC_fixed0.5':
                m05[
                    'MCC'
                ],

            'frozen_threshold':
                thr,

            'F1_OOFthr':
                mdev[
                    'F1'
                ],

            'MCC_OOFthr':
                mdev[
                    'MCC'
                ],
        }


        external_rows.append(
            row
        )


ext = pd.DataFrame(
    external_rows
)


# ============================================================
# 8. Add bind21 / bind24
# ============================================================

baseline_fp = (
    BASELINE_WORK
    / 'results'
    / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
)


if baseline_fp.exists():

    bb = pd.read_csv(
        baseline_fp
    )


    bb = bb[
        bb.label.eq(
            'nuclear_binding'
        )
    ].copy()


    if (
        'operating_point'
        in bb.columns
    ):

        bb = bb[
            bb.operating_point.eq(
                'fixed_0.5'
            )
        ]


    base_rows = []


    for _, r in (
        bb.iterrows()
    ):

        if str(
            r.method
        ) not in {
            'bindNode24',
            'bindEmbed21DL'
        }:

            continue


        apcol = (
            'AP_AUPRC'
            if 'AP_AUPRC'
            in bb.columns
            else 'AUPRC'
        )


        base_rows.append(
            {
                'dataset':
                    str(
                        r.dataset
                    ),

                'method':
                    str(
                        r.method
                    ),

                'AUPRC':
                    float(
                        r[
                            apcol
                        ]
                    ),

                'AUROC':
                    float(
                        r.AUROC
                    ),

                'precision_fixed0.5':
                    np.nan,

                'recall_fixed0.5':
                    np.nan,

                'F1_fixed0.5':
                    float(
                        r.global_positive_F1
                    ),

                'MCC_fixed0.5':
                    float(
                        r.global_MCC
                    ),

                'frozen_threshold':
                    0.5,

                'F1_OOFthr':
                    np.nan,

                'MCC_OOFthr':
                    np.nan,
            }
        )


    ext = pd.concat(
        [
            ext,
            pd.DataFrame(
                base_rows
            )
        ],
        ignore_index=True
    )


ext = ext.sort_values(
    [
        'dataset',
        'AUPRC'
    ],
    ascending=[
        True,
        False
    ]
)


print(
    '\nFINAL NUCLEAR ENSEMBLE COMPARISON'
)


display(
    ext
)


ext.to_csv(
    ENS_ROOT
    / 'FINAL_COMPARISON.csv',
    index=False
)


summary = {
    'w_V1':
        ENS_W_V1,

    'w_MS':
        ENS_W_MS,

    'V1_Dev_AUPRC':
        V1_DEV,

    'Ensemble_Dev_AUPRC':
        ENS_DEV,

    'Dev_gain':
        ENS_DEV
        -
        V1_DEV,

    'Promote':
        bool(
            ENSEMBLE_PASS
        ),
}


(
    ENS_ROOT
    / 'FINAL_DECISION.json'
).write_text(
    json.dumps(
        summary,
        indent=2
    )
)


print(
    '\nDecision:',
    summary
)

print(
    '\nSaved:',
    ENS_ROOT
)

,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
14,0.70,0.30,0.508961,0.088127,0.499399,0.621704,0.569264,0.416005,0.506799,0.431034
15,0.75,0.25,0.508836,0.088383,0.498268,0.622199,0.569397,0.415494,0.505746,0.431343
12,0.60,0.40,0.508830,0.087920,0.500777,0.620899,0.568362,0.415829,0.508760,0.430301
16,0.80,0.20,0.508746,0.088352,0.496696,0.622117,0.569541,0.414385,0.504810,0.432879
13,0.65,0.35,0.508732,0.088106,0.500164,0.621396,0.568819,0.415991,0.507109,0.430348
11,0.55,0.45,0.508344,0.088076,0.500859,0.620327,0.567899,0.414923,0.509061,0.429511
17,0.85,0.15,0.508255,0.088613,0.494368,0.622004,0.569522,0.412994,0.503455,0.433299
10,0.50,0.50,0.507818,0.088120,0.500562,0.619743,0.566955,0.414553,0.509612,0.428228
18,0.90,0.10,0.507795,0.088558,0.490964,0.621642,0.569000,0.411015,0.502073,0.435245
9,0.45,0.55,0.507183,0.088312,0.499897,0.619011,0.566335,0.413627,0.509952,0.426988



DEV-LOCKED ENSEMBLE
V1 weight: 0.7000000000000001
MS weight: 0.29999999999999993
V1 Dev mean AUPRC: 0.5019338304768628
Ensemble Dev mean AUPRC: 0.5089611583669623
Delta: 0.007027327890099522
Promote ensemble? True
Frozen Dev threshold: 0.7177777777777777
Dev OOF MCC: 0.5218963131704384

FINAL NUCLEAR ENSEMBLE COMPARISON


,dataset,method,AUPRC,AUROC,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,frozen_threshold,F1_OOFthr,MCC_OOFthr
2,TestSet300,DEV_LOCKED_ENSEMBLE,0.432840,0.941657,0.534572,0.352381,0.424764,0.423217,0.717778,0.313805,0.353770
0,TestSet300,V1_Attention,0.432288,0.941411,0.548712,0.333333,0.414727,0.417265,0.767273,0.264943,0.318856
1,TestSet300,MS_Attention,0.418940,0.935185,0.491597,0.397959,0.439850,0.430340,0.668283,0.393443,0.402899
8,TestSet300,bindNode24,0.400676,0.943253,NaN,NaN,0.383162,0.386394,0.500000,NaN,NaN
6,TestSet300,bindEmbed21DL,0.371532,0.921621,NaN,NaN,0.378126,0.379045,0.500000,NaN,NaN
4,TestSetNew46,MS_Attention,0.128783,0.936511,0.135294,0.597403,0.220624,0.267326,0.668283,0.206061,0.227228
9,TestSetNew46,bindNode24,0.124657,0.948962,NaN,NaN,0.221198,0.272449,0.500000,NaN,NaN
5,TestSetNew46,DEV_LOCKED_ENSEMBLE,0.124501,0.936898,0.127796,0.519481,0.205128,0.240296,0.717778,0.168582,0.169267
3,TestSetNew46,V1_Attention,0.122551,0.936822,0.125413,0.493506,0.200000,0.231350,0.767273,0.108108,0.098353
7,TestSetNew46,bindEmbed21DL,0.094095,0.927154,NaN,NaN,0.198582,0.239288,0.500000,NaN,NaN



Decision: {'w_V1': 0.7000000000000001, 'w_MS': 0.29999999999999993, 'V1_Dev_AUPRC': 0.5019338304768628, 'Ensemble_Dev_AUPRC': 0.5089611583669623, 'Dev_gain': 0.007027327890099522, 'Promote': True}

Saved: /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_V1_MS_ENSEMBLE


In [ ]:
# ============================================================
# NUCLEAR FINAL ONE-CELL
# Dev-lock V1 + Multi-scale ensemble, then report ALL metrics
# including protein-mean F1/MCC and bindEmbed21DL/bindNode24.
#
# Requires completed Nuclear V1 + multi-scale cells:
# PATCH_OOF, PATCH_STATES, V2_PATCH_RAM, ms_oof, ms_states,
# NuclearPatchHypergraphMS, collate_patch_ms, etc.
# ============================================================
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    matthews_corrcoef, precision_score, recall_score
)

REQ = [
    "PATCH_OOF","PATCH_STATES","V2_PATCH_RAM","ms_oof","ms_states",
    "NuclearPatchHypergraphMS","collate_patch_ms","PatchProteinDataset",
    "predict_patch","PATCH_FACTORY","PATCH_RAM","collate_patch",
    "train_df","SPLIT_DF","OFFICIAL_FOLDS","BASELINE_WORK"
]
missing = [x for x in REQ if x not in globals()]
if missing:
    raise RuntimeError(
        "Run the completed Nuclear V1 + MS cells first. Missing: "
        + ", ".join(missing)
    )

OUT = BASE/'results'/'NUCLEAR_FINAL_ONECELL'
OUT.mkdir(parents=True, exist_ok=True)

V1_NAME = 'NA_PatchHG_Attention'
v1_oof = np.asarray(PATCH_OOF[V1_NAME], dtype=np.float32)
ms_oof_ = np.asarray(ms_oof, dtype=np.float32)

y_dev = np.concatenate(
    train_df.nuclear_binding.apply(
        lambda x: np.asarray(x, dtype=np.int8)
    ).tolist()
)

assert v1_oof.shape == ms_oof_.shape == y_dev.shape

# ------------------------------------------------------------
# 1. Official-fold residue indices
# ------------------------------------------------------------
offsets, cur = {}, 0
for r in train_df.itertuples():
    L = len(r.sequence)
    offsets[str(r.prot_id)] = (cur, cur+L)
    cur += L

fold_idx = {}
for fold, ids in OFFICIAL_FOLDS.items():
    z = []
    for pid in ids:
        a,b = offsets[str(pid)]
        z.extend(range(a,b))
    fold_idx[fold] = np.asarray(z, dtype=np.int64)

# ------------------------------------------------------------
# 2. Dev-only ensemble weight search
# ------------------------------------------------------------
rows = []
for w in np.linspace(0,1,21):
    p = w*v1_oof + (1-w)*ms_oof_
    aps = [
        average_precision_score(y_dev[idx], p[idx])
        for _,idx in sorted(fold_idx.items())
    ]
    rows.append({
        'w_V1':float(w),
        'w_MS':float(1-w),
        'CV_AUPRC_mean':float(np.mean(aps)),
        'CV_AUPRC_std':float(np.std(aps,ddof=1)),
        'OOF_AUPRC':float(average_precision_score(y_dev,p)),
        **{f'fold{i+1}_AUPRC':float(v) for i,v in enumerate(aps)}
    })

weight_search = pd.DataFrame(rows).sort_values(
    'CV_AUPRC_mean', ascending=False
).reset_index(drop=True)

display(weight_search.head(10))
weight_search.to_csv(OUT/'DEV_WEIGHT_SEARCH.csv',index=False)

best = weight_search.iloc[0]
w1, wm = float(best.w_V1), float(best.w_MS)
ens_dev = float(best.CV_AUPRC_mean)

v1_dev = float(
    patch_summary.loc[
        patch_summary.architecture.eq(V1_NAME),
        'CV_AUPRC_mean'
    ].iloc[0]
)

PROMOTE = (ens_dev - v1_dev) >= 0.002
if not PROMOTE:
    w1,wm = 1.0,0.0
    ens_dev = v1_dev

ens_oof = w1*v1_oof + wm*ms_oof_

def best_mcc_thr(y,p):
    best=(-999,0.5)
    for t in np.linspace(.01,.99,199):
        m=matthews_corrcoef(y,(p>=t).astype(int))
        if m>best[0]:
            best=(m,float(t))
    return best[1],best[0]

ens_thr, ens_oof_mcc = best_mcc_thr(y_dev,ens_oof)

print("\nDEV FINAL")
print("w_V1 =",w1,"w_MS =",wm)
print("V1 Dev =",v1_dev)
print("Final Dev =",ens_dev)
print("Delta =",ens_dev-v1_dev)
print("Promote ensemble =",PROMOTE)
print("Frozen Dev threshold =",ens_thr)
print("OOF MCC =",ens_oof_mcc)

# ------------------------------------------------------------
# 3. External predictors
# ------------------------------------------------------------
def predict_v1(split):
    ds = PatchProteinDataset(split,SPLIT_DF[split])
    pred=[]
    for fold,state in PATCH_STATES[V1_NAME].items():
        model = PATCH_FACTORY[V1_NAME](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p,_ = predict_patch(model,ds)
        pred.append(p)
        del model
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return np.mean(np.stack(pred),axis=0)

def predict_ms(split):
    global PATCH_RAM,PATCH_FACTORY,collate_patch
    old_ram,old_fac,old_col = PATCH_RAM,PATCH_FACTORY,collate_patch
    try:
        PATCH_RAM = V2_PATCH_RAM
        PATCH_FACTORY = {
            'NA_PatchHG_Attention_MS':
                lambda dims:NuclearPatchHypergraphMS(
                    dims, mode='attention',
                    latent=PATCH_LATENT, drop=.2
                )
        }
        collate_patch = collate_patch_ms
        ds = PatchProteinDataset(split,SPLIT_DF[split])
        pred=[]
        for fold,state in ms_states.items():
            model = PATCH_FACTORY['NA_PatchHG_Attention_MS'](DIMS).to(DEVICE)
            model.load_state_dict(state)
            p,_ = predict_patch(model,ds)
            pred.append(p)
            del model
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
        return np.mean(np.stack(pred),axis=0)
    finally:
        PATCH_RAM,PATCH_FACTORY,collate_patch = old_ram,old_fac,old_col

# ------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------
def global_metrics(y,p,t=.5):
    pr=(p>=t).astype(int)
    return {
        'AUPRC':float(average_precision_score(y,p)),
        'AUROC':float(roc_auc_score(y,p)),
        'precision':float(precision_score(y,pr,zero_division=0)),
        'recall':float(recall_score(y,pr,zero_division=0)),
        'F1':float(f1_score(y,pr,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pr)),
    }

def protein_metrics(df,p,t=.5):
    f1s,mccs=[],[]
    pos=0
    for r in df.itertuples():
        L=len(r.sequence)
        y=np.asarray(r.nuclear_binding,dtype=int)
        pr=(p[pos:pos+L]>=t).astype(int)
        f1s.append(f1_score(y,pr,zero_division=0))
        mccs.append(matthews_corrcoef(y,pr))
        pos+=L
    return {
        'protein_mean_F1':float(np.mean(f1s)),
        'protein_mean_MCC':float(np.mean(mccs)),
    }

our_rows=[]
pred_cache={}

for split in ['TestSet300','TestSetNew46']:
    df=SPLIT_DF[split]
    y=np.concatenate(df.nuclear_binding.apply(
        lambda x:np.asarray(x,dtype=int)
    ).tolist())

    pv1=predict_v1(split)
    pms=predict_ms(split)
    pens=w1*pv1+wm*pms

    pred_cache[split]=pens

    for name,p,thr in [
        ('V1_Attention',pv1,.5),
        ('MS_Attention',pms,.5),
        ('FINAL_DEV_LOCKED',pens,.5),
    ]:
        g=global_metrics(y,p,thr)
        pm=protein_metrics(df,p,thr)
        our_rows.append({
            'dataset':split,'method':name,
            **g,**pm
        })

    # Save final per-residue predictions.
    pids=[]; ridx=[]
    for r in df.itertuples():
        pids.extend([str(r.prot_id)]*len(r.sequence))
        ridx.extend(range(len(r.sequence)))

    pd.DataFrame({
        'prot_id':pids,
        'residue_index':ridx,
        'y_true':y,
        'probability':pens
    }).to_csv(
        OUT/f'FINAL_predictions_{split}.csv.gz',
        index=False
    )

ours=pd.DataFrame(our_rows)

# ------------------------------------------------------------
# 5. Direct reproduced baselines
# ------------------------------------------------------------
bfp=BASELINE_WORK/'results'/'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
bb=pd.read_csv(bfp)
bb=bb[bb.label.eq('nuclear_binding')].copy()
if 'operating_point' in bb.columns:
    bb=bb[bb.operating_point.eq('fixed_0.5')].copy()

base_rows=[]
for _,r in bb.iterrows():
    if str(r.method) not in {'bindNode24','bindEmbed21DL'}:
        continue
    apcol='AP_AUPRC' if 'AP_AUPRC' in bb.columns else 'AUPRC'
    base_rows.append({
        'dataset':str(r.dataset),
        'method':str(r.method),
        'AUPRC':float(r[apcol]),
        'AUROC':float(r.AUROC),
        'precision':np.nan,
        'recall':np.nan,
        'F1':float(r.global_positive_F1),
        'MCC':float(r.global_MCC),
        'protein_mean_F1':float(r.protein_mean_F1),
        'protein_mean_MCC':float(r.protein_mean_MCC),
    })

final=pd.concat(
    [ours,pd.DataFrame(base_rows)],
    ignore_index=True
).sort_values(
    ['dataset','AUPRC'],
    ascending=[True,False]
)

print("\nFINAL NUCLEAR — ALL AVAILABLE METRICS")
display(final)

final.to_csv(OUT/'FINAL_ALL_METRICS.csv',index=False)

decision={
    'w_V1':w1,'w_MS':wm,
    'V1_Dev_mean_AUPRC':v1_dev,
    'Final_Dev_mean_AUPRC':ens_dev,
    'Dev_gain':ens_dev-v1_dev,
    'Promoted':bool(PROMOTE),
    'Frozen_Dev_MCC_threshold':ens_thr,
}
(OUT/'FINAL_DECISION.json').write_text(json.dumps(decision,indent=2))
print("\nDecision:",decision)
print("Saved:",OUT)


,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.70,0.30,0.508961,0.088127,0.499399,0.621704,0.569264,0.416005,0.506799,0.431034
1,0.75,0.25,0.508836,0.088383,0.498268,0.622199,0.569397,0.415494,0.505746,0.431343
2,0.60,0.40,0.508830,0.087920,0.500777,0.620899,0.568362,0.415829,0.508760,0.430301
3,0.80,0.20,0.508746,0.088352,0.496696,0.622117,0.569541,0.414385,0.504810,0.432879
4,0.65,0.35,0.508732,0.088106,0.500164,0.621396,0.568819,0.415991,0.507109,0.430348
5,0.55,0.45,0.508344,0.088076,0.500859,0.620327,0.567899,0.414923,0.509061,0.429511
6,0.85,0.15,0.508255,0.088613,0.494368,0.622004,0.569522,0.412994,0.503455,0.433299
7,0.50,0.50,0.507818,0.088120,0.500562,0.619743,0.566955,0.414553,0.509612,0.428228
8,0.90,0.10,0.507795,0.088558,0.490964,0.621642,0.569000,0.411015,0.502073,0.435245
9,0.45,0.55,0.507183,0.088312,0.499897,0.619011,0.566335,0.413627,0.509952,0.426988



DEV FINAL
w_V1 = 0.7000000000000001 w_MS = 0.29999999999999993
V1 Dev = 0.5019338304768628
Final Dev = 0.5089611583669623
Delta = 0.007027327890099522
Promote ensemble = True
Frozen Dev threshold = 0.7177777777777777
OOF MCC = 0.5218963131704384


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th


FINAL NUCLEAR — ALL AVAILABLE METRICS


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th

,dataset,method,AUPRC,AUROC,precision,recall,F1,MCC,protein_mean_F1,protein_mean_MCC
2,TestSet300,FINAL_DEV_LOCKED,0.432841,0.941657,0.534572,0.352381,0.424764,0.423217,0.083390,0.083331
0,TestSet300,V1_Attention,0.432286,0.941411,0.548712,0.333333,0.414727,0.417265,0.080760,0.081064
1,TestSet300,MS_Attention,0.418941,0.935185,0.491597,0.397959,0.439850,0.430340,0.090381,0.088579
8,TestSet300,bindNode24,0.400676,0.943253,NaN,NaN,0.383162,0.386394,0.261493,0.248807
6,TestSet300,bindEmbed21DL,0.371532,0.921621,NaN,NaN,0.378126,0.379045,0.238822,0.224839
4,TestSetNew46,MS_Attention,0.128783,0.936511,0.135294,0.597403,0.220624,0.267326,0.070031,0.065173
9,TestSetNew46,bindNode24,0.124657,0.948962,NaN,NaN,0.221198,0.272449,0.250441,0.254545
5,TestSetNew46,FINAL_DEV_LOCKED,0.124501,0.936898,0.127796,0.519481,0.205128,0.240296,0.063387,0.058582
3,TestSetNew46,V1_Attention,0.122551,0.936822,0.125413,0.493506,0.200000,0.231350,0.057315,0.053058
7,TestSetNew46,bindEmbed21DL,0.094095,0.927154,NaN,NaN,0.198582,0.239288,0.193721,0.192283



Decision: {'w_V1': 0.7000000000000001, 'w_MS': 0.29999999999999993, 'V1_Dev_mean_AUPRC': 0.5019338304768628, 'Final_Dev_mean_AUPRC': 0.5089611583669623, 'Dev_gain': 0.007027327890099522, 'Promoted': True, 'Frozen_Dev_MCC_threshold': 0.7177777777777777}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_ONECELL


In [ ]:
# ============================================================
# NUCLEAR FINAL PAPER-READY REPORT — ONE CELL
#
# Run AFTER completed V1 + multi-scale training cells.
#
# What this cell does:
# 1) Dev-only V1/MS ensemble weight lock (official 5 folds)
# 2) Cross-fitted ensemble audit (selection-bias sanity check)
# 3) Dev OOF threshold locks for EACH model:
#       - fixed 0.5
#       - MCC-optimal
#       - positive-class F1-optimal
#       - Youden-J-optimal
# 4) External Test300/Test46 metrics at ALL locked thresholds
# 5) Direct fixed-0.5 comparison to bindEmbed21DL / bindNode24
# 6) Correct metric names:
#       - global residue-level positive-class F1
#       - global residue-level MCC
#       - formal micro-F1 across both classes (= accuracy here)
#       - protein-mean F1/MCC using baseline convention:
#         include protein iff TP+FP+FN > 0; exclude TN-only
# 7) Saves paper-ready CSV/JSON and per-residue predictions.
# ============================================================

from pathlib import Path
import json, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    accuracy_score, balanced_accuracy_score
)

TASK_NAME = "NUCLEAR"
LABEL_COL = "nuclear_binding"
V1_NAME = "NA_PatchHG_Attention"
MS_NAME = "NA_PatchHG_Attention_MS"
FINAL_NAME = "FINAL_DEV_LOCKED_ENSEMBLE"
MIN_DEV_GAIN = 0.002
WEIGHT_GRID = np.linspace(0.0, 1.0, 21)

OUT = BASE / "results" / "NUCLEAR_FINAL_PAPER_READY"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# A. Required globals
# ------------------------------------------------------------
REQ = [
    "PATCH_OOF", "PATCH_STATES", "V2_PATCH_RAM",
    "ms_oof", "ms_states",
    "NuclearPatchHypergraphMS", "collate_patch_ms",
    "PatchProteinDataset", "predict_patch",
    "PATCH_FACTORY", "PATCH_RAM", "collate_patch",
    "train_df", "SPLIT_DF", "OFFICIAL_FOLDS",
    "BASELINE_WORK", "patch_summary",
    "DIMS", "DEVICE", "PATCH_LATENT"
]
missing = [x for x in REQ if x not in globals()]
if missing:
    raise RuntimeError(
        "Run the completed V1 + multi-scale cells first. Missing: "
        + ", ".join(missing)
    )

# ------------------------------------------------------------
# B. Metric helpers
# ------------------------------------------------------------
def _counts(y, pred):
    y = np.asarray(y, dtype=np.int8)
    pred = np.asarray(pred, dtype=np.int8)
    tp = int(np.sum((y == 1) & (pred == 1)))
    tn = int(np.sum((y == 0) & (pred == 0)))
    fp = int(np.sum((y == 0) & (pred == 1)))
    fn = int(np.sum((y == 1) & (pred == 0)))
    return tp, tn, fp, fn

def _mcc_counts(tp, tn, fp, fn):
    den = ((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) ** 0.5
    return 0.0 if den == 0 else float((tp*tn - fp*fn) / den)

def _safe_div(a, b):
    return 0.0 if b == 0 else float(a / b)

def full_global_metrics(y, p, threshold):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)
    pred = (p >= float(threshold)).astype(np.int8)
    tp, tn, fp, fn = _counts(y, pred)

    precision_pos = _safe_div(tp, tp+fp)
    recall_pos = _safe_div(tp, tp+fn)
    specificity = _safe_div(tn, tn+fp)
    npv = _safe_div(tn, tn+fn)
    f1_pos = _safe_div(2*tp, 2*tp+fp+fn)

    # Formal micro-F1 across both classes.
    # For binary single-label prediction this equals accuracy.
    micro_f1 = float(f1_score(y, pred, average="micro", zero_division=0))
    macro_f1 = float(f1_score(y, pred, average="macro", zero_division=0))

    return {
        "n_residues": int(len(y)),
        "n_positive": int(np.sum(y == 1)),
        "prevalence": float(np.mean(y)),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,

        # Threshold-free pooled/global residue-level ranking metrics
        "AUPRC_global_residue": float(average_precision_score(y, p)),
        "AUROC_global_residue": float(roc_auc_score(y, p)),

        # Threshold-dependent pooled/global residue-level metrics
        "Accuracy_global_residue": float(accuracy_score(y, pred)),
        "Balanced_accuracy_global_residue": float(
            balanced_accuracy_score(y, pred)
        ),
        "Precision_positive_global_residue": precision_pos,
        "Recall_positive_global_residue": recall_pos,
        "Sensitivity_global_residue": recall_pos,
        "Specificity_global_residue": specificity,
        "NPV_global_residue": npv,
        "F1_positive_global_residue": f1_pos,
        "MCC_global_residue": _mcc_counts(tp, tn, fp, fn),

        # Formal class-averaged F1 metrics
        "F1_micro_all_classes": micro_f1,
        "F1_macro_two_classes": macro_f1,
    }

def protein_mean_metrics_baseline_convention(df, p, threshold):
    """
    Same aggregation convention used by the reproduced direct baseline:
    calculate each protein separately and INCLUDE it iff TP+FP+FN > 0.
    Therefore completely TN-only proteins are excluded.
    """
    p = np.asarray(p, dtype=np.float64)
    pos = 0
    f1s, mccs = [], []
    n_total = 0
    n_included = 0
    n_excluded_tn_only = 0

    for r in df.itertuples():
        L = len(r.sequence)
        y = np.asarray(getattr(r, LABEL_COL), dtype=np.int8)
        pred = (p[pos:pos+L] >= float(threshold)).astype(np.int8)
        pos += L
        n_total += 1

        tp, tn, fp, fn = _counts(y, pred)

        if (tp + fp + fn) == 0:
            n_excluded_tn_only += 1
            continue

        n_included += 1
        f1s.append(_safe_div(2*tp, 2*tp+fp+fn))
        mccs.append(_mcc_counts(tp, tn, fp, fn))

    assert pos == len(p)

    return {
        "protein_mean_F1_baseline_convention":
            float(np.mean(f1s)) if f1s else np.nan,
        "protein_mean_MCC_baseline_convention":
            float(np.mean(mccs)) if mccs else np.nan,
        "protein_n_total": int(n_total),
        "protein_n_included_baseline_convention": int(n_included),
        "protein_n_excluded_TN_only": int(n_excluded_tn_only),
    }

def optimize_thresholds_exact(y, p):
    """
    Exact search at unique OOF probability cutoffs for pred = (p >= threshold).
    Returns thresholds optimized ONLY on Dev OOF predictions.
    """
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)

    order = np.argsort(-p, kind="mergesort")
    ys = y[order]
    ps = p[order]

    tp_c = np.cumsum(ys == 1)
    fp_c = np.cumsum(ys == 0)

    # End index of each equal-score group so >= threshold includes ties.
    ends = np.r_[np.where(ps[:-1] != ps[1:])[0], len(ps)-1]

    tp = tp_c[ends].astype(np.float64)
    fp = fp_c[ends].astype(np.float64)
    P = float(np.sum(y == 1))
    N = float(np.sum(y == 0))
    fn = P - tp
    tn = N - fp
    thr = ps[ends]

    den = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc = np.divide(
        tp*tn - fp*fn,
        den,
        out=np.zeros_like(tp),
        where=den > 0
    )

    f1 = np.divide(
        2*tp,
        2*tp + fp + fn,
        out=np.zeros_like(tp),
        where=(2*tp + fp + fn) > 0
    )

    tpr = np.divide(tp, P, out=np.zeros_like(tp), where=P > 0)
    tnr = np.divide(tn, N, out=np.zeros_like(tn), where=N > 0)
    youden = tpr + tnr - 1.0

    i_mcc = int(np.nanargmax(mcc))
    i_f1 = int(np.nanargmax(f1))
    i_y = int(np.nanargmax(youden))

    return {
        "fixed_0.5": 0.5,
        "dev_oof_mcc_opt": float(thr[i_mcc]),
        "dev_oof_positive_f1_opt": float(thr[i_f1]),
        "dev_oof_youden_j_opt": float(thr[i_y]),
    }

# ------------------------------------------------------------
# C. Dev OOF + official-fold indices
# ------------------------------------------------------------
V1_OOF = np.asarray(PATCH_OOF[V1_NAME], dtype=np.float64)
MS_OOF = np.asarray(ms_oof, dtype=np.float64)

Y_DEV = np.concatenate(
    train_df[LABEL_COL].apply(
        lambda x: np.asarray(x, dtype=np.int8)
    ).tolist()
)

assert V1_OOF.shape == MS_OOF.shape == Y_DEV.shape
assert np.all(np.isfinite(V1_OOF))
assert np.all(np.isfinite(MS_OOF))

offsets, cur = {}, 0
for r in train_df.itertuples():
    L = len(r.sequence)
    offsets[str(r.prot_id)] = (cur, cur+L)
    cur += L

FOLD_IDX = {}
for fold, ids in OFFICIAL_FOLDS.items():
    z = []
    for pid in ids:
        a, b = offsets[str(pid)]
        z.extend(range(a, b))
    FOLD_IDX[fold] = np.asarray(z, dtype=np.int64)

# ------------------------------------------------------------
# D. Dev-only ensemble weight lock
# ------------------------------------------------------------
weight_rows = []

for w in WEIGHT_GRID:
    p = w*V1_OOF + (1.0-w)*MS_OOF
    fold_ap = []

    for fold in sorted(FOLD_IDX):
        idx = FOLD_IDX[fold]
        fold_ap.append(
            float(average_precision_score(Y_DEV[idx], p[idx]))
        )

    weight_rows.append({
        "w_V1": float(w),
        "w_MS": float(1.0-w),
        "CV_AUPRC_mean": float(np.mean(fold_ap)),
        "CV_AUPRC_std": float(np.std(fold_ap, ddof=1)),
        "OOF_AUPRC": float(average_precision_score(Y_DEV, p)),
        **{
            f"fold{i+1}_AUPRC": float(v)
            for i, v in enumerate(fold_ap)
        }
    })

weight_search = pd.DataFrame(weight_rows).sort_values(
    ["CV_AUPRC_mean", "w_V1"],
    ascending=[False, False]
).reset_index(drop=True)

best = weight_search.iloc[0]
W_V1_CAND = float(best.w_V1)
W_MS_CAND = float(best.w_MS)
ENS_DEV_CAND = float(best.CV_AUPRC_mean)

V1_DEV = float(
    patch_summary.loc[
        patch_summary.architecture.eq(V1_NAME),
        "CV_AUPRC_mean"
    ].iloc[0]
)

PROMOTE = (ENS_DEV_CAND - V1_DEV) >= MIN_DEV_GAIN

if PROMOTE:
    W_V1, W_MS = W_V1_CAND, W_MS_CAND
    FINAL_DEV = ENS_DEV_CAND
else:
    W_V1, W_MS = 1.0, 0.0
    FINAL_DEV = V1_DEV

FINAL_OOF = W_V1*V1_OOF + W_MS*MS_OOF

print("="*100)
print(f"{TASK_NAME} DEV ENSEMBLE LOCK")
print("="*100)
display(weight_search.head(10))
print("Candidate weights:", W_V1_CAND, W_MS_CAND)
print("Candidate Dev mean AUPRC:", ENS_DEV_CAND)
print("V1 Dev mean AUPRC:", V1_DEV)
print("Candidate gain:", ENS_DEV_CAND - V1_DEV)
print("Promote by >=0.002 rule:", PROMOTE)
print("FINAL weights:", W_V1, W_MS)
print("FINAL Dev mean AUPRC:", FINAL_DEV)

weight_search.to_csv(
    OUT / "DEV_ENSEMBLE_WEIGHT_SEARCH.csv",
    index=False
)

# ------------------------------------------------------------
# E. Cross-fitted ensemble audit
#    For each held fold, choose weight using only other 4 folds.
# ------------------------------------------------------------
cf_rows = []

for held_fold in sorted(FOLD_IDX):
    train_folds = [f for f in sorted(FOLD_IDX) if f != held_fold]

    cand = []
    for w in WEIGHT_GRID:
        p = w*V1_OOF + (1.0-w)*MS_OOF
        aps = [
            average_precision_score(
                Y_DEV[FOLD_IDX[f]],
                p[FOLD_IDX[f]]
            )
            for f in train_folds
        ]
        cand.append((float(np.mean(aps)), float(w)))

    # Higher mean AP first; on exact tie prefer more V1.
    cand.sort(key=lambda x: (x[0], x[1]), reverse=True)
    _, wcf = cand[0]
    pm = 1.0 - wcf

    idx = FOLD_IDX[held_fold]
    pcf = wcf*V1_OOF[idx] + pm*MS_OOF[idx]

    cf_rows.append({
        "held_fold": held_fold,
        "selected_w_V1_from_other_4_folds": wcf,
        "selected_w_MS_from_other_4_folds": pm,
        "held_fold_AUPRC_crossfit_ensemble":
            float(average_precision_score(Y_DEV[idx], pcf)),
        "held_fold_AUPRC_V1":
            float(average_precision_score(Y_DEV[idx], V1_OOF[idx])),
        "held_fold_AUPRC_MS":
            float(average_precision_score(Y_DEV[idx], MS_OOF[idx])),
    })

crossfit = pd.DataFrame(cf_rows)
crossfit.to_csv(
    OUT / "DEV_CROSSFIT_ENSEMBLE_AUDIT.csv",
    index=False
)

print("\nCross-fitted ensemble audit:")
display(crossfit)
print(
    "Crossfit mean AUPRC ensemble =",
    crossfit.held_fold_AUPRC_crossfit_ensemble.mean()
)
print(
    "Crossfit mean AUPRC V1       =",
    crossfit.held_fold_AUPRC_V1.mean()
)

# ------------------------------------------------------------
# F. Lock thresholds independently for V1, MS, FINAL from Dev OOF
# ------------------------------------------------------------
DEV_PROBS = {
    "V1_Attention": V1_OOF,
    "MS_Attention": MS_OOF,
    FINAL_NAME: FINAL_OOF,
}

THRESHOLDS = {}
threshold_rows = []
dev_metric_rows = []

for method, p in DEV_PROBS.items():
    THRESHOLDS[method] = optimize_thresholds_exact(Y_DEV, p)

    for rule, thr in THRESHOLDS[method].items():
        threshold_rows.append({
            "method": method,
            "threshold_rule": rule,
            "threshold": float(thr),
            "threshold_selected_on": (
                "fixed" if rule == "fixed_0.5" else "DevSet1010 OOF only"
            ),
        })

        m = full_global_metrics(Y_DEV, p, thr)
        pm = protein_mean_metrics_baseline_convention(
            train_df, p, thr
        )

        dev_metric_rows.append({
            "dataset": "DevSet1010_OOF",
            "method": method,
            "threshold_rule": rule,
            "threshold": float(thr),
            **m, **pm
        })

threshold_df = pd.DataFrame(threshold_rows)
dev_metrics = pd.DataFrame(dev_metric_rows)

threshold_df.to_csv(
    OUT / "DEV_THRESHOLD_LOCK.csv",
    index=False
)
dev_metrics.to_csv(
    OUT / "DEV_OOF_FULL_METRICS.csv",
    index=False
)

print("\nDEV threshold locks:")
display(threshold_df)

# ------------------------------------------------------------
# G. External prediction helpers with prediction cache
# ------------------------------------------------------------
PRED_CACHE = OUT / "prediction_cache"
PRED_CACHE.mkdir(parents=True, exist_ok=True)

def _load_cached(method, split, expected_len):
    fp = PRED_CACHE / f"{method}__{split}.npy"
    if fp.exists():
        try:
            p = np.load(fp)
            if p.shape == (expected_len,) and np.all(np.isfinite(p)):
                return p.astype(np.float64)
        except Exception:
            pass
    return None

def _save_cached(method, split, p):
    np.save(
        PRED_CACHE / f"{method}__{split}.npy",
        np.asarray(p, dtype=np.float32)
    )

def predict_v1_external(split):
    expected = int(sum(SPLIT_DF[split].sequence.str.len()))
    cached = _load_cached("V1_Attention", split, expected)
    if cached is not None:
        return cached

    ds = PatchProteinDataset(split, SPLIT_DF[split])
    preds = []

    for fold, state in PATCH_STATES[V1_NAME].items():
        model = PATCH_FACTORY[V1_NAME](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p, _ = predict_patch(model, ds)
        preds.append(p)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    p = np.mean(np.stack(preds), axis=0)
    _save_cached("V1_Attention", split, p)
    return p.astype(np.float64)

def predict_ms_external(split):
    expected = int(sum(SPLIT_DF[split].sequence.str.len()))
    cached = _load_cached("MS_Attention", split, expected)
    if cached is not None:
        return cached

    global PATCH_RAM, PATCH_FACTORY, collate_patch

    old_ram = PATCH_RAM
    old_factory = PATCH_FACTORY
    old_collate = collate_patch

    try:
        PATCH_RAM = V2_PATCH_RAM
        PATCH_FACTORY = {
            MS_NAME:
                lambda dims: NuclearPatchHypergraphMS(
                    dims,
                    mode="attention",
                    latent=PATCH_LATENT,
                    drop=0.2
                )
        }
        collate_patch = collate_patch_ms

        ds = PatchProteinDataset(split, SPLIT_DF[split])
        preds = []

        for fold, state in ms_states.items():
            model = PATCH_FACTORY[MS_NAME](DIMS).to(DEVICE)
            model.load_state_dict(state)
            p, _ = predict_patch(model, ds)
            preds.append(p)
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        p = np.mean(np.stack(preds), axis=0)
        _save_cached("MS_Attention", split, p)
        return p.astype(np.float64)

    finally:
        PATCH_RAM = old_ram
        PATCH_FACTORY = old_factory
        collate_patch = old_collate

# ------------------------------------------------------------
# H. External FULL metrics at all Dev-locked thresholds
# ------------------------------------------------------------
external_rows = []
final_prediction_files = {}

for split in ["TestSet300", "TestSetNew46"]:
    df = SPLIT_DF[split]
    y = np.concatenate(
        df[LABEL_COL].apply(
            lambda x: np.asarray(x, dtype=np.int8)
        ).tolist()
    )

    p_v1 = predict_v1_external(split)
    p_ms = predict_ms_external(split)
    p_final = W_V1*p_v1 + W_MS*p_ms

    pred_map = {
        "V1_Attention": p_v1,
        "MS_Attention": p_ms,
        FINAL_NAME: p_final,
    }

    for method, p in pred_map.items():
        for rule, thr in THRESHOLDS[method].items():
            gm = full_global_metrics(y, p, thr)
            pm = protein_mean_metrics_baseline_convention(df, p, thr)

            external_rows.append({
                "dataset": split,
                "method": method,
                "threshold_rule": rule,
                "threshold": float(thr),
                "threshold_selected_on": (
                    "fixed" if rule == "fixed_0.5"
                    else "DevSet1010 OOF only"
                ),
                "directly_comparable_to_baseline_fixed0.5":
                    bool(rule == "fixed_0.5"),
                **gm, **pm
            })

    # Save final ensemble per-residue predictions
    pids, ridx = [], []
    for r in df.itertuples():
        L = len(r.sequence)
        pids.extend([str(r.prot_id)] * L)
        ridx.extend(range(L))

    pred_df = pd.DataFrame({
        "prot_id": pids,
        "residue_index": ridx,
        "y_true": y,
        "prob_V1": p_v1,
        "prob_MS": p_ms,
        "prob_FINAL": p_final,
    })

    pred_fp = OUT / f"FINAL_RESIDUE_PREDICTIONS_{split}.csv.gz"
    pred_df.to_csv(pred_fp, index=False)
    final_prediction_files[split] = str(pred_fp)

external_full = pd.DataFrame(external_rows)

# ------------------------------------------------------------
# I. Add direct baselines — fixed 0.5 only
# ------------------------------------------------------------
def _first_existing(row, candidates, default=np.nan):
    for c in candidates:
        if c in row.index and pd.notna(row[c]):
            return float(row[c])
    return default

baseline_fp = (
    BASELINE_WORK
    / "results"
    / "FULL_DIRECT_BASELINE_METRICS_300_46.csv"
)

bb = pd.read_csv(baseline_fp)
bb = bb[bb.label.eq(LABEL_COL)].copy()

if "operating_point" in bb.columns:
    bb = bb[bb.operating_point.eq("fixed_0.5")].copy()

baseline_rows = []

for _, r in bb.iterrows():
    if str(r.method) not in {"bindNode24", "bindEmbed21DL"}:
        continue

    ap = _first_existing(r, ["AP_AUPRC", "AUPRC"])
    auroc = _first_existing(r, ["AUROC"])
    acc = _first_existing(r, ["accuracy", "Accuracy", "global_accuracy"])
    prec = _first_existing(
        r,
        ["global_positive_precision", "precision", "Precision"]
    )
    rec = _first_existing(
        r,
        ["global_positive_recall", "recall", "Recall"]
    )
    f1p = _first_existing(
        r,
        ["global_positive_F1", "F1", "f1"]
    )
    mcc = _first_existing(
        r,
        ["global_MCC", "MCC"]
    )
    pmf1 = _first_existing(
        r,
        ["protein_mean_F1"]
    )
    pmmcc = _first_existing(
        r,
        ["protein_mean_MCC"]
    )

    baseline_rows.append({
        "dataset": str(r.dataset),
        "method": str(r.method),
        "threshold_rule": "fixed_0.5",
        "threshold": 0.5,
        "threshold_selected_on": "reproduced baseline operating point",
        "directly_comparable_to_baseline_fixed0.5": True,

        "n_residues": np.nan,
        "n_positive": np.nan,
        "prevalence": np.nan,
        "TP": np.nan, "TN": np.nan, "FP": np.nan, "FN": np.nan,

        "AUPRC_global_residue": ap,
        "AUROC_global_residue": auroc,
        "Accuracy_global_residue": acc,
        "Balanced_accuracy_global_residue": np.nan,
        "Precision_positive_global_residue": prec,
        "Recall_positive_global_residue": rec,
        "Sensitivity_global_residue": rec,
        "Specificity_global_residue": np.nan,
        "NPV_global_residue": np.nan,
        "F1_positive_global_residue": f1p,
        "MCC_global_residue": mcc,

        # Formal micro-F1 equals accuracy for binary single-label,
        # but only fill when reproduced baseline accuracy exists.
        "F1_micro_all_classes": acc,
        "F1_macro_two_classes": np.nan,

        "protein_mean_F1_baseline_convention": pmf1,
        "protein_mean_MCC_baseline_convention": pmmcc,
        "protein_n_total": np.nan,
        "protein_n_included_baseline_convention": np.nan,
        "protein_n_excluded_TN_only": np.nan,
    })

all_metrics = pd.concat(
    [external_full, pd.DataFrame(baseline_rows)],
    ignore_index=True,
    sort=False
)

all_metrics.to_csv(
    OUT / "EXTERNAL_FULL_METRICS_LONG.csv",
    index=False
)

# ------------------------------------------------------------
# J. Paper-ready fixed-0.5 headline comparison
# ------------------------------------------------------------
headline_cols = [
    "dataset", "method", "threshold",
    "AUPRC_global_residue",
    "AUROC_global_residue",
    "Accuracy_global_residue",
    "Precision_positive_global_residue",
    "Recall_positive_global_residue",
    "F1_positive_global_residue",
    "MCC_global_residue",
    "F1_micro_all_classes",
    "F1_macro_two_classes",
    "protein_mean_F1_baseline_convention",
    "protein_mean_MCC_baseline_convention",
]

headline = all_metrics[
    all_metrics.threshold_rule.eq("fixed_0.5")
][headline_cols].copy()

headline = headline.sort_values(
    ["dataset", "AUPRC_global_residue"],
    ascending=[True, False]
)

headline.to_csv(
    OUT / "HEADLINE_FIXED05_COMPARISON.csv",
    index=False
)

# ------------------------------------------------------------
# K. Paper-ready FINAL model threshold table
# ------------------------------------------------------------
final_threshold_report = external_full[
    external_full.method.eq(FINAL_NAME)
].copy()

final_threshold_report = final_threshold_report[
    [
        "dataset", "method", "threshold_rule", "threshold",
        "AUPRC_global_residue", "AUROC_global_residue",
        "Accuracy_global_residue",
        "Balanced_accuracy_global_residue",
        "Precision_positive_global_residue",
        "Recall_positive_global_residue",
        "Specificity_global_residue",
        "F1_positive_global_residue",
        "MCC_global_residue",
        "F1_micro_all_classes",
        "F1_macro_two_classes",
        "protein_mean_F1_baseline_convention",
        "protein_mean_MCC_baseline_convention",
        "protein_n_included_baseline_convention",
    ]
]

final_threshold_report.to_csv(
    OUT / "FINAL_MODEL_ALL_THRESHOLDS.csv",
    index=False
)

# ------------------------------------------------------------
# L. Final decision JSON + display
# ------------------------------------------------------------
decision = {
    "task": TASK_NAME,
    "final_method": FINAL_NAME,
    "V1_name": V1_NAME,
    "MS_name": MS_NAME,
    "candidate_w_V1": W_V1_CAND,
    "candidate_w_MS": W_MS_CAND,
    "final_w_V1": W_V1,
    "final_w_MS": W_MS,
    "V1_Dev_mean_AUPRC": V1_DEV,
    "candidate_ensemble_Dev_mean_AUPRC": ENS_DEV_CAND,
    "final_Dev_mean_AUPRC": FINAL_DEV,
    "candidate_Dev_gain_vs_V1": ENS_DEV_CAND - V1_DEV,
    "promoted_by_0.002_rule": bool(PROMOTE),
    "crossfit_mean_AUPRC_ensemble":
        float(crossfit.held_fold_AUPRC_crossfit_ensemble.mean()),
    "crossfit_mean_AUPRC_V1":
        float(crossfit.held_fold_AUPRC_V1.mean()),
    "metric_naming": {
        "F1_positive_global_residue":
            "binary positive-class F1 after pooling all residues",
        "MCC_global_residue":
            "MCC from one pooled residue-level confusion matrix",
        "F1_micro_all_classes":
            "formal sklearn-style micro-F1 across both classes; "
            "equals accuracy in this binary single-label task",
        "protein_mean_metrics":
            "per-protein metrics averaged after excluding TN-only proteins "
            "(TP+FP+FN=0), matching reproduced baseline convention",
    },
    "threshold_rules": {
        method: {k: float(v) for k, v in rules.items()}
        for method, rules in THRESHOLDS.items()
    },
    "prediction_files": final_prediction_files,
}

(OUT / "FINAL_DECISION_AND_DEFINITIONS.json").write_text(
    json.dumps(decision, indent=2)
)

print("\n" + "="*100)
print(f"{TASK_NAME} — HEADLINE FIXED 0.5 (DIRECT BASELINE COMPARISON)")
print("="*100)
display(headline)

print("\n" + "="*100)
print(f"{TASK_NAME} — FINAL MODEL AT ALL DEV-LOCKED THRESHOLDS")
print("="*100)
display(final_threshold_report)

print("\nMetric naming:")
print("  F1_positive_global_residue = pooled binary positive-class F1")
print("  MCC_global_residue         = pooled/global residue-level MCC")
print("  F1_micro_all_classes       = formal micro-F1; equals Accuracy here")
print("  protein_mean_*             = baseline convention, TN-only excluded")

print("\nSaved:", OUT)
print("Key files:")
for fn in [
    "DEV_ENSEMBLE_WEIGHT_SEARCH.csv",
    "DEV_CROSSFIT_ENSEMBLE_AUDIT.csv",
    "DEV_THRESHOLD_LOCK.csv",
    "DEV_OOF_FULL_METRICS.csv",
    "EXTERNAL_FULL_METRICS_LONG.csv",
    "HEADLINE_FIXED05_COMPARISON.csv",
    "FINAL_MODEL_ALL_THRESHOLDS.csv",
    "FINAL_DECISION_AND_DEFINITIONS.json",
]:
    print(" -", OUT / fn)


NUCLEAR DEV ENSEMBLE LOCK


,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.70,0.30,0.508961,0.088127,0.499399,0.621704,0.569264,0.416005,0.506799,0.431034
1,0.75,0.25,0.508836,0.088383,0.498268,0.622199,0.569397,0.415494,0.505746,0.431343
2,0.60,0.40,0.508830,0.087920,0.500777,0.620899,0.568362,0.415829,0.508760,0.430301
3,0.80,0.20,0.508746,0.088352,0.496696,0.622117,0.569541,0.414385,0.504810,0.432879
4,0.65,0.35,0.508732,0.088106,0.500164,0.621396,0.568819,0.415991,0.507109,0.430348
5,0.55,0.45,0.508344,0.088076,0.500859,0.620327,0.567899,0.414923,0.509061,0.429511
6,0.85,0.15,0.508255,0.088613,0.494368,0.622004,0.569522,0.412994,0.503455,0.433299
7,0.50,0.50,0.507818,0.088120,0.500562,0.619743,0.566955,0.414553,0.509612,0.428228
8,0.90,0.10,0.507795,0.088558,0.490964,0.621642,0.569000,0.411015,0.502073,0.435245
9,0.45,0.55,0.507183,0.088312,0.499897,0.619011,0.566335,0.413627,0.509952,0.426988


Candidate weights: 0.7000000000000001 0.29999999999999993
Candidate Dev mean AUPRC: 0.5089611583669623
V1 Dev mean AUPRC: 0.5019338304768628
Candidate gain: 0.007027327890099522
Promote by >=0.002 rule: True
FINAL weights: 0.7000000000000001 0.29999999999999993
FINAL Dev mean AUPRC: 0.5089611583669623

Cross-fitted ensemble audit:


,held_fold,selected_w_V1_from_other_4_folds,selected_w_MS_from_other_4_folds,held_fold_AUPRC_crossfit_ensemble,held_fold_AUPRC_V1,held_fold_AUPRC_MS
0,1,0.6,0.4,0.620899,0.617442,0.605184
1,2,0.6,0.4,0.568362,0.567193,0.542638
2,3,0.8,0.2,0.414385,0.402570,0.394629
3,4,0.8,0.2,0.504810,0.493936,0.483249
4,5,0.6,0.4,0.430301,0.428528,0.397694


Crossfit mean AUPRC ensemble = 0.5077515515088662
Crossfit mean AUPRC V1       = 0.5019338304768628

DEV threshold locks:


...


NUCLEAR — HEADLINE FIXED 0.5 (DIRECT BASELINE COMPARISON)


,dataset,method,threshold,AUPRC_global_residue,AUROC_global_residue,Accuracy_global_residue,Precision_positive_global_residue,Recall_positive_global_residue,F1_positive_global_residue,MCC_global_residue,F1_micro_all_classes,F1_macro_two_classes,protein_mean_F1_baseline_convention,protein_mean_MCC_baseline_convention
8,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,0.5,0.432839,0.941656,0.977620,0.534572,0.352381,0.424764,0.423217,0.977620,0.706676,0.294317,0.294110
0,TestSet300,V1_Attention,0.5,0.432288,0.941411,0.977939,0.548712,0.333333,0.414727,0.417265,0.977939,0.701742,0.295463,0.296574
4,TestSet300,MS_Attention,0.5,0.418940,0.935185,0.976232,0.491597,0.397959,0.439850,0.430340,0.976232,0.713854,0.285415,0.279724
26,TestSet300,bindNode24,0.5,0.400676,0.943253,0.977093,NaN,NaN,0.383162,0.386394,0.977093,NaN,0.261493,0.248807
24,TestSet300,bindEmbed21DL,0.5,0.371532,0.921621,0.976599,NaN,NaN,0.378126,0.379045,0.976599,NaN,0.238822,0.224839
16,TestSetNew46,MS_Attention,0.5,0.128783,0.936511,0.947808,0.135294,0.597403,0.220624,0.267326,0.947808,0.596812,0.169548,0.157788
27,TestSetNew46,bindNode24,0.5,0.124657,0.948962,0.945720,NaN,NaN,0.221198,0.272449,0.945720,NaN,0.250441,0.254545
20,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,0.5,0.124501,0.936898,0.950217,0.127796,0.519481,0.205128,0.240296,0.950217,0.589716,0.171518,0.158515
12,TestSetNew46,V1_Attention,0.5,0.122551,0.936822,0.951180,0.125413,0.493506,0.200000,0.231350,0.951180,0.587411,0.155089,0.143569
25,TestSetNew46,bindEmbed21DL,0.5,0.094095,0.927154,0.945560,NaN,NaN,0.198582,0.239288,0.945560,NaN,0.193721,0.192283



NUCLEAR — FINAL MODEL AT ALL DEV-LOCKED THRESHOLDS


,dataset,method,threshold_rule,threshold,AUPRC_global_residue,AUROC_global_residue,Accuracy_global_residue,Balanced_accuracy_global_residue,Precision_positive_global_residue,Recall_positive_global_residue,Specificity_global_residue,F1_positive_global_residue,MCC_global_residue,F1_micro_all_classes,F1_macro_two_classes,protein_mean_F1_baseline_convention,protein_mean_MCC_baseline_convention,protein_n_included_baseline_convention
8,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,fixed_0.5,0.500000,0.432839,0.941656,0.977620,0.672507,0.534572,0.352381,0.992633,0.424764,0.423217,0.977620,0.706676,0.294317,0.294110,85
9,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_mcc_opt,0.735698,0.432839,0.941656,0.978545,0.599284,0.633833,0.201361,0.997207,0.305627,0.349445,0.978545,0.647366,0.225327,0.247042,76
10,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_positive_f1_opt,0.735698,0.432839,0.941656,0.978545,0.599284,0.633833,0.201361,0.997207,0.305627,0.349445,0.978545,0.647366,0.225327,0.247042,76
11,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_youden_j_opt,0.019658,0.432839,0.941656,0.903907,0.867807,0.174435,0.829932,0.905683,0.288280,0.353575,0.903907,0.618377,0.109988,0.099723,259
20,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,fixed_0.5,0.500000,0.124501,0.936898,0.950217,0.737545,0.127796,0.519481,0.955610,0.205128,0.240296,0.950217,0.589716,0.171518,0.158515,17
21,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_mcc_opt,0.735698,0.124501,0.936898,0.965794,0.617187,0.113636,0.259740,0.974634,0.158103,0.156287,0.965794,0.570323,0.179468,0.198826,14
22,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_positive_f1_opt,0.735698,0.124501,0.936898,0.965794,0.617187,0.113636,0.259740,0.974634,0.158103,0.156287,0.965794,0.570323,0.179468,0.198826,14
23,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_youden_j_opt,0.019658,0.124501,0.936898,0.807291,0.870378,0.056827,0.935065,0.805691,0.107143,0.203343,0.807291,0.499566,0.054913,0.043092,39



Metric naming:
  F1_positive_global_residue = pooled binary positive-class F1
  MCC_global_residue         = pooled/global residue-level MCC
  F1_micro_all_classes       = formal micro-F1; equals Accuracy here
  protein_mean_*             = baseline convention, TN-only excluded

Saved: /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY
Key files:
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/DEV_ENSEMBLE_WEIGHT_SEARCH.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/DEV_CROSSFIT_ENSEMBLE_AUDIT.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/DEV_THRESHOLD_LOCK.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/DEV_OOF_FULL_METRICS.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/EXTERNAL_FULL_METRICS_LONG.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_FINAL_PAPER_READY/HE